# PDSF Pipeline v11.4 — Geometry & Continuation Experiments

## What This Notebook Does

This notebook runs the **PDSF (Predictive–Discriminative–Situational–Framework) decomposition**
experiments on transformer language models. The core idea: when a model processes a prompt,
its internal representation (the "residual stream") can be decomposed into four orthogonal
subspaces that serve different functions:

- **P (Predictive)**: The direction(s) used to predict the next token. Defined by the
  unembedding vectors for expected answers. Rank set adaptively by PR threshold.
- **D (Discriminative)**: The dominant variance directions after removing P — captures how
  different prompts diverge from each other. Dimensionality set adaptively by PR.
- **S (Situational)**: The next layer of variance after removing P and D — captures
  slower-varying structural patterns. Rank set by `S_SMALL_R`.
- **F (Framework-Shared)**: Everything not captured by P, D, or S — the high-dimensional
  residual occupying thousands of dimensions. Lowest density per dimension, largest subspace.

The decomposition H = H_P + H_D + H_S + H_F is computed at every layer, enabling us to
track how information flows through the network.

> **Naming history (read this if comparing old vs new output files):**
> Before v10, the pipeline had three subspaces (P, D, S) where S was the entire
> remainder after removing P and D. In v10+, the old S was split into S (the top-k
> variance directions, set by `S_SMALL_R`) and F (the high-dimensional residual).
> Old output files use `S_small` for the current S and `h_perp` for the full
> P-perpendicular complement. The Part E field `mu_energy_frac_S_small` in pre-v10
> outputs is the combined S+F energy, not the current S alone. When loading old
> results, note that F fraction must be computed as `1.0 - P - D - S_small` only
> if `S_small` already reflects the split; otherwise re-run Part E with the
> current pipeline.

## Two Experiments

### Experiment 1: SpecA (Geometry Tests, Parts A–H)
Uses 224 prompts (14 groups × 16 variants) with a 2⁴ factorial design. All prompts
ask simple factual/logical questions ("Is 2+2=4?") with known single-token answers.
The 4 factors (paraphrase, constraint, clutter, format) systematically vary surface
form while holding meaning constant. Analyses measure:

- **Part A**: Effective dimensionality (participation ratio) of P and P⊥
- **Part B**: Whether same-question variants cluster together (family separation)
- **Part C**: Which surface-form factor has the largest geometric effect
- **Part D**: How D dimensionality evolves across layers
- **Part E**: Energy decomposition of mean vectors into P, D, S
- **Part F**: Per-prompt trajectories through P-D-S-F space across layers
- **Part G**: Causal test — PDSF interventions (perturb each subspace, measure KL impact)
- **Part H**: Injectivity — whether different prompts map to distinguishable states
- **Part I (Energy Spectrum)**: Singular value distribution in P⊥ — k_D, cumulative energy, PR strategy
- **Part J (Resolution Geometry)**: Manifold curvature per PDSF subspace, PCA controls, P-PCA alignment

### Experiment 2: Continuation Tests (SpecB + Diverse)
Uses two prompt sets for behavioral validation:
- **SpecB** (80 open-ended narrative starters): P/D/S scrambles and random control.
  Tests temporal dissociation — D-scramble changes the first token, S-scramble
  changes where the continuation goes after a shared opening.
- **Diverse** (84 prompts × 8 linguistic regimes): F-attenuate, F-mix, F-transplant.
  Tests whether F is generic infrastructure (within vs cross regime transplant)
  and whether F's directional structure matters (mix destroys it, transplant preserves it).




## Files Required

| File | Role |
|------|------|
| `specA_analysis.py` | Part A–H analysis functions, PDSF math utilities |
| `pds_geometry.py` | Model loading, hidden state extraction, Part G scramble |
| `pds_continuation.py` | SpecB continuation generation and scramble |
| `pds_prompt_analysis.py` | Token distribution comparison (optional) |
| `SpecA_prompts_v4_full_factorial.json` | 224 geometry test prompts |
| `SpecB_experimental_prompts.json` | 96 continuation test prompts |

## Notebook Structure

| Section | Cells | Purpose |
|---------|-------|---------|
| 0. Environment | 2–4 | CUDA setup, pip installs, HuggingFace auth |
| 1. Configuration | 9–10 | Model selection, experiment parameters |
| 2. Modules | 13 | Import all Python modules |
| 3. Data | 15 | Load and parse prompt JSON files |
| 4. Prompt Analysis | 17–19 | Optional token distribution analysis |
| 5. Functions | 21–22 | Energy spectrum + full pipeline definition |
| 6. Execute | 24 | Run pipeline for selected models |
| 7. Cleanup | 26 | Free GPU memory |

## Quick Start

1. Edit Cell 9 (`MODELS_TO_RUN`) to select your model(s)
2. Run all cells
3. Results are saved to `/workspace/results/Geometry/{model_name}/`


---
# Section 0: Environment Setup

**CRITICAL: Run these cells FIRST before any other imports**

In [ ]:
# MUST BE FIRST CELL - Configure environment before any imports
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TMPDIR"] = "/workspace/tmp"
os.environ["TEMP"] = "/workspace/tmp"
os.environ["TMP"] = "/workspace/tmp"
os.makedirs("/workspace/tmp", exist_ok=True)
os.makedirs("/workspace/hf_cache", exist_ok=True)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs("/workspace/offload", exist_ok=True)

print("✓ Environment configured - all caches redirected to /workspace/")

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes tqdm scipy matplotlib hf_xet huggingface_hub -q
!pip install numpy==1.26.4 -q
import hf_xet
import transformers
import accelerate
print(f"transformers: {transformers.__version__}")
print(f"accelerate: {accelerate.__version__}")
print("External dependencies loaded")

In [ ]:
# HuggingFace Authentication
import os
from huggingface_hub import login, HfApi

try:
    api = HfApi()
    user = api.whoami()
    print(f"Logged in as: {user['name']}")
except Exception:
    print("Not logged in. Running interactive login...")
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
    else:
        login()

---
# Section 0.1: Check and Clear Models 

In [ ]:
# CHECK CACHE STATUS
# ============================================================
# Can run before or after imports - has fallback for early runs

try:
    print_cache_status()
except NameError:
    print("(Run after Cell 13 for full cache info, or run this standalone check)")
    from pathlib import Path
    import shutil
    for p in [Path("/workspace/hf_cache"), Path("/workspace/models")]:
        if p.exists():
            for item in sorted(p.iterdir()):
                if item.is_dir() and not item.name.startswith("."):
                    try:
                        sz = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / 1e9
                        if sz > 0.01: print(f"  {item.name}: {sz:.1f} GB")
                    except: pass

In [ ]:
# CLEAR MODELS FROM CACHE
# ============================================================
# Self-contained - can run before or after imports

CLEAR_SPECIFIC_MODEL = ""  # e.g., "meta-llama/Llama-3.3-70B-Instruct"
CLEAR_ALL = True  # Set True to clear all cached models

try:
    if CLEAR_SPECIFIC_MODEL:
        clear_hf_cache_for_model(CLEAR_SPECIFIC_MODEL, verbose=True)
    elif CLEAR_ALL:
        clear_all_hf_cache(confirm=True, verbose=True)
    else:
        print("Set CLEAR_SPECIFIC_MODEL or CLEAR_ALL=True to clear cache")
except NameError:
    # Standalone version
    import shutil
    from pathlib import Path
    
    if CLEAR_ALL:
        to_del = []
        for p in [Path("/workspace/hf_cache"), Path("/workspace/models")]:
            if p.exists():
                for item in p.iterdir():
                    if item.is_dir() and not item.name.startswith("."):
                        sz = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / 1e9
                        if sz > 0.01: to_del.append((item, sz))
        if to_del:
            print(f"Found {len(to_del)} models:")
            for item, sz in to_del: print(f"  {item.name}: {sz:.1f} GB")
            if input("Delete all? (yes/no): ").lower() == "yes":
                for item, _ in to_del: shutil.rmtree(item); print(f"  Deleted {item.name}")
        else:
            print("No cached models found")
    else:
        print("Set CLEAR_ALL=True or run after Cell 13 for full functionality")

---
# Section 1: Configuration

Model selection and experiment parameters.

In [ ]:
# ============================================================
# TEST MODE
# ============================================================
# Set TEST_MODE = True to verify the pipeline runs end-to-end
# on a small model with minimal compute (~10-15 minutes on a
# single consumer GPU with >=8GB VRAM).
#
# When enabled, TEST_MODE automatically:
#   - Uses gemma_2b (smallest model in registry, ~5GB)
#   - Limits prompts to 4-8 per set (one per group)
#   - Disables expensive parts (spirality, regime transplants)
#   - Reduces generation length to 16 tokens
#   - Enables only core geometry + basic continuation tests
#
# Set to False for full experimental runs.
# ============================================================

TEST_MODE = False  # <-- Set True for quick verification run

# to analyze. You can run multiple models sequentially.
# WHAT TO EDIT: Change the list below to select which model(s)
#
# (defined in pds_geometry.py).
# Each string maps to a HuggingFace model ID in MODEL_REGISTRY
# See the table of available models below.
# ============================================================
# ============================================================
# MODEL SELECTION
# ============================================================
# This is the ONLY cell you need to edit for most runs.
# Select which model(s) to run.
# ============================================================

MODELS_TO_RUN = ["gpt_oss_120b"]  # <-- EDIT THIS LIST (ignored in TEST_MODE)

# TEST_MODE override
if TEST_MODE:
    MODELS_TO_RUN = ["gemma_2b"]
    print("⚡ TEST MODE ENABLED — using gemma_2b with minimal prompts")

# ============================================================
# AVAILABLE MODELS (uncomment to use)
# ============================================================
# --- Small Models (< 5GB) ---
# "phi3_mini"    : microsoft/Phi-3-mini-4k-instruct
# "gemma_2b"     : google/gemma-2-2b-it
# "llama_1b"     : meta-llama/Llama-3.2-1B-Instruct
# "llama_3b"     : meta-llama/Llama-3.2-3B-Instruct

# --- Medium Models (5-20GB) ---
# "gemma_9b"     : google/gemma-2-9b-it
# "llama_8b"     : meta-llama/Llama-3.1-8B-Instruct
# "qwen_7b"      : Qwen/Qwen2.5-7B-Instruct
# "mistral_7b"   : mistralai/Mistral-7B-Instruct-v0.3

# --- Large Models (20-80GB) ---
# "gemma_27b"    : google/gemma-2-27b-it
# "qwen_14b"     : Qwen/Qwen2.5-14B-Instruct
# "qwen_32b"     : Qwen/Qwen2.5-32B-Instruct
# "llama_70b"    : meta-llama/Llama-3.3-70B-Instruct
# "qwen_72b"     : Qwen/Qwen2.5-72B-Instruct
# "gpt_oss_20b"  : openai/gpt-oss-20b  (MXFP4, ~14GB)

# --- Extra Large Models (>100GB) ---
# "gpt_oss_120b" : openai/gpt-oss-120b  (MXFP4, ~65GB, single H100)
# "llama_405b"   : meta-llama/Llama-3.1-405B-Instruct

print(f"✓ Models to run: {MODELS_TO_RUN}")



In [ ]:
# ============================================================
# PDSF PIPELINE V11.4 — EXPERIMENT CONFIGURATION
# ============================================================
# 
# CHANGELOG:
# #   • Part L (Complexity Depth Profiles): baseline manifold geometry at every layer.
#     Curvature ratio, local/global PR, info density, softmax gradient per PDSF subspace.
#     Runs alongside Part G — uses same layer_to_H cache, no extra forward passes.
#     Separate output: {model}-Geometry-{set}-part_l_complexity_depth_profile.json
#   • Part K ordering checks revised: replaced failing PCA_shuffled_near_zero with
#     PDSF_D_gt_PCA_shuffled_D (z-score test); added PDSF_D_gt_PDSF_F; demoted
#     PDSF_D_gt_PCA_D to legacy/informational.
# #   • Part M (Spirality Measures): quantifies helical/spiral structure in
#     residual stream trajectories using three independent measures (spectral
#     concentration / ringing, phase linearity, winding number).
#     Phase A: baseline spirality from layer_to_H cache (no forward passes).
#     Phase B: intervention spirality piggybacks on Part G forward passes.
#     Phase C: cross-metric correlation with Part F rotation data.
#   • Two-toggle gating: part_m (Phase A+C, no Part G needed),
#     part_m_intervention (Phase B, requires part_g).
#   • New module: pds_spirality.py
#   • New dependency: scipy (for Pearson correlation in Phase C)
# #   • Part G: regime-aware F-transplant (within/cross/null) with domain inference.
#   • Part K: cosine similarity subspace discrimination test.
#
# SECTION 1: WHAT TO RUN  (change frequently)
# SECTION 2: DETAIL PARAMETERS  (change rarely)
# SECTION 3: FILE PATHS & MODEL REGISTRY  (change per environment)
# ============================================================

import json
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════╗
# ║  SECTION 1: WHAT TO RUN                                     ║
# ║  Toggle tests, prompt sets, and parts here.                 ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────
# 1A. GEOMETRY TESTS
# ─────────────────────────────────────────
RUN_GEOMETRY = True                         # Master switch for all geometry

GEOMETRY_PROMPT_SETS = {
    "SpecA": True,                          # SpecA (factorial design prompts)
    "SpecB": True,                          # SpecB (continuation prompts)
    "Diverse_group": True,                  # Diverse prompts (group-level analysis)
    "Diverse_regime": True,                 # Diverse prompts (regime-level analysis)
}

# Which geometry parts to run (applies to all enabled prompt sets)
# Part A: Participation Ratio — effective dimensionality of P and P-perp
# Part B: Family Separation — principal angles between per-group D subspaces
# Part C: Factor Effects — Cohen's d for each design factor (SpecA only)
# Part D: PR(D) Trajectory — effective D dimensionality across layers
# Part E: mu Landscape — decompose group means into P, D, S energy
# Part F: Rotation + Per-Prompt — layer-to-layer D rotation, PDSF trajectories
# Part G: PDSF Interventions — causal scramble: perturb P/D/S/F, measure impact
# Part H: Injectivity — near-collision detection in hidden state space
# Part I: Energy Spectrum — singular value distribution in P-perp space (k_D, capture %)
# Part J: Resolution Geometry — manifold curvature per PDSF subspace (P < D < S < F ordering)
RUN_PARTS = {
    # Only Part G (regime-aware transplant) and Part K (cosine discrimination) active
    "part_a": False,                        # Participation Ratio
    "part_b": False,                        # Family Angles
    "part_c": False,                        # Factor Effects (SpecA only)
    "part_d": False,                        # Trajectory
    "part_e": False,                        # mu Landscape
    "part_f": False,                        # Rotation + Per-prompt trajectories
    "part_g": True,                         # PDSF Interventions (regime-aware transplant)
    "part_h": False,                        # Injectivity
    "part_i": False,                        # Energy Spectrum
    "part_j": False,                        # Resolution Geometry
    "part_k": True,                         # Cosine Similarity Subspace Discrimination
    "part_l": True,                         # Complexity Depth Profiles (baseline manifold geometry)
    "part_m": True,                         # Spirality Measures — Phase A (baseline) + Phase C (correlation)
    "part_m_intervention": True,            # Spirality Measures — Phase B (piggyback on Part G; requires part_g)
}

# Per-prompt-set Part G overrides
# Cross-validated analysis (V10) showed ρ>0.96 rank correlation across all
# three prompt sets. SpecA is closest to the 3-dataset mean and supports
# Part C (factor effects). SpecB adds little for Part G specifically.
# Set to None to inherit from RUN_PARTS["part_g"] above.
GEOMETRY_PART_G_OVERRIDES = {
    "SpecA": True,                          # Always run Part G on SpecA (primary)
    "SpecB": True,                         # Skip Part G on SpecB (default: False)
    "Diverse": True,                        # Run Part G on Diverse (replication)
}

# Per-prompt-set Part M intervention overrides
# The per-prompt causal correlation (spirality × angle) needs n≥40 to detect
# moderate effects (r~0.5). SpecB (n=24) is underpowered; Diverse (n=42)
# mixes regimes, washing out within-regime signal. SpecA is the primary test.
# Set to False to skip Part M Phase B (intervention spirality) on that set.
# Part M Phase A (baseline) still runs on all sets (it's free — CPU only).
GEOMETRY_PART_M_INTERVENTION_OVERRIDES = {
    "SpecA": True,                          # Primary: n=40, single domain
    "SpecB": False,                         # Underpowered: n=24
    "Diverse": False,                       # Cross-regime confound: n=42 but 8 regimes
}

# ─────────────────────────────────────────
# 1B. CONTINUATION TESTS
# ─────────────────────────────────────────
RUN_CONTINUATION = True                    # Continuation tests disabled (geometry focus run)

CONTINUATION_PROMPT_SETS = {
    "SpecB": True,                          # Standard continuation prompts
    "Diverse": True,                        # Diverse prompts
}

# Global scramble & F switches (apply to both prompt sets unless overridden)
CONTINUATION_SCRAMBLE_TESTS = {
    "P_scramble": False,                     # P subspace rotation
    "D_scramble": True,                     # D subspace rotation
    "S_scramble": True,                     # S subspace rotation
    "random_control": True,                 # Random subspace control
}
# Note on random_control: scrambles a random orthonormal basis of rank
# matching D, using the same rotation method as P/D/S. Unlike the P/D/S
# scrambles, it operates on the full hidden state (no PDSF decomposition).
# This tests whether perturbation effects are subspace-specific or generic.

CONTINUATION_F_TESTS = {
    "F_standard": True,                     # F interventions at standard depth (70%)
    "F_early": False,                        # F interventions at Part G depth (~12.5%)
}

# Per-prompt-set overrides: PDSF scrambles and F interventions
# Cross-validated analysis (V10) showed ρ=0.987 rank correlation between
# prompt sets, so running each test type on one set is sufficient.
# Set to None to inherit from the global switches above.
CONTINUATION_SPECB_OVERRIDES = {
    "PDS_scrambles": True,                  # Run P/D/S/F scrambles on SpecB (default: yes)
    "F_tests": False,                       # Run F interventions on SpecB (default: no)
}
CONTINUATION_DIVERSE_OVERRIDES = {
    "PDS_scrambles": False,                 # Run P/D/S/F scrambles on Diverse (default: no)
    "F_tests": True,                        # Run F interventions on Diverse (default: yes)
}
# Set any override to None to fall back to the global switches.

# ─────────────────────────────────────────
# 1C. ANALYSIS & COMPARISON
# ─────────────────────────────────────────
RUN_TOKEN_DISTRIBUTION_ANALYSIS = False
RUN_GEOMETRY_COMPARISON = False

# ─────────────────────────────────────────
# 1D. CACHING
# ─────────────────────────────────────────
USE_EXTRACTION_CACHE = True
SAVE_EXTRACTION_CACHE = True
USE_CACHED_BASES = True
USE_CACHED_BASELINES = True
CLEAR_CACHE_BETWEEN_MODELS = False          # Delete extraction .npz cache after each model (saves disk)

VERBOSE = True


# ╔══════════════════════════════════════════════════════════════╗
# ║  SECTION 2: DETAIL PARAMETERS                               ║
# ║  Tuning knobs — rarely need changing.                       ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────
# 2A. Geometry Part G Parameters
# ─────────────────────────────────────────
SCRAMBLE_LAYERS = ["early"]                 # "early", "mid", "late", or layer numbers
N_SCRAMBLES = 3                             # Scrambles per prompt
S_RANK = 13                                 # S basis rank for Part G
SCRAMBLE_SEED = 42
N_SCRAMBLE_PROMPTS = 40                     # Number of prompts for SpecA (None = all)
SPECB_N_SCRAMBLE_PROMPTS = 24               # SpecB experimental: min for all 12 groups (stride=4)
                                            # n=20 misses group_11 and group_12.
                                            # n=48 full coverage with 4 variants/group.
DIVERSE_N_SCRAMBLE_PROMPTS = 42             # Diverse: min for 8-regime x >=2-group coverage (stride=2)
                                            # n=20 misses unusual_register (1 group; within-regime degenerate).
                                            # n=84 full dataset; ~2x cost of n=42, no structural benefit.
# Note: "rotation" applies to P/D/S only. "mix" applies to F only (random F injection).
# "transplant" applies to all subspaces (cross-prompt donor injection).
# "attenuate" applies to all subspaces (scale by alpha).
GEOMETRY_SUBSPACES = ["P", "D", "S", "F"]
GEOMETRY_INTERVENTION_TYPES = [
    "attenuate", "rotation", "mix",
    "transplant",                        # P/D/S: original cross-group donor
    "transplant_within",                 # F: same regime/domain, different group
    "transplant_cross",                  # F: different regime/domain (near, +1 index)
    "transplant_null",                   # F: null-state donor (single space " ")
]
GEOMETRY_ATTENUATE_ALPHA = 0.5

# Part G: KL Divergence & Entropy
COMPUTE_KL = True                           # KL divergence + top-k entropy
KL_TOPK = 2048                              # Top-k approximation size

# Part M: Spirality Measures (runs alongside Part G and Parts A-F)
SPIRALITY_N_PC_PAIRS = 3                    # Number of PC pairs for phase/winding analysis

# ─────────────────────────────────────────
# 2B. Geometry Other Parameters
# ─────────────────────────────────────────
K_MAX = 64                                  # Max rank for PCA
S_SMALL_R = 16                              # S basis rank for Parts A-F
# NOTE: S_SMALL_R defines the S/F boundary. The top S_SMALL_R variance directions
# (after removing P and D) become S; everything else becomes F. Old pipeline outputs
# (pre-v10) have no S/F split — their "S_small" field is the current S+F combined,
# and "h_perp" is the full P-perpendicular space. See Cell 0 naming history note.

# Part J: resolution geometry layer depth
# "all" runs at every layer (slow but complete).
# "sparse" runs at 25%, 50%, 75%, and final layer only (faster for large models).
RESOLUTION_GEOMETRY_LAYERS = "all"          # "all" or "sparse"

# ─────────────────────────────────────────
# 2C. Part K — Cosine Discrimination Parameters
# ─────────────────────────────────────────
# Part K measures within-group vs between-group pairwise cosine similarity
# for each PDSF subspace and two PCA controls. Added in V11.2.
# Primary question: does PDSF decomposition separate groups better than PCA?
# Secondary: is F group separation lower than D and S (shared infrastructure)?
# Runs on all three prompt sets (SpecA, SpecB, Diverse).
PART_K_LAYER = "L-2"      # Target layer: "L-2" (second-to-last) or explicit int index
PART_K_N_SHUFFLES = 20    # PCA-shuffled null permutations (averaged for stable baseline)

# ─────────────────────────────────────────
# 2D. Continuation Parameters
# ─────────────────────────────────────────
SPECB2_SCRAMBLE_LAYER_FRACTION = 0.70       # Standard depth (70%)
CONTINUATION_F_EARLY_LAYER_FRACTION = "match_geometry"  # Early F depth ("match_geometry" or 0.0-1.0)
SPECB2_N_GENERATE = 64                      # Tokens per prompt
SPECB2_SCRAMBLE_TYPE = "rotation"           # "rotation" or "permutation"
SPECB2_SEED = 42

# Basis rank configuration
SPECB2_P_RANK = "adaptive"
SPECB2_D_RANK = "adaptive"
SPECB2_S_RANK = "adaptive"
SPECB2_RANDOM_RANK = "adaptive"             # Matches D rank for fair comparison

# Prompt tiers
SPECB2_TIER = "standard"                    # "lite", "standard", "full"
DIVERSE_TIER = "full"
DIVERSE_S_RANK = 13

# F intervention parameters
ATTENUATE_ALPHA_CONTROL = 0.5
PAIR_POLICY = "fast"

# ─────────────────────────────────────────
# 2E. Token Distribution Analysis
# ─────────────────────────────────────────
PROMPT_ANALYSIS_N_PROMPTS = 50
PROMPT_ANALYSIS_TOP_K = 20


# ─────────────────────────────────────────
# 2F. V11.2 Changelog
# ─────────────────────────────────────────
# V11.2 (this version):
#   • Part G: Regime-aware F-transplant with null control (3 typed donors)
#     transplant_within — same domain/regime, different group (baseline disruption)
#     transplant_cross  — adjacent domain/regime +1 (tests regime-specificity)
#     transplant_null   — single space " " near-prior donor (maximally foreign F)
#     Four-outcome decision tree: within≈cross≈null / within<cross<null /
#                                 within<cross≈null / within≈cross<null
#   • Part K: New cosine similarity subspace discrimination test
#     Measures within-group vs between-group pairwise cosine sim for D, S, F,
#     PCA-contiguous control, and PCA-shuffled null distribution.
#   • SpecA domain mapping: 14 groups → 4 domains (arithmetic/logic/factual/linguistic)
#   • SpecB domain mapping: 12 groups → 4 domains (internal_states/agent_action/
#                           narrative_frame/context_setting)
#   • Diverse regime mapping: passes regime_ids directly (8 regimes)
#   • Sample sizes: SpecA n=20, SpecB n=24 (covers all 12 groups), Diverse n=42
#   • BUG FIX: RUN_PARTS["part_i"/"part_j"] gating was OR-logic (always ran);
#     corrected to nested .get() pattern so disabling actually disables.
#   • OUTPUT_FILES_DIVERSE_REGIME: added missing part_g_regime_transplant, part_k,
#     resolution_geometry, energy_spectrum, _prompt_set_label keys.
# V11.1 (previous):
#   • Multi-component Part G output (P, D, S, F divergence all recorded)
#   • Part I (Energy Spectrum) and Part J (Resolution Geometry) added
#   • Per-prompt-set output file routing to prevent cross-set overwrite
#   • S_RANK / S_SMALL_R distinction (Part G vs Parts A–F)

# ╔══════════════════════════════════════════════════════════════╗
# ║  SECTION 3: FILE PATHS, MODEL REGISTRY, SETUP               ║
# ║  Environment-specific. Change once per machine.              ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────
# 3A. File Paths
# ─────────────────────────────────────────
BASE_DIR = Path("/workspace")
# Prompt files ship in the prompts/ subdirectory of the repository.
# If running from the repo root, use Path(".") as BASE_DIR.
GEOMETRY_PROMPTS_PATH = BASE_DIR / "prompts" / "SpecA_prompts_v4_full_factorial.json"
CONTINUATION_PROMPTS_PATH = BASE_DIR / "prompts" / "SpecB_experimental_prompts.json"
DIVERSE_PROMPTS_PATH = BASE_DIR / "prompts" / "specB_diverse_prompts.json"
OUTPUT_DIR = BASE_DIR / "results" / "Geometry"
CONTINUATION_OUTPUT_DIR = BASE_DIR / "results" / "Continuation"
DIVERSE_OUTPUT_DIR = CONTINUATION_OUTPUT_DIR  # Diverse continuation shares Continuation dir
CACHE_DIR = BASE_DIR / "cache"

# Backwards compatibility aliases
SPECA_PROMPTS_PATH = GEOMETRY_PROMPTS_PATH
SPECB_PROMPTS_PATH = CONTINUATION_PROMPTS_PATH
SPECB2_OUTPUT_DIR = CONTINUATION_OUTPUT_DIR

for d in [OUTPUT_DIR, CONTINUATION_OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────
# 3B. Build PIPELINE_OPTIONS from Section 1
# ─────────────────────────────────────────
PIPELINE_OPTIONS = {
    # Geometry
    "run_geometry_analysis": RUN_GEOMETRY and GEOMETRY_PROMPT_SETS["SpecA"],
    "run_geometry_part_g": RUN_GEOMETRY and RUN_PARTS["part_g"],
    "run_geometry_part_l": RUN_GEOMETRY and RUN_PARTS.get("part_l", False),
    "run_geometry_part_g_specB": RUN_GEOMETRY and RUN_PARTS["part_g"] and (
        GEOMETRY_PART_G_OVERRIDES.get("SpecB") is not False),
    "run_geometry_part_l_specB": RUN_GEOMETRY and RUN_PARTS.get("part_l", False) and (
        GEOMETRY_PART_G_OVERRIDES.get("SpecB") is not False),
    "run_geometry_part_g_diverse": RUN_GEOMETRY and RUN_PARTS["part_g"] and (
        GEOMETRY_PART_G_OVERRIDES.get("Diverse") is not False),
    "run_geometry_part_l_diverse": RUN_GEOMETRY and RUN_PARTS.get("part_l", False) and (
        GEOMETRY_PART_G_OVERRIDES.get("Diverse") is not False),
    # Part M: Phase A (baseline) + C (correlation) — independent of Part G
    "run_geometry_part_m": RUN_GEOMETRY and RUN_PARTS.get("part_m", False),
    "run_geometry_part_m_specB": RUN_GEOMETRY and RUN_PARTS.get("part_m", False) and (
        GEOMETRY_PART_G_OVERRIDES.get("SpecB") is not False),
    "run_geometry_part_m_diverse": RUN_GEOMETRY and RUN_PARTS.get("part_m", False) and (
        GEOMETRY_PART_G_OVERRIDES.get("Diverse") is not False),
    # Part M: Phase B (intervention spirality) — requires Part G
    "run_geometry_part_m_intervention": (
        RUN_GEOMETRY and RUN_PARTS.get("part_m", False)
        and RUN_PARTS.get("part_m_intervention", True)
        and RUN_PARTS["part_g"]
    ),
    "run_geometry_part_m_intervention_specB": (
        RUN_GEOMETRY and RUN_PARTS.get("part_m", False)
        and RUN_PARTS.get("part_m_intervention", True)
        and RUN_PARTS["part_g"]
        and (GEOMETRY_PART_G_OVERRIDES.get("SpecB") is not False)
        and GEOMETRY_PART_M_INTERVENTION_OVERRIDES.get("SpecB", True)
    ),
    "run_geometry_part_m_intervention_diverse": (
        RUN_GEOMETRY and RUN_PARTS.get("part_m", False)
        and RUN_PARTS.get("part_m_intervention", True)
        and RUN_PARTS["part_g"]
        and (GEOMETRY_PART_G_OVERRIDES.get("Diverse") is not False)
        and GEOMETRY_PART_M_INTERVENTION_OVERRIDES.get("Diverse", True)
    ),
    "run_geometry_on_continuation_prompts": RUN_GEOMETRY and GEOMETRY_PROMPT_SETS["SpecB"],
    "run_geometry_on_diverse_prompts": RUN_GEOMETRY and (
        GEOMETRY_PROMPT_SETS["Diverse_group"] or GEOMETRY_PROMPT_SETS["Diverse_regime"]
    ),

    # Continuation — SpecB
    # Per-prompt-set overrides: None = inherit global, True/False = force
    "run_continuation_tests": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["SpecB"],
    "run_continuation_P_scramble": RUN_CONTINUATION and CONTINUATION_SCRAMBLE_TESTS["P_scramble"] and (
        CONTINUATION_SPECB_OVERRIDES.get("PDS_scrambles") is not False),
    "run_continuation_D_scramble": RUN_CONTINUATION and CONTINUATION_SCRAMBLE_TESTS["D_scramble"] and (
        CONTINUATION_SPECB_OVERRIDES.get("PDS_scrambles") is not False),
    "run_continuation_S_scramble": RUN_CONTINUATION and CONTINUATION_SCRAMBLE_TESTS["S_scramble"] and (
        CONTINUATION_SPECB_OVERRIDES.get("PDS_scrambles") is not False),
    "run_continuation_F_tests": RUN_CONTINUATION and CONTINUATION_F_TESTS["F_standard"] and (
        CONTINUATION_SPECB_OVERRIDES.get("F_tests") is not False),
    "run_continuation_F_early": RUN_CONTINUATION and CONTINUATION_F_TESTS["F_early"] and (
        CONTINUATION_SPECB_OVERRIDES.get("F_tests") is not False),
    "run_continuation_random_control": RUN_CONTINUATION and CONTINUATION_SCRAMBLE_TESTS["random_control"] and (
        CONTINUATION_SPECB_OVERRIDES.get("PDS_scrambles") is not False),

    # Continuation — Diverse
    "run_diverse_continuation_tests": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"],
    "run_diverse_P_scramble": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"] and CONTINUATION_SCRAMBLE_TESTS["P_scramble"] and (
        CONTINUATION_DIVERSE_OVERRIDES.get("PDS_scrambles") is not False),
    "run_diverse_D_scramble": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"] and CONTINUATION_SCRAMBLE_TESTS["D_scramble"] and (
        CONTINUATION_DIVERSE_OVERRIDES.get("PDS_scrambles") is not False),
    "run_diverse_S_scramble": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"] and CONTINUATION_SCRAMBLE_TESTS["S_scramble"] and (
        CONTINUATION_DIVERSE_OVERRIDES.get("PDS_scrambles") is not False),
    "run_diverse_F_tests": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"] and CONTINUATION_F_TESTS["F_standard"] and (
        CONTINUATION_DIVERSE_OVERRIDES.get("F_tests") is not False),
    "run_diverse_F_early": RUN_CONTINUATION and CONTINUATION_PROMPT_SETS["Diverse"] and CONTINUATION_F_TESTS["F_early"] and (
        CONTINUATION_DIVERSE_OVERRIDES.get("F_tests") is not False),

    # Energy spectrum (Part I) & resolution geometry (Part J)
    "run_energy_spectrum": RUN_GEOMETRY and RUN_PARTS["part_i"],
    "run_resolution_geometry": RUN_GEOMETRY and RUN_PARTS["part_j"],

    # Analysis
    "run_token_distribution_analysis": RUN_TOKEN_DISTRIBUTION_ANALYSIS,
    "run_geometry_comparison": RUN_GEOMETRY_COMPARISON,

    # Caching
    "use_extraction_cache": USE_EXTRACTION_CACHE,
    "save_extraction_cache": SAVE_EXTRACTION_CACHE,
    "use_cached_bases": USE_CACHED_BASES,
    "use_cached_baselines": USE_CACHED_BASELINES,
    "clear_cache_between_models": CLEAR_CACHE_BETWEEN_MODELS,

    # Verbosity
    "verbose": VERBOSE,
}

# Backward compatibility aliases
_alias_map = {
    "run_specA_analysis": "run_geometry_analysis",
    "run_specA_part_g": "run_geometry_part_g",
    "run_specA_part_l": "run_geometry_part_l",
    "run_specA_part_m": "run_geometry_part_m",
    "run_specA_part_m_intervention": "run_geometry_part_m_intervention",
    "run_specB2": "run_continuation_tests",
    "run_specB2_P_scramble": "run_continuation_P_scramble",
    "run_specB2_D_scramble": "run_continuation_D_scramble",
    "run_specB2_S_scramble": "run_continuation_S_scramble",
    "run_specB2_random_control": "run_continuation_random_control",
    "run_prompt_analysis": "run_token_distribution_analysis",
    "run_specA_style_on_specB": "run_geometry_on_continuation_prompts",
    "run_specB_part_g": "run_geometry_part_g_specB",
    "run_specB_part_l": "run_geometry_part_l_specB",
    "run_specB_part_m": "run_geometry_part_m_specB",
    "run_specB_part_m_intervention": "run_geometry_part_m_intervention_specB",
    "run_diverse_part_g": "run_geometry_part_g_diverse",
    "run_diverse_part_l": "run_geometry_part_l_diverse",
    "run_diverse_part_m": "run_geometry_part_m_diverse",
    "run_diverse_part_m_intervention": "run_geometry_part_m_intervention_diverse",
    "run_diverse_specB2": "run_diverse_continuation_tests",
    "run_token_distribution_comparison": "run_token_distribution_analysis",
}
for old, new in _alias_map.items():
    PIPELINE_OPTIONS[old] = PIPELINE_OPTIONS[new]

# Fan out F_tests and F_early master switches for individual interventions
for prefix in ("run_continuation", "run_diverse"):
    for suffix in ("F_attenuate", "F_mix", "F_transplant"):
        PIPELINE_OPTIONS[f"{prefix}_{suffix}"] = PIPELINE_OPTIONS.get(f"{prefix}_F_tests", False)
    for suffix in ("F_early_attenuate", "F_early_mix", "F_early_transplant"):
        PIPELINE_OPTIONS[f"{prefix}_{suffix}"] = PIPELINE_OPTIONS.get(f"{prefix}_F_early", False)

# Print summary of per-prompt-set configuration
if VERBOSE:
    _specb_pds = CONTINUATION_SPECB_OVERRIDES.get("PDS_scrambles")
    _specb_f = CONTINUATION_SPECB_OVERRIDES.get("F_tests")
    _div_pds = CONTINUATION_DIVERSE_OVERRIDES.get("PDS_scrambles")
    _div_f = CONTINUATION_DIVERSE_OVERRIDES.get("F_tests")
    print(f"Continuation config: SpecB(PDS={_specb_pds}, F={_specb_f})  Diverse(PDS={_div_pds}, F={_div_f})")

# ─────────────────────────────────────────
# 3C. Model Registry
# ─────────────────────────────────────────
MODEL_REGISTRY = {
    # Small
    "phi3_mini": "microsoft/Phi-3-mini-4k-instruct",
    "gemma_2b": "google/gemma-2-2b-it",
    "llama_1b": "meta-llama/Llama-3.2-1B-Instruct",
    "llama_3b": "meta-llama/Llama-3.2-3B-Instruct",
    # Medium
    "gemma_9b": "google/gemma-2-9b-it",
    "llama_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "qwen_7b": "Qwen/Qwen2.5-7B-Instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    # Large
    "gemma_27b": "google/gemma-2-27b-it",
    "qwen_14b": "Qwen/Qwen2.5-14B-Instruct",
    "qwen_32b": "Qwen/Qwen2.5-32B-Instruct",
    "llama_70b": "meta-llama/Llama-3.3-70B-Instruct",
    "qwen_72b": "Qwen/Qwen2.5-72B-Instruct",
    "mixtral_8x7b": "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "gpt_oss_20b": "openai/gpt-oss-20b",
    # XL
    "llama_405b": "meta-llama/Llama-3.1-405B-Instruct",
    "llama_405b_fp8": "meta-llama/Llama-3.1-405B-Instruct-FP8",
    "gpt_oss_120b": "openai/gpt-oss-120b",
}

MODEL_SIZES_GB = {
    "phi3_mini": 8, "gemma_2b": 5, "llama_1b": 3, "llama_3b": 7,
    "gemma_9b": 19, "llama_8b": 16, "qwen_7b": 15, "mistral_7b": 15,
    "gemma_27b": 55, "qwen_14b": 29, "qwen_32b": 65, "llama_70b": 140,
    "qwen_72b": 145, "mixtral_8x7b": 94, "llama_405b": 810, "llama_405b_fp8": 450, 
    "gpt_oss_120b": 65, "gpt_oss_20b": 14,    # MXFP4 MoE weights + BF16 attn/embed    
}



# =================================
# MODEL SETUP FUNCTION
# =================================
def setup_model_run(model_key):
    """Configure globals for a specific model run."""
    global MODEL_ID, MODEL_KEY, MODEL_OUTPUT_DIR
    global SPECB2_MODEL_OUTPUT_DIR, DIVERSE_MODEL_OUTPUT_DIR
    global CACHE_NPZ, CACHE_META
    global OUTPUT_FILES, OUTPUT_FILES_SPECB, OUTPUT_FILES_COMPARISON
    global OUTPUT_FILES_DIVERSE_GROUP, OUTPUT_FILES_DIVERSE_REGIME
    global USE_4BIT_QUANTIZATION
    
    # Use notebook-level registry (immune to Cell 13 import overwrite)
    _reg = _NOTEBOOK_MODEL_REGISTRY if '_NOTEBOOK_MODEL_REGISTRY' in dir() or '_NOTEBOOK_MODEL_REGISTRY' in globals() else MODEL_REGISTRY
    MODEL_ID = _reg[model_key]
    MODEL_KEY = model_key
    
    MODEL_OUTPUT_DIR = OUTPUT_DIR / model_key
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    SPECB2_MODEL_OUTPUT_DIR = CONTINUATION_OUTPUT_DIR / model_key
    DIVERSE_MODEL_OUTPUT_DIR = DIVERSE_OUTPUT_DIR / model_key
    SPECB2_MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    DIVERSE_MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    CACHE_NPZ = CACHE_DIR / f"extraction_{model_key}.npz"
    CACHE_META = CACHE_DIR / f"extraction_{model_key}.meta.json"
    
    # Output files for Geometry Tests on SpecA Prompts
    OUTPUT_FILES = {
        "part_a": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_a_pr.json",
        "part_b": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_b_angles.json",
        "part_c": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_c_factors.json",
        "part_d": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_d_trajectory.json",
        "part_e": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_e_mu_landscape.json",
        "part_f": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_f_rotation.json",
        "part_g": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_g_scramble.json",
        # Multi-component Part G output (preserves V10 part_g file)
        "part_g_v11": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_g_v11_scramble.json",
        "part_g_regime_transplant": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_g_v11_regime_transplant.json",
        "part_k":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_k_cosine_discrimination.json",
        "part_l":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_l_complexity_depth_profile.json",
        "part_m_baseline":          MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_baseline_spirality.json",
        "part_m":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_spirality.json",
        "part_m_correlation":       MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_correlation.json",
        "part_h": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_h_injectivity.json",
        "per_prompt": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-per_prompt_trajectories.json",
        "token_dist": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-token_distributions.json",
        # Route resolution geometry and energy spectrum through output_files to prevent overwrite
        "resolution_geometry": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-resolution_geometry_v11.json",
        "energy_spectrum": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-energy_spectrum.json",
        "_prompt_set_label": "SpecA",
    }
    
    # Output files for Geometry Tests on SpecB Prompts
    OUTPUT_FILES_SPECB = {
        "part_a": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_a_pr.json",
        "part_b": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_b_angles.json",
        "part_c": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_c_factors.json",
        "part_d": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_d_trajectory.json",
        "part_e": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_e_mu_landscape.json",
        "part_f": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_f_rotation.json",
        "part_g": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_g_scramble.json",
        "part_g_v11": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_g_v11_scramble.json",
        "part_g_regime_transplant": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_g_v11_regime_transplant.json",
        "part_k":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_k_cosine_discrimination.json",
        "part_l":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_l_complexity_depth_profile.json",
        "part_m_baseline":          MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_m_baseline_spirality.json",
        "part_m":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_m_spirality.json",
        "part_m_correlation":       MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_m_correlation.json",
        "part_h": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_h_injectivity.json",
        "per_prompt": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-per_prompt_trajectories.json",
        "token_dist": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-token_distributions.json",
        # Route resolution geometry and energy spectrum through output_files to prevent overwrite
        "resolution_geometry": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-resolution_geometry_v11.json",
        "energy_spectrum": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-energy_spectrum.json",
        "_prompt_set_label": "SpecB",
    }
    
    # Output files for Diverse Prompt Geometry (group-level)
    OUTPUT_FILES_DIVERSE_GROUP = {
        "part_a": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_a_pr.json",
        "part_b": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_b_angles.json",
        "part_c": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_c_factors.json",
        "part_d": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_d_trajectory.json",
        "part_e": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_e_mu_landscape.json",
        "part_f": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_f_rotation.json",
        "part_g": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_g_scramble.json",
        # Multi-component Part G output (preserves V10 part_g file)
        "part_g_v11": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_g_v11_scramble.json",
        "part_g_regime_transplant": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_g_v11_regime_transplant.json",
        "part_k":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_k_cosine_discrimination.json",
        "part_l":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_l_complexity_depth_profile.json",
        "part_m_baseline":          MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_m_baseline_spirality.json",
        "part_m":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_m_spirality.json",
        "part_m_correlation":       MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_m_correlation.json",
        "part_h": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_h_injectivity.json",
        "per_prompt": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-per_prompt_trajectories.json",
        "token_dist": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-token_distributions.json",
        # Route resolution geometry and energy spectrum through output_files to prevent overwrite
        "resolution_geometry": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11.json",
        "energy_spectrum": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-energy_spectrum.json",
        "_prompt_set_label": "Diverse",
    }
    
    # Output files for Diverse Prompt Geometry (regime-level)
    OUTPUT_FILES_DIVERSE_REGIME = {
        "part_a": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_a_pr.json",
        "part_b": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_b_angles.json",
        "part_c": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_c_factors.json",
        "part_d": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_d_trajectory.json",
        "part_e": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_e_mu_landscape.json",
        "part_f": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_f_rotation.json",
        "part_g": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_g_scramble.json",
        # Multi-component Part G output (preserves V10 part_g file)
        "part_g_v11": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_g_v11_scramble.json",
        "part_h": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_h_injectivity.json",
        "per_prompt": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-per_prompt_trajectories.json",
        "token_dist": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-token_distributions.json",
        # v11.1/v11.2: Keys added to match group-level dict (prevent silent no-save)
        "part_g_regime_transplant": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_g_v11_regime_transplant.json",
        "part_k":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_k_cosine_discrimination.json",
        "part_l":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_l_complexity_depth_profile.json",
        "part_m_baseline":          MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_m_baseline_spirality.json",
        "part_m":                   MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_m_spirality.json",
        "part_m_correlation":       MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_m_correlation.json",
        "resolution_geometry": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-resolution_geometry_v11.json",
        "energy_spectrum":     MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-energy_spectrum.json",
        "_prompt_set_label": "Diverse-regime",
    }
    
    # Geometry Comparison output
    OUTPUT_FILES_COMPARISON = {
        "comparison": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-comparison.json",
        "cross_prompt_set": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-cross_prompt_set_comparison.json",
    }
    
    # Determine if 4-bit quantization is needed
    _sizes = _NOTEBOOK_MODEL_SIZES_GB if '_NOTEBOOK_MODEL_SIZES_GB' in dir() or '_NOTEBOOK_MODEL_SIZES_GB' in globals() else MODEL_SIZES_GB
    USE_4BIT_QUANTIZATION = _sizes.get(model_key, 50) > profile.get("quantize_threshold", 40)
    
    print(f"\n{'='*60}")
    print(f"CONFIGURED: {MODEL_ID}")
    print(f"Short key: {model_key}")
    print(f"4-bit quantization: {'YES' if USE_4BIT_QUANTIZATION else 'NO'}")
    print(f"Geometry output: {MODEL_OUTPUT_DIR}")
    print(f"Continuation output: {SPECB2_MODEL_OUTPUT_DIR}")
    print(f"{'='*60}")
    
    return MODEL_ID, MODEL_KEY

print("✓ All parameters configured")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Geometry parts enabled: {sum(RUN_PARTS.values())}/{len(RUN_PARTS)}")

# =================================
# NOTEBOOK-LEVEL REGISTRY (authoritative)
# =================================
# The MODEL_REGISTRY defined above is the authoritative source.
# pds_geometry.py may contain a stale copy — save ours so that
# setup_model_run() always uses the notebook version, even if
# Cell 13's import overwrites the global.
_NOTEBOOK_MODEL_REGISTRY = dict(MODEL_REGISTRY)
_NOTEBOOK_MODEL_SIZES_GB = dict(MODEL_SIZES_GB)


# ============================================================
# TEST_MODE OVERRIDES
# ============================================================
# When TEST_MODE is active, override all settings for a fast
# end-to-end verification run. These overrides come last to
# ensure they take precedence over all other configuration.
# ============================================================
if TEST_MODE:
    # Minimal geometry: only Parts A, E, G, K
    RUN_PARTS.update({
        "part_a": True, "part_b": False, "part_c": False, "part_d": False,
        "part_e": True, "part_f": False, "part_g": True, "part_h": False,
        "part_i": False, "part_j": False, "part_k": True, "part_l": False,
        "part_m": False, "part_m_intervention": False,
    })
    # Reduce prompt counts
    N_SCRAMBLE_PROMPTS = 4
    SPECB_N_SCRAMBLE_PROMPTS = 4
    DIVERSE_N_SCRAMBLE_PROMPTS = 4
    # Reduce generation
    SPECB2_N_GENERATE = 16
    N_SCRAMBLES = 1
    # Disable expensive transplant types
    GEOMETRY_INTERVENTION_TYPES = ["attenuate", "rotation", "mix"]
    # Limit prompt sets
    GEOMETRY_PROMPT_SETS.update({"SpecB": False, "Diverse_group": False, "Diverse_regime": False})
    CONTINUATION_PROMPT_SETS.update({"Diverse": False})
    # Disable F-early and diverse continuation
    CONTINUATION_F_TESTS.update({"F_early": False})
    CONTINUATION_DIVERSE_OVERRIDES.update({"PDS_scrambles": False, "F_tests": False})
    # Rebuild PIPELINE_OPTIONS with overrides
    PIPELINE_OPTIONS["run_geometry_on_continuation_prompts"] = False
    PIPELINE_OPTIONS["run_geometry_on_diverse_prompts"] = False
    PIPELINE_OPTIONS["run_geometry_part_g_specB"] = False
    PIPELINE_OPTIONS["run_geometry_part_g_diverse"] = False
    PIPELINE_OPTIONS["run_geometry_part_m"] = False
    PIPELINE_OPTIONS["run_geometry_part_m_intervention"] = False
    PIPELINE_OPTIONS["run_diverse_continuation_tests"] = False
    PIPELINE_OPTIONS["run_continuation_F_early"] = False
    for prefix in ("run_continuation", "run_diverse"):
        for suffix in ("F_early_attenuate", "F_early_mix", "F_early_transplant"):
            PIPELINE_OPTIONS[f"{prefix}_{suffix}"] = False
        PIPELINE_OPTIONS[f"{prefix}_F_tests"] = prefix == "run_continuation"
        for suffix in ("F_attenuate", "F_mix", "F_transplant"):
            PIPELINE_OPTIONS[f"{prefix}_{suffix}"] = prefix == "run_continuation"
    print("⚡ TEST_MODE: Reduced to core geometry (A,E,G,K) + basic continuation")
    print(f"  Prompts: {N_SCRAMBLE_PROMPTS} SpecA, {SPECB_N_SCRAMBLE_PROMPTS} SpecB")
    print(f"  Generation: {SPECB2_N_GENERATE} tokens, {N_SCRAMBLES} scramble(s)")





In [ ]:
# GPU ENVIRONMENT DETECTION
# ============================================================
# Auto-detects GPU configuration and sets memory profile.
# No user edits needed.
# ============================================================

import torch

print("=" * 60)
print("GPU ENVIRONMENT DETECTION")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available! This notebook requires GPU.")

n_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {n_gpus}")

total_vram = 0
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024**3)
    total_vram += vram_gb
    print(f"  GPU {i}: {props.name} ({vram_gb:.1f} GB)")

print(f"\nTotal VRAM: {total_vram:.1f} GB")

# Set memory profile based on available VRAM
if total_vram >= 160:
    profile = {"name": "multi_h100", "quantize_threshold": 200, "batch_size": 32}
elif total_vram >= 80:
    profile = {"name": "single_h100", "quantize_threshold": 80, "batch_size": 16}
elif total_vram >= 48:
    profile = {"name": "a6000", "quantize_threshold": 45, "batch_size": 8}
elif total_vram >= 24:
    profile = {"name": "rtx4090", "quantize_threshold": 25, "batch_size": 4}
else:
    profile = {"name": "low_vram", "quantize_threshold": 15, "batch_size": 2}

print(f"\nMemory profile: {profile['name']}")
print(f"  Quantize models > {profile['quantize_threshold']} GB")
print(f"  Batch size: {profile['batch_size']}")
print("\n✓ GPU environment configured")


---
# Section 2: Load Modules

Load the analysis modules from the workspace scripts.

In [ ]:
# ============================================================
# MODULE IMPORTS
# ============================================================
# Module roles:
#   specA_analysis.py — Geometry analysis (Parts A-H)
#     Contains: PDSF math utilities (pca_basis, participation_ratio,
#     project_onto_basis, principal_angles), data structures
#     (LayerTensors, ModelExtraction), and all Part A-H functions.
#     This module's pca_basis returns a single ndarray (not a tuple).
#
#   pds_geometry.py — Model loading, extraction, Part G scramble
#     Contains: MODEL_REGISTRY, ModelBundle, load_model_bundle_multigpu,
#     extract_hidden_states, run_scramble_experiment, etc.
#
#   pds_continuation.py — SpecB continuation/generation experiments
#     Contains: PromptRow, SpecB2Bases, run_specB2_experiment, etc.
#     Note: this module's pca_basis returns (basis, singular_values) tuple.
#
#   pds_prompt_analysis.py — Optional token distribution analysis
#     Compares prediction sharpness between SpecA and SpecB prompts.
# ============================================================

# Load modules
import importlib
import sys

# Ensure workspace is in path
if "/workspace" not in sys.path:
    sys.path.insert(0, "/workspace")

# Import core modules
import pds_geometry
import specA_analysis

# Reload to pick up changes
importlib.reload(pds_geometry)
importlib.reload(specA_analysis)

# Import from pds_geometry (core utilities + model registry)
from pds_geometry import (
    # Model registry
    MODEL_REGISTRY,
    MODEL_SIZES_GB,
    MODEL_FAMILIES,
    # Core utilities
    ModelBundle,
    load_model_bundle_multigpu,
    extract_hidden_states,
    get_model_input_device,
    get_model_layers,
    cleanup_gpu_memory,
    clear_model_from_memory,
    clear_hf_cache_for_model,
    clear_all_hf_cache,
    get_cache_usage,
    print_cache_status,
    apply_chat_template_if_needed,
    # Domain mappings for regime-aware F transplant (Part G v5.1)
    SPECA_GROUP_TO_DOMAIN,
    SPECB_GROUP_TO_DOMAIN,
    NULL_DONOR_IDX,
)

# Import from specA_analysis (geometry analysis)
from specA_analysis import (
    LayerTensors,
    ModelExtraction,
    get_unembed_weight,
    participation_ratio,
    pca_basis,
    project_onto_basis,
    build_P_basis_from_expected_tokens,
    compute_part_A_PR,
    compute_part_B_family_separation,
    compute_part_C_factor_effects,
    compute_part_E_mu_landscape,
    compute_part_F_rotation_with_per_prompt,
    compute_part_H_injectivity,
)

# Import from pds_geometry (Part G scramble functions)
from pds_geometry import (
    run_invariance_control,
    run_scramble_experiment,
    resolve_scramble_layers,
)

# Try to import SpecB2 utilities
try:
    import pds_continuation
    importlib.reload(pds_continuation)
    from pds_continuation import (
        PromptRow,
        SpecB2Bases,
        compute_P_bases_from_predictions,
        BaselineResult,
        compute_specB2_bases,
        generate_baselines_with_decomposition,
        run_specB2_experiment,
        run_random_control_experiment,
        save_specB2_bases,
        save_specB2_baselines,
        load_specB2_baselines,
        save_specB2_results,
        load_prompts_from_json,
        run_specB2_for_pipeline,
    )
    SPECB2_AVAILABLE = True
    print("✓ pds_continuation loaded (continuation experiments)")
except ImportError as e:
    SPECB2_AVAILABLE = False
    print(f"⚠️  pds_continuation not available: {e}")

# Import spirality measures (Part M)
try:
    import pds_spirality
    importlib.reload(pds_spirality)
    from pds_spirality import (
        compute_spirality_profile,
        compute_spirality_disruption,
        compute_baseline_spirality_from_cache,
        correlate_spirality_with_rotation,
        correlate_spirality_disruption_with_part_g,
        aggregate_spirality_results,
        aggregate_spirality_disruptions,
    )
    print("✓ pds_spirality loaded (Part M spirality measures)")
except ImportError as e:
    pds_spirality = None
    print(f"⚠️  pds_spirality not available: {e}")
    print("   Continuation experiments will be skipped")

# Alias for backward compatibility
get_unembed_weight = get_unembed_weight  # from specA_analysis

import numpy as np
from tqdm import tqdm

print("\n✓ All modules loaded")
print("  specA_analysis: geometry analysis (Parts A-H)")
print("  pds_geometry: model loading, extraction, scramble")
print("  pds_continuation: continuation scramble experiments")
print("  pds_spirality: Part M spirality measures")

# Note: setup_model_run() is called automatically by run_full_pipeline() for each model in MODELS_TO_RUN


---
# Section 3: Load Data

Load prompt data for Geometry and Continuation tests.

In [ ]:
#
# which is critical for Part C (compute_part_C_factor_effects).
# The **{k:v} splat in meta.append() copies A, B, C, D factor values into meta,
# Parse SpecA prompts from JSON into flat lists. Each of the 14 groups × 16 variants.
# group shares the same expected answer token. The 16 variants span 4 binary
# controlled surface-form factors (paraphrase, constraint, clutter, format).
# Each variant also carries its factor levels:
#   A=0, B=1, C=0, D=1 — these integer values are preserved
# in the meta dict so that Part C can compute factor effects.
# ============================================================
# LOAD SPECA PROMPTS
# ============================================================

print("Loading Geometry prompts...")

if not GEOMETRY_PROMPTS_PATH.exists():
    raise FileNotFoundError(f"Geometry prompts not found: {GEOMETRY_PROMPTS_PATH}")

with open(GEOMETRY_PROMPTS_PATH) as f:
    geometry_prompts_raw = json.load(f)

# The Geometry prompts file has structure:
# {
#   "groups": [
#     {
#       "group_id": "G01_...",
#       "expected_next_token": "Yes",
#       "variants": [
#         {"variant_id": "...", "prompt": "..."},
#         ...
#       ]
#     },
#     ...
#   ]
# }

# Parse into flat lists
prompts = []
meta = []
group_ids = []
variant_ids = []
expected_tokens = []

if "groups" in geometry_prompts_raw:
    # New format: nested groups with variants
    for group in geometry_prompts_raw["groups"]:
        gid = group["group_id"]
        expected = group.get("expected_next_token", group.get("expected", ""))
        
        for variant in group["variants"]:
            prompts.append(variant["prompt"])
            group_ids.append(gid)
            variant_ids.append(variant["variant_id"])
            expected_tokens.append(expected)
            meta.append({
                "group_id": gid,
                "variant_id": variant["variant_id"],
                "prompt": variant["prompt"],
                "expected_token": expected,
                **{k: v for k, v in variant.items() if k not in ["prompt", "variant_id"]}
            })
elif isinstance(geometry_prompts_raw, list):
    # Old format: flat list of prompts
    for item in geometry_prompts_raw:
        prompts.append(item["prompt"])
        meta.append(item)
        group_ids.append(item["group_id"])
        variant_ids.append(item["variant_id"])
        expected_tokens.append(item.get("expected_token", item.get("expected", "")))
else:
    raise ValueError(f"Unexpected SpecA JSON structure. Keys: {list(geometry_prompts_raw.keys()) if isinstance(geometry_prompts_raw, dict) else 'N/A'}")

print(f"  ✓ Loaded {len(prompts)} Geometry prompts")
print(f"    Groups: {len(set(group_ids))}")
print(f"    Variants per group: {len(prompts) // len(set(group_ids))}")

# LOAD SPECB PROMPTS
# ============================================================

print("\nLoading Continuation prompts...")

continuation_prompts_data = None
if CONTINUATION_PROMPTS_PATH.exists():
    try:
        with open(CONTINUATION_PROMPTS_PATH) as f:
            continuation_prompts_data = json.load(f)
        n_groups = len(continuation_prompts_data.get("groups", []))
        n_prompts = sum(len(g.get("variants", [])) for g in continuation_prompts_data.get("groups", []))
        print(f"  ✓ Loaded {n_prompts} Continuation prompts from {n_groups} groups")
    except Exception as e:
        print(f"  ⚠️  Failed to load Continuation data: {e}")
else:
    print(f"  ⚠️  Continuation prompts not found: {CONTINUATION_PROMPTS_PATH}")

print("\n" + "=" * 60)


# ============================================================
# LOAD DIVERSE PROMPTS
# ============================================================

print("\nLoading Diverse prompts...")

diverse_prompts_data = None
if DIVERSE_PROMPTS_PATH.exists():
    try:
        with open(DIVERSE_PROMPTS_PATH) as f:
            diverse_prompts_data = json.load(f)
        
        if "prompts" in diverse_prompts_data:
            n_diverse = len(diverse_prompts_data["prompts"])
            n_diverse_groups = len(set(p["group_id"] for p in diverse_prompts_data["prompts"]))
            regimes = sorted(set(p.get("regime", "") for p in diverse_prompts_data["prompts"]))
            print(f"  ✓ Loaded {n_diverse} Diverse prompts from {n_diverse_groups} groups, {len(regimes)} regimes")
            print(f"    Regimes: {', '.join(regimes)}")
            
            # Check variant counts
            from collections import Counter
            group_counts = Counter(p["group_id"] for p in diverse_prompts_data["prompts"])
            min_v = min(group_counts.values())
            max_v = max(group_counts.values())
            if min_v < 8:
                print(f"    ⚠ {min_v}-{max_v} variants per group (D basis will be rank ≤ {min_v - 1})")
        else:
            n_diverse = sum(len(g.get("variants", [])) for g in diverse_prompts_data.get("groups", []))
            print(f"  ✓ Loaded {n_diverse} Diverse prompts (old format)")
    except Exception as e:
        print(f"  ⚠️  Failed to load Diverse data: {e}")
else:
    print(f"  ⚠️  Diverse prompts not found: {DIVERSE_PROMPTS_PATH}")

print("\n" + "=" * 60)


# Backwards compatibility aliases
speca_raw = geometry_prompts_raw
specb_data = continuation_prompts_data


---
# Section 4: Prompt Analysis Utilities (Optional)

Token distribution analysis tools. These run automatically in the pipeline.

In [ ]:
# === PROMPT ANALYSIS UTILITIES ===
# Import the prompt analysis module

try:
    import pds_prompt_analysis
    importlib.reload(pds_prompt_analysis)
    from pds_prompt_analysis import (
        compute_token_distribution,
        compute_prompt_set_distributions,
        compare_prompt_set_distributions,
        plot_distribution_comparison_figure,
        plot_cumulative_probability_curves,
        run_specA_analysis_on_prompts,
        run_prompt_comparison_analysis,
        save_distribution_stats,
    )
    PROMPT_ANALYSIS_AVAILABLE = True
    print("✓ prompt_analysis_utils loaded")
except Exception as e:
    PROMPT_ANALYSIS_AVAILABLE = False
    print(f"⚠️  prompt_analysis_utils not available: {e}")

In [ ]:
# === MANUAL PROMPT ANALYSIS (Optional) ===
# NOTE: Prompt analysis is now integrated into the main pipeline!
# This cell is for manual/standalone analysis if needed.
#
# To run in pipeline: Set "run_prompt_analysis": True in PIPELINE_OPTIONS
# 
# To run standalone (after loading a model):
#   1. Set RUN_PROMPT_ANALYSIS_MANUAL = True below
#   2. Run this cell

RUN_PROMPT_ANALYSIS_MANUAL = False

if RUN_PROMPT_ANALYSIS_MANUAL and PROMPT_ANALYSIS_AVAILABLE:
    print("Running manual prompt analysis...")
    print("(This is normally done in the pipeline - see PIPELINE_OPTIONS)")
    # ... manual analysis code would go here if needed ...
else:
    print("Manual prompt analysis disabled.")
    print("Prompt analysis runs automatically in the pipeline when run_prompt_analysis=True")

In [ ]:
# === MANUAL SPECA-STYLE ON SPECB (Optional) ===
# NOTE: This analysis is now integrated into the main pipeline!
# This cell is for manual/standalone analysis if needed.
#
# To run in pipeline: Set "run_specA_style_on_specB": True in PIPELINE_OPTIONS
#
# To run standalone (after loading a model):
#   1. Set RUN_SPECA_ON_SPECB_MANUAL = True below
#   2. Run this cell

RUN_SPECA_ON_SPECB_MANUAL = False

if RUN_SPECA_ON_SPECB_MANUAL and PROMPT_ANALYSIS_AVAILABLE:
    print("Running manual SpecA-style analysis on SpecB prompts...")
    print("(This is normally done in the pipeline - see PIPELINE_OPTIONS)")
    # ... manual analysis code would go here if needed ...
else:
    print("Manual SpecA-on-SpecB analysis disabled.")
    print("This runs automatically in the pipeline when run_specA_style_on_specB=True")

---
# Section 5: Pipeline Functions

Define the unified pipeline for Geometry + Continuation tests.

In [ ]:
# ============================================================
# SPECA ANALYSIS FUNCTIONS (Parts A-H)
# Core functions imported from specA_analysis.py (recovered original)
# ============================================================
# Additional analysis: Energy Spectrum (added v6)
# ============================================================

def compute_energy_spectrum(
    extraction,
    P_bases,
    layers=None,
    n_dims=100,
):
    """
    Compute energy (singular value squared) distribution across dimensions
    in P-perp space, at three levels of granularity.
    
    PURPOSE:
    Shows how concentrated the representational energy is in P⊥ space.
    A steep dropoff means a low-dimensional structure (good for the PDSF
    framework — most information is captured by a small D basis).
    A long tail means information is spread across many dimensions.
    
    THREE VIEWS:
    1. Global: All 224 prompts → up to 223 singular values. Shows the
       full energy tail, including 100+ dimensions past the PDSF subspaces.
    2. Per-group: 16 variants per group → 15 singular values max.
       Shows how concentrated within-group variation is.
    3. Per-family: Groups pooled by answer token (e.g., all "Yes" groups
       → 112 prompts → 111 singular values). Middle ground between
       group-level detail and global sample size.
    
    KEY OUTPUT FIELDS:
    - energy_frac[i]: fraction of total variance in the i-th singular direction
    - cumulative[i]: fraction of variance captured by the top-(i+1) directions
    - k_D: the PR-adaptive rank used by the pipeline's D basis
    - energy_at_k_D: cumulative energy captured at rank k_D
    
    COMPUTE COST: ~50ms total (SVD of pre-extracted matrices, no GPU needed).
    
    Args:
        extraction: ModelExtraction with .layers, .group_ids, .meta_rows
        P_bases: Dict[layer][group_id] → P basis matrix
        layers: Which layers to analyze (default: all)
        n_dims: How many singular values to report (default: 100)
    
    Returns:
        Dict with by_layer → layer → {global, by_group, by_family}
    """
    if layers is None:
        layers = sorted(extraction.layers.keys())
    
    # Build group -> answer-family mapping from meta_rows
    group_to_family = {}
    for m in extraction.meta_rows:
        gid = m.get("group_id", "")
        tok = m.get("expected_token", "")
        if gid and tok:
            group_to_family[gid] = tok
    
    # Group index
    group_to_idx = {}
    for i, g in enumerate(extraction.group_ids):
        group_to_idx.setdefault(g, []).append(i)
    
    # Family index (pool groups by answer token)
    family_to_idx = {}
    for g, idxs in group_to_idx.items():
        fam = group_to_family.get(g, "unknown")
        family_to_idx.setdefault(fam, []).extend(idxs)
    
    def _spectrum(X, n_dims):
        """Compute energy spectrum for a data matrix."""
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        Xc = X - X.mean(axis=0, keepdims=True)
        pr = participation_ratio(X, center=True)
        k_D = max(1, min(int(round(pr)), 64, X.shape[1],
                         X.shape[0] - 1 if X.shape[0] > 1 else 1))
        try:
            s = np.linalg.svd(Xc, compute_uv=False, full_matrices=False)
        except np.linalg.LinAlgError:
            return None
        energy = s ** 2
        total = float(energy.sum())
        if total < 1e-12:
            return None
        n_report = min(n_dims, len(s))
        efrac = (energy[:n_report] / total)
        cumul = np.cumsum(efrac)
        return {
            "n_samples": int(X.shape[0]),
            "n_dims_available": int(len(s)),
            "n_dims_reported": n_report,
            "k_D": int(k_D),
            "PR": round(float(pr), 2),
            "energy_frac": [round(float(x), 6) for x in efrac],
            "cumulative": [round(float(x), 6) for x in cumul],
            "energy_at_k_D": round(float(cumul[min(k_D - 1, len(cumul) - 1)]), 4),
            "top_10_energy": round(float(cumul[min(9, len(cumul) - 1)]), 4),
            "top_50_energy": round(float(cumul[min(49, len(cumul) - 1)]), 4) if len(cumul) >= 50 else None,
        }
    
    result = {"by_layer": {}}
    
    for layer in layers:
        H = extraction.layers[layer].H
        
        # Build H_perp (project out per-group P)
        HP = np.zeros_like(H)
        for i, g in enumerate(extraction.group_ids):
            Bp = P_bases[layer].get(i)
            if Bp is not None and Bp.size > 0:
                HP[i:i+1] = project_onto_basis(H[i:i+1], Bp)
        H_perp = H - HP
        
        layer_result = {}
        
        # Global (all 224 prompts → up to 223 dims)
        layer_result["global"] = _spectrum(H_perp, n_dims)
        
        # Per group (16 variants → up to 15 dims each)
        layer_result["by_group"] = {}
        for g, idxs in sorted(group_to_idx.items()):
            spec = _spectrum(H_perp[idxs], min(n_dims, len(idxs) - 1))
            if spec is not None:
                layer_result["by_group"][g] = spec
        
        # Per answer family (pooled groups → more dims)
        layer_result["by_family"] = {}
        for fam, idxs in sorted(family_to_idx.items()):
            spec = _spectrum(H_perp[idxs], n_dims)
            if spec is not None:
                layer_result["by_family"][fam] = spec
        
        result["by_layer"][str(layer)] = layer_result
    
    return result





def compute_resolution_geometry(
    extraction,
    P_bases,
    unembed_weight,
    layers=None,
    k_max=64,
    s_small_r=16,
    k_neighbors=8,
):
    """
    Resolution Geometry: Manifold curvature and information density per PDSF subspace.
    
    v11.1: Full-trajectory, PR-adaptive PDSF decomposition.
    
    Changes from v11:
    - Runs at ALL layers by default (not 5 representative), giving full depth trajectories.
    - P basis is PR-adaptive: at each layer, collect per-prompt rank-1 P vectors,
      PCA them, and truncate to the effective dimensionality (participation ratio).
      This replaces the "all unique token directions" envelope, which was the total
      span rather than the effective dimensionality.
    - D and S are recomputed at each layer from the layer-specific P residual,
      maintaining the sequential orthogonal projection P → D → S → F.
    
    THEORETICAL MOTIVATION (Resolution Hypothesis):
    The model's representation can be viewed as a landscape centered on the
    predicted token. Near the prediction peak (P subspace), the landscape is
    sharply peaked and highly resolved — small representational differences
    correspond to meaningful prediction differences. Further from the peak
    (D, then S, then residual), the landscape should become smoother
    and broader — the model makes coarser distinctions in these regions.
    
    PREDICTION: curvature_ratio should increase P → D → S → F.
    P < D is expected because P captures the sharpest prediction-proximal
    structure. D < S < F is universal; the P < D boundary is prompt-set
    dependent (holds on SpecA/SpecB, reverses on Diverse due to
    between-regime D structure).
    
    WHAT WE MEASURE per subspace:
    1. Curvature ratio: mean_kNN_distance / mean_global_distance
    2. Local PR: mean participation ratio of kNN neighborhoods
    3. Global PR: participation ratio of the full projection
    4. Info density: total variance / subspace rank
    5. P-gradient (P only): softmax sensitivity
    
    Args:
        extraction: ModelExtraction with .layers, .group_ids, .meta_rows
        P_bases: Dict[layer][prompt_idx] -> P basis (rank-1 per prompt)
        unembed_weight: Unembedding matrix (vocab × hidden_dim)
        layers: Which layers (default: ALL layers for full trajectory)
        k_max: Max D rank for PCA truncation
        s_small_r: S basis rank
        k_neighbors: Number of nearest neighbors for curvature estimate
    
    Returns:
        Dict with by_layer -> layer -> {
          subspaces: {P, D, S, residual} -> curvature + density metrics,
          summary: comparison across subspaces,
          basis_ranks: {P, D, S, residual} -> rank at this layer
        }
    """
    from pds_continuation import (
        compute_D_basis_global as _compute_D_canon,
        compute_S_basis_global as _compute_S_canon,
    )
    
    if layers is None:
        layers = sorted(extraction.layers.keys())
    
    # Prepare unembedding for P-gradient
    W = unembed_weight
    if W.shape[0] < W.shape[1]:
        W = W.T  # ensure (vocab, hidden_dim)
    
    result = {"by_layer": {}}
    
    def _subspace_geometry(X_proj, k_nn, label):
        """
        Compute manifold geometry metrics for prompts projected into a subspace.
        X_proj: (n_prompts, subspace_rank) — coordinates in the subspace.
        """
        n, rank = X_proj.shape
        if n < 3 or rank == 0:
            return None
        
        dists = squareform(pdist(X_proj, metric="euclidean"))
        np.fill_diagonal(dists, np.inf)
        
        k_nn_actual = min(k_nn, n - 1)
        knn_dists = np.sort(dists, axis=1)[:, :k_nn_actual]
        mean_knn = float(knn_dists.mean())
        
        np.fill_diagonal(dists, 0)
        triu = dists[np.triu_indices(n, k=1)]
        mean_global = float(triu.mean())
        
        curvature_ratio = mean_knn / max(mean_global, 1e-12)
        
        local_prs = []
        for idx in range(n):
            nn_idx = np.argsort(dists[idx])[:k_nn_actual]
            local_patch = X_proj[nn_idx]
            if local_patch.shape[0] >= 3:
                pr_local = participation_ratio(local_patch, center=True)
                local_prs.append(pr_local)
        mean_local_pr = float(np.mean(local_prs)) if local_prs else float("nan")
        
        global_pr = participation_ratio(X_proj, center=True)
        total_var = float(np.var(X_proj, axis=0).sum())
        info_density = total_var / max(rank, 1)
        
        return {
            "rank": rank,
            "n_prompts": n,
            "curvature_ratio": round(curvature_ratio, 4),
            "mean_knn_dist": round(mean_knn, 4),
            "mean_global_dist": round(mean_global, 4),
            "local_pr": round(mean_local_pr, 2),
            "global_pr": round(global_pr, 2),
            "pr_folding_ratio": round(mean_local_pr / max(global_pr, 0.01), 3),
            "info_density": round(info_density, 4),
            "total_variance": round(total_var, 4),
        }
    
    try:
        from tqdm import tqdm as _tqdm
        _layer_iter = _tqdm(layers, desc="Resolution geometry", unit="layer")
    except ImportError:
        _layer_iter = layers
    
    for layer in _layer_iter:
        H = extraction.layers[layer].H
        n_prompts, d = H.shape
        
        # === Step 1: Build per-prompt P projections ===
        HP = np.zeros_like(H)
        for idx in range(n_prompts):
            Bp = P_bases[layer].get(idx)
            if Bp is not None and Bp.size > 0:
                HP[idx:idx+1] = project_onto_basis(H[idx:idx+1], Bp)
        H_perp = H - HP
        
        # === Step 2: PR-adaptive P basis ===
        # Collect all per-prompt P projection vectors (rank-1 each, but in full d-space)
        # Stack them and PCA to find the effective P subspace at this layer.
        P_proj_vectors = []
        for idx in range(n_prompts):
            Bp = P_bases[layer].get(idx)
            if Bp is not None and Bp.size > 0:
                # The P projection of this prompt: a vector in d-space
                P_proj_vectors.append(HP[idx])
        
        if len(P_proj_vectors) >= 3:
            P_proj_matrix = np.stack(P_proj_vectors, axis=0)  # (n_with_P, d)
            # PCA of the P projections to find their effective subspace
            P_proj_centered = P_proj_matrix - P_proj_matrix.mean(axis=0, keepdims=True)
            # Participation ratio determines effective rank
            p_pr = participation_ratio(P_proj_matrix, center=True)
            p_rank = max(1, min(int(round(p_pr)), len(P_proj_vectors) - 1, d))
            # Get top-k PCA directions
            P_basis = pca_basis(P_proj_matrix, k=p_rank, center=True)  # (d, p_rank)
            # Project ALL prompts (including those without P) onto this basis
            P_coords = H @ P_basis  # (n_prompts, p_rank)
        else:
            p_rank = 1
            p_pr = 1.0
            P_coords = np.zeros((n_prompts, 1), dtype=np.float32)
        
        # === Step 3: D basis from H_perp (recomputed at this layer) ===
        Bd, D_info = _compute_D_canon(H, P_bases[layer], k_D=min(k_max, 12))
        k_d = D_info["k"]
        
        # === Step 4: S basis from double-residual (recomputed at this layer) ===
        Bs, S_info = _compute_S_canon(H, P_bases[layer], Bd, k_S=s_small_r)
        
        # === Step 5: Project into subspaces ===
        D_coords = H_perp @ Bd  # (n, k_d)
        S_coords = H_perp @ Bs  # (n, k_s)
        
        # Residual: H_perp minus D and S components
        H_d_component = D_coords @ Bd.T
        H_s_component = S_coords @ Bs.T
        H_residual = H_perp - H_d_component - H_s_component
        res_rank = min(50, H_residual.shape[0] - 1, H_residual.shape[1])
        if res_rank > 0:
            Res_coords = H_residual @ pca_basis(H_residual, k=res_rank, center=True)
        else:
            Res_coords = np.zeros((n_prompts, 1), dtype=np.float32)
        
        # === Step 6: Compute geometry for each subspace ===
        subspaces = {}
        for name, coords in [("P", P_coords), ("D", D_coords), ("S", S_coords), ("residual", Res_coords)]:
            geo = _subspace_geometry(coords, k_neighbors, name)
            if geo is not None:
                subspaces[name] = geo
        
        # P also gets softmax gradient
        if "P" not in subspaces:
            subspaces["P"] = {"rank": p_rank, "n_prompts": n_prompts}
        subspaces["P"]["pr_adaptive_rank"] = p_rank
        subspaces["P"]["pr_raw"] = round(float(p_pr), 2)
        
        p_grads = []
        for idx in range(n_prompts):
            Bp = P_bases[layer].get(idx)
            if Bp is not None and Bp.shape[1] > 0:
                p_dir = Bp[:, 0]
                logit_grad = W @ p_dir
                logits = W @ H[idx]
                logits_shifted = logits - logits.max()
                probs = np.exp(logits_shifted) / np.exp(logits_shifted).sum()
                weighted_grad = float(np.sqrt((probs * logit_grad**2).sum() - (probs * logit_grad).sum()**2))
                p_grads.append(weighted_grad)
        subspaces["P"]["softmax_gradient_mean"] = round(float(np.mean(p_grads)), 4) if p_grads else None
        subspaces["P"]["softmax_gradient_std"] = round(float(np.std(p_grads)), 4) if p_grads else None
        
        # === Step 7: Summary and ordering ===
        summary = {}
        for name in ["P", "D", "S", "residual"]:
            if name in subspaces and "curvature_ratio" in subspaces[name]:
                summary[f"{name}_curvature_ratio"] = subspaces[name]["curvature_ratio"]
                summary[f"{name}_info_density"] = subspaces[name]["info_density"]
                summary[f"{name}_local_pr"] = subspaces[name]["local_pr"]
                summary[f"{name}_global_pr"] = subspaces[name]["global_pr"]
                summary[f"{name}_pr_folding"] = subspaces[name]["pr_folding_ratio"]
        if "P" in subspaces and subspaces["P"].get("softmax_gradient_mean") is not None:
            summary["P_softmax_gradient"] = subspaces["P"]["softmax_gradient_mean"]
        
        # Legacy P→D→S→F ordering test (backward compat)
        ratios_PDSF = []
        for name in ["P", "D", "S", "residual"]:
            if name in subspaces and "curvature_ratio" in subspaces[name]:
                ratios_PDSF.append((name, subspaces[name]["curvature_ratio"]))
        summary["curvature_ordering"] = [(n, round(r, 4)) for n, r in ratios_PDSF]
        summary["ordering_monotonic"] = all(
            ratios_PDSF[i][1] <= ratios_PDSF[i+1][1] for i in range(len(ratios_PDSF)-1)
        ) if len(ratios_PDSF) > 1 else None
        
        # Canonical ordering test: P < D < S < F (curvature_ratio)
        P_curv = subspaces.get("P", {}).get("curvature_ratio")
        D_curv = subspaces.get("D", {}).get("curvature_ratio")
        S_curv = subspaces.get("S", {}).get("curvature_ratio")
        F_curv = subspaces.get("residual", {}).get("curvature_ratio")
        summary["P_curvature_ratio"] = round(P_curv, 4) if P_curv is not None else None
        summary["D_curvature_ratio"] = round(D_curv, 4) if D_curv is not None else None
        summary["S_curvature_ratio"] = round(S_curv, 4) if S_curv is not None else None
        summary["residual_curvature_ratio"] = round(F_curv, 4) if F_curv is not None else None
        pdsf = [P_curv, D_curv, S_curv, F_curv]
        pdsf_valid = [v for v in pdsf if v is not None]
        summary["ordering_PDSF"] = ["P", "D", "S", "F"]
        summary["ordering_PDSF_values"] = [round(v, 4) for v in pdsf_valid]
        summary["ordering_PDSF_monotonic"] = all(
            pdsf_valid[i] < pdsf_valid[i+1] for i in range(len(pdsf_valid)-1)
        ) if len(pdsf_valid) > 1 else None
        # Pairwise checks
        summary["P_lt_D"] = (P_curv < D_curv) if (P_curv is not None and D_curv is not None) else None
        summary["D_lt_S"] = (D_curv < S_curv) if (D_curv is not None and S_curv is not None) else None
        summary["P_lt_S"] = (P_curv < S_curv) if (P_curv is not None and S_curv is not None) else None
        summary["S_lt_F"] = (S_curv < F_curv) if (S_curv is not None and F_curv is not None) else None
        
        # === Step 8: P-vs-PCA alignment check ===
        # How much do the PR-adaptive P directions overlap with the top PCA of H?
        # If they align strongly, the PCA control is less independent from PDSF.
        H_centered = H - H.mean(axis=0, keepdims=True)
        H_pca_basis = pca_basis(H, k=min(50, n_prompts - 1), center=True)  # (d, 50)
        
        if len(P_proj_vectors) >= 3 and p_rank >= 1:
            # Principal angles between P basis and top-k PCA of H
            # cos(angle) = singular values of P_basis.T @ H_pca_top_k
            H_pca_top_p = H_pca_basis[:, :p_rank]  # same rank as P
            cross = P_basis.T @ H_pca_top_p  # (p_rank, p_rank)
            try:
                svs = np.linalg.svd(cross, compute_uv=False)
                svs = np.clip(svs, 0, 1)
                principal_angles_deg = np.degrees(np.arccos(svs)).tolist()
                mean_principal_angle = float(np.mean(principal_angles_deg))
                # Variance of H explained by P subspace
                P_proj_var = float(np.var(P_coords, axis=0).sum())
                H_total_var = float(np.var(H, axis=0).sum())
                P_var_fraction = P_proj_var / max(H_total_var, 1e-12)
            except:
                principal_angles_deg = []
                mean_principal_angle = None
                P_var_fraction = None
        else:
            principal_angles_deg = []
            mean_principal_angle = None
            P_var_fraction = None
        
        summary["P_vs_PCA_principal_angles_deg"] = [round(a, 2) for a in principal_angles_deg]
        summary["P_vs_PCA_mean_angle_deg"] = round(mean_principal_angle, 2) if mean_principal_angle is not None else None
        summary["P_variance_fraction_of_H"] = round(P_var_fraction, 4) if P_var_fraction is not None else None
        
        # === Step 9: PCA control — contiguous variance-ranked bins ===
        # Partition PCA components of H into bins with same sizes as PDSF,
        # but aligned to raw variance ranking, not to prediction.
        pca_control_bins = {}
        total_pca_rank = H_pca_basis.shape[1]
        bin_sizes = {"bin_P": p_rank, "bin_D": k_d, "bin_S": Bs.shape[1]}
        # bin_F gets everything from end of bin_S to rank 50 (PCA truncation)
        offset = 0
        for bin_name, bsize in bin_sizes.items():
            end = min(offset + bsize, total_pca_rank)
            if end <= offset:
                continue
            bin_coords = H @ H_pca_basis[:, offset:end]  # project onto this PCA slice
            geo = _subspace_geometry(bin_coords, k_neighbors, bin_name)
            if geo is not None:
                pca_control_bins[bin_name] = geo
            offset = end
        # Residual PCA bin
        if offset < total_pca_rank:
            bin_coords = H @ H_pca_basis[:, offset:total_pca_rank]
            geo = _subspace_geometry(bin_coords, k_neighbors, "bin_F")
            if geo is not None:
                pca_control_bins["bin_F"] = geo
        
        # Check ordering in PCA bins
        pca_curvatures = []
        for bn in ["bin_P", "bin_D", "bin_S", "bin_F"]:
            cr = pca_control_bins.get(bn, {}).get("curvature_ratio")
            if cr is not None:
                pca_curvatures.append((bn, cr))
        pca_control_ordering_monotonic = all(
            pca_curvatures[i][1] < pca_curvatures[i+1][1]
            for i in range(len(pca_curvatures)-1)
        ) if len(pca_curvatures) > 1 else None
        
        # === Step 10: Shuffled PCA control — random assignment of PCA dims to bins ===
        # Each PCA component is randomly assigned to a bin (maintaining bin sizes).
        # Repeated N_SHUFFLES times and averaged.
        N_SHUFFLES = 20
        shuffled_curvatures = {bn: [] for bn in ["bin_P", "bin_D", "bin_S", "bin_F"]}
        
        for _shuf in range(N_SHUFFLES):
            perm = np.random.permutation(total_pca_rank)
            offset = 0
            for bin_name, bsize in [("bin_P", p_rank), ("bin_D", k_d), ("bin_S", Bs.shape[1])]:
                end = min(offset + bsize, total_pca_rank)
                if end <= offset:
                    continue
                bin_idx = perm[offset:end]
                bin_coords = H @ H_pca_basis[:, bin_idx]
                geo = _subspace_geometry(bin_coords, k_neighbors, bin_name)
                if geo is not None:
                    shuffled_curvatures[bin_name].append(geo["curvature_ratio"])
                offset = end
            if offset < total_pca_rank:
                bin_idx = perm[offset:total_pca_rank]
                bin_coords = H @ H_pca_basis[:, bin_idx]
                geo = _subspace_geometry(bin_coords, k_neighbors, "bin_F")
                if geo is not None:
                    shuffled_curvatures["bin_F"].append(geo["curvature_ratio"])
        
        shuffled_control_bins = {}
        for bn in ["bin_P", "bin_D", "bin_S", "bin_F"]:
            vals = shuffled_curvatures[bn]
            if vals:
                shuffled_control_bins[bn] = {
                    "curvature_ratio_mean": round(float(np.mean(vals)), 4),
                    "curvature_ratio_std": round(float(np.std(vals)), 4),
                    "n_shuffles": len(vals),
                }
        
                result["by_layer"][str(layer)] = {
            "subspaces": subspaces,
            "summary": summary,
            "basis_ranks": {
                "P": p_rank, "P_pr_raw": round(float(p_pr), 2),
                "D": k_d, "S": Bs.shape[1], "residual": res_rank,
            },
            "pca_control": {
                "contiguous_bins": pca_control_bins,
                "contiguous_ordering_monotonic": pca_control_ordering_monotonic,
                "contiguous_curvatures": [(n, round(c, 4)) for n, c in pca_curvatures],
            },
            "shuffled_control": {
                "bins": shuffled_control_bins,
            },
        }
    
    return result


def compute_resolution_geometry_within_regime(
    extraction,
    P_bases,
    unembed_weight,
    regime_labels,
    layers=None,
    k_max=64,
    s_small_r=16,
    k_neighbors=5,
):
    """
    Within-regime curvature using GLOBAL PR-adaptive bases.

    v11.1: Full-trajectory version with PR-adaptive P.

    This is the critical test for whether the curvature ordering P < D < S < F
    is a genuine property of the representational geometry or an artifact of
    between-regime clustering.

    1. At each layer, compute PR-adaptive P, D, S bases from ALL prompts.
    2. Project ALL prompts into the global D, S, P, F coordinate spaces.
    3. For each regime, compute curvature using ONLY that regime's points.

    If the ordering holds within each regime separately, it cannot be explained
    by between-regime clustering inflating D's apparent flatness.

    Runs at ALL layers by default for full depth trajectories.

    Args:
        extraction: ModelExtraction with .layers, .group_ids, .meta_rows
        P_bases: Dict[layer][prompt_idx] -> P basis (rank-1 per prompt)
        unembed_weight: Unembedding matrix (vocab x hidden_dim)
        regime_labels: List of regime labels, one per prompt
        layers: Which layers (default: ALL)
        k_max: Max D rank
        s_small_r: S basis rank
        k_neighbors: k for kNN (adapted per regime for small n)

    Returns:
        Dict with per_regime, pooled, regime_sizes, global_basis_info,
        ordering_summary
    """
    from pds_continuation import (
        compute_D_basis_global as _compute_D_canon,
        compute_S_basis_global as _compute_S_canon,
    )

    if layers is None:
        layers = sorted(extraction.layers.keys())

    # Build regime -> prompt indices
    regime_to_idx = {}
    for i, r in enumerate(regime_labels):
        regime_to_idx.setdefault(r, []).append(i)
    unique_regimes = sorted(regime_to_idx.keys())
    regime_sizes = {r: len(idxs) for r, idxs in regime_to_idx.items()}

    # Unembedding matrix
    W = unembed_weight
    if W.shape[0] < W.shape[1]:
        W = W.T

    def _subspace_geometry_subset(X_proj, subset_idx, k_nn):
        """Curvature for a subset of points in a precomputed projection."""
        X_sub = X_proj[subset_idx]
        n, rank = X_sub.shape
        if n < 3 or rank == 0:
            return None

        dists = squareform(pdist(X_sub, metric="euclidean"))
        np.fill_diagonal(dists, np.inf)

        k_nn_actual = min(k_nn, n - 1)
        knn_dists = np.sort(dists, axis=1)[:, :k_nn_actual]
        mean_knn = float(knn_dists.mean())

        np.fill_diagonal(dists, 0)
        triu = dists[np.triu_indices(n, k=1)]
        mean_global = float(triu.mean())

        curvature_ratio = mean_knn / max(mean_global, 1e-12)

        local_prs = []
        for i in range(n):
            nn_idx = np.argsort(dists[i])[:k_nn_actual]
            local_patch = X_sub[nn_idx]
            if local_patch.shape[0] >= 3:
                local_prs.append(participation_ratio(local_patch, center=True))
        mean_local_pr = float(np.mean(local_prs)) if local_prs else float("nan")

        global_pr = participation_ratio(X_sub, center=True)
        total_var = float(np.var(X_sub, axis=0).sum())
        info_density = total_var / max(rank, 1)

        return {
            "rank": rank,
            "n_prompts": n,
            "curvature_ratio": round(curvature_ratio, 4),
            "mean_knn_dist": round(mean_knn, 4),
            "mean_global_dist": round(mean_global, 4),
            "local_pr": round(mean_local_pr, 2),
            "global_pr": round(global_pr, 2),
            "pr_folding_ratio": round(mean_local_pr / max(global_pr, 0.01), 3),
            "info_density": round(info_density, 4),
            "total_variance": round(total_var, 4),
        }

    per_regime_results = {r: {"by_layer": {}} for r in unique_regimes}
    global_basis_info = {}
    ordering_summary = {r: {} for r in unique_regimes}

    try:
        from tqdm import tqdm as _tqdm
        _layer_iter = _tqdm(layers, desc="Within-regime curvature", unit="layer")
    except ImportError:
        _layer_iter = layers
    
    for layer in _layer_iter:
        H = extraction.layers[layer].H
        n_prompts, d = H.shape

        # === Global PR-adaptive P basis ===
        HP = np.zeros_like(H)
        for idx in range(n_prompts):
            Bp = P_bases[layer].get(idx)
            if Bp is not None and Bp.size > 0:
                HP[idx:idx+1] = project_onto_basis(H[idx:idx+1], Bp)
        H_perp = H - HP

        P_proj_vectors = []
        for idx in range(n_prompts):
            Bp = P_bases[layer].get(idx)
            if Bp is not None and Bp.size > 0:
                P_proj_vectors.append(HP[idx])

        if len(P_proj_vectors) >= 3:
            P_proj_matrix = np.stack(P_proj_vectors, axis=0)
            p_pr = participation_ratio(P_proj_matrix, center=True)
            p_rank = max(1, min(int(round(p_pr)), len(P_proj_vectors) - 1, d))
            P_basis = pca_basis(P_proj_matrix, k=p_rank, center=True)
            P_coords = H @ P_basis
        else:
            p_rank = 1
            p_pr = 1.0
            P_coords = np.zeros((n_prompts, 1), dtype=np.float32)

        # === Global D basis ===
        Bd, D_info = _compute_D_canon(H, P_bases[layer], k_D=min(k_max, 12))
        k_d = D_info["k"]

        # === Global S basis ===
        Bs, S_info = _compute_S_canon(H, P_bases[layer], Bd, k_S=s_small_r)

        # === Project into subspaces ===
        D_coords = H_perp @ Bd
        S_coords = H_perp @ Bs

        H_d_component = D_coords @ Bd.T
        H_s_component = S_coords @ Bs.T
        H_residual = H_perp - H_d_component - H_s_component
        res_rank = min(50, H_residual.shape[0] - 1, H_residual.shape[1])
        if res_rank > 0:
            Res_coords = H_residual @ pca_basis(H_residual, k=res_rank, center=True)
        else:
            Res_coords = np.zeros((n_prompts, 1), dtype=np.float32)

        global_basis_info[str(layer)] = {
            "D_rank": k_d, "S_rank": Bs.shape[1],
            "P_rank": p_rank, "P_pr_raw": round(float(p_pr), 2),
            "residual_rank": res_rank,
        }

        # === Per-regime curvature using global coordinates ===
        all_coords = {"P": P_coords, "D": D_coords, "S": S_coords, "residual": Res_coords}

        for regime in unique_regimes:
            idxs = regime_to_idx[regime]
            n_r = len(idxs)
            if n_r < 3:
                per_regime_results[regime]["by_layer"][str(layer)] = {
                    "skipped": True, "reason": f"only {n_r} prompts",
                }
                continue

            k_nn = min(k_neighbors, n_r - 1)
            subspaces = {}
            for name, coords in all_coords.items():
                geo = _subspace_geometry_subset(coords, idxs, k_nn)
                if geo is not None:
                    subspaces[name] = geo

            # Summary and ordering
            summary = {}
            for name in ["P", "D", "S", "residual"]:
                if name in subspaces and "curvature_ratio" in subspaces[name]:
                    summary[f"{name}_curvature_ratio"] = subspaces[name]["curvature_ratio"]
                    summary[f"{name}_info_density"] = subspaces[name]["info_density"]
                    summary[f"{name}_local_pr"] = subspaces[name]["local_pr"]
                    summary[f"{name}_global_pr"] = subspaces[name]["global_pr"]

            D_curv = subspaces.get("D", {}).get("curvature_ratio")
            P_curv = subspaces.get("P", {}).get("curvature_ratio")
            S_curv = subspaces.get("S", {}).get("curvature_ratio")
            F_curv = subspaces.get("residual", {}).get("curvature_ratio")

            pdsf = [P_curv, D_curv, S_curv, F_curv]
            pdsf_valid = [v for v in pdsf if v is not None]
            summary["ordering_PDSF"] = ["P", "D", "S", "F"]
            summary["ordering_PDSF_values"] = [round(v, 4) for v in pdsf_valid]
            summary["ordering_PDSF_monotonic"] = all(
                pdsf_valid[i] < pdsf_valid[i+1] for i in range(len(pdsf_valid)-1)
            ) if len(pdsf_valid) > 1 else None
            summary["P_lt_D"] = (P_curv < D_curv) if (P_curv is not None and D_curv is not None) else None
            summary["D_lt_S"] = (D_curv < S_curv) if (D_curv is not None and S_curv is not None) else None
            summary["P_lt_S"] = (P_curv < S_curv) if (P_curv is not None and S_curv is not None) else None
            summary["S_lt_F"] = (S_curv < F_curv) if (S_curv is not None and F_curv is not None) else None

            ordering_summary[regime][str(layer)] = summary.get("ordering_PDSF_monotonic")

            per_regime_results[regime]["by_layer"][str(layer)] = {
                "subspaces": subspaces,
                "summary": summary,
            }

    # === Pooled average across regimes ===
    pooled = {"by_layer": {}}
    for layer in layers:
        layer_key = str(layer)
        curvatures = {"P": [], "D": [], "S": [], "residual": []}
        for regime in unique_regimes:
            layer_data = per_regime_results[regime].get("by_layer", {}).get(layer_key, {})
            if "skipped" in layer_data:
                continue
            for name in curvatures:
                cr = layer_data.get("subspaces", {}).get(name, {}).get("curvature_ratio")
                if cr is not None:
                    curvatures[name].append(cr)

        pooled_summary = {}
        for name in ["P", "D", "S", "residual"]:
            if curvatures[name]:
                pooled_summary[f"{name}_curvature_ratio"] = round(float(np.mean(curvatures[name])), 4)
                pooled_summary[f"{name}_curvature_std"] = round(float(np.std(curvatures[name])), 4)
                pooled_summary[f"{name}_n_regimes"] = len(curvatures[name])

        pdsf_pooled = [
            pooled_summary.get("P_curvature_ratio"),
            pooled_summary.get("D_curvature_ratio"),
            pooled_summary.get("S_curvature_ratio"),
            pooled_summary.get("residual_curvature_ratio"),
        ]
        pdsf_valid = [v for v in pdsf_pooled if v is not None]
        pooled_summary["ordering_PDSF_monotonic"] = all(
            pdsf_valid[i] < pdsf_valid[i+1] for i in range(len(pdsf_valid)-1)
        ) if len(pdsf_valid) > 1 else None

        pooled["by_layer"][layer_key] = {"summary": pooled_summary}

    return {
        "per_regime": per_regime_results,
        "pooled": pooled,
        "regime_sizes": regime_sizes,
        "global_basis_info": global_basis_info,
        "ordering_summary": ordering_summary,
        "method": "global_basis_within_regime_curvature",
        "description": (
            "Curvature measured within each regime using PR-adaptive bases computed "
            "from ALL prompts. Isolates within-regime geometry from between-regime "
            "clustering artifacts. Full depth trajectory."
        ),
    }



def compute_resolution_geometry_stratified(
    extraction,
    P_bases,
    unembed_weight,
    regime_labels,
    layers=None,
    k_max=64,
    s_small_r=16,
    k_neighbors=5,
):
    """
    Regime-stratified curvature computation for diverse prompt sets.

    Runs compute_resolution_geometry() independently on each regime's prompts,
    then aggregates results. This avoids confounding curvature measurements
    with within-regime clustering when regimes are linguistically very different.

    Args:
        extraction: ModelExtraction with .layers, .group_ids, .meta_rows
        P_bases: Dict[layer][prompt_idx] -> P basis matrix
        unembed_weight: Unembedding matrix (vocab x hidden_dim)
        regime_labels: List of regime labels, one per prompt (len = n_prompts)
        layers: Which layers (default: 5 representative)
        k_max: Max D rank
        s_small_r: S basis rank
        k_neighbors: k for k-NN curvature (default 5, adapted for small regimes)

    Returns:
        Dict with:
          - per_regime: Dict[regime] -> compute_resolution_geometry output
          - averaged: Equal-weighted average of curvature values across regimes
          - size_weighted: Size-weighted average
          - ordering_per_regime: Whether P < D < S < F holds per regime
          - ordering_averaged: Whether ordering holds in averaged curvatures
    """
    from specA_analysis import ModelExtraction, LayerTensors

    unique_regimes = sorted(set(regime_labels))
    n_prompts = len(regime_labels)

    # Build regime -> prompt indices mapping
    regime_to_idx = {}
    for i, r in enumerate(regime_labels):
        regime_to_idx.setdefault(r, []).append(i)

    per_regime_results = {}
    regime_sizes = {}

    for regime in unique_regimes:
        idxs = regime_to_idx[regime]
        n_r = len(idxs)
        regime_sizes[regime] = n_r

        if n_r < 3:
            per_regime_results[regime] = {"skipped": True, "reason": f"only {n_r} prompts"}
            continue

        # Build sub-extraction with only this regime's prompts
        sub_layers = {}
        for layer_idx, lt in extraction.layers.items():
            sub_H = lt.H[idxs]
            sub_layers[layer_idx] = LayerTensors(H=sub_H)

        sub_extraction = ModelExtraction(
            model_id=extraction.model_id,
            layers=sub_layers,
            group_ids=[extraction.group_ids[i] for i in idxs],
            variant_ids=[extraction.variant_ids[i] for i in idxs],
            meta_rows=[extraction.meta_rows[i] for i in idxs],
        )

        # Build sub-P_bases: remap prompt indices
        sub_P_bases = {}
        for layer_idx in extraction.layers.keys():
            sub_P_bases[layer_idx] = {}
            for new_idx, old_idx in enumerate(idxs):
                bp = P_bases[layer_idx].get(old_idx)
                if bp is not None:
                    sub_P_bases[layer_idx][new_idx] = bp

        # Adapt k_neighbors for small sample size
        k_nn = min(k_neighbors, n_r - 1)

        try:
            regime_result = compute_resolution_geometry(
                sub_extraction, sub_P_bases, unembed_weight,
                layers=layers, k_max=k_max, s_small_r=s_small_r,
                k_neighbors=k_nn,
            )
            per_regime_results[regime] = regime_result
        except Exception as e:
            per_regime_results[regime] = {"skipped": True, "reason": str(e)}

    # --- Aggregate across regimes ---
    all_layers_in_results = set()
    for regime, res in per_regime_results.items():
        if isinstance(res, dict) and "by_layer" in res:
            all_layers_in_results.update(res["by_layer"].keys())

    averaged = {"by_layer": {}}
    size_weighted = {"by_layer": {}}
    ordering_per_regime = {}

    for layer_key in sorted(all_layers_in_results):
        subspace_curvatures = {"P": [], "D": [], "S": [], "residual": []}
        subspace_curvatures_weighted = {"P": [], "D": [], "S": [], "residual": []}

        for regime in unique_regimes:
            res = per_regime_results.get(regime, {})
            if not isinstance(res, dict) or "by_layer" not in res:
                continue
            layer_data = res["by_layer"].get(layer_key, {})
            subspaces = layer_data.get("subspaces", {})
            n_r = regime_sizes.get(regime, 1)

            for name in ["P", "D", "S", "residual"]:
                cr = subspaces.get(name, {}).get("curvature_ratio")
                if cr is not None:
                    subspace_curvatures[name].append(cr)
                    subspace_curvatures_weighted[name].append((cr, n_r))

            # Per-regime ordering check
            summary = layer_data.get("summary", {})
            if summary.get("ordering_monotonic") is not None:
                ordering_per_regime.setdefault(regime, {})[layer_key] = summary["ordering_monotonic"]

        # Equal-weighted average
        avg_summary = {}
        for name in ["P", "D", "S", "residual"]:
            vals = subspace_curvatures[name]
            if vals:
                avg_summary[f"{name}_curvature_ratio"] = round(float(np.mean(vals)), 4)

        # Check ordering on averaged values
        avg_ratios = []
        for name in ["P", "D", "S", "residual"]:
            if f"{name}_curvature_ratio" in avg_summary:
                avg_ratios.append((name, avg_summary[f"{name}_curvature_ratio"]))
        avg_summary["curvature_ordering"] = [(n, r) for n, r in avg_ratios]
        avg_summary["ordering_monotonic"] = all(
            avg_ratios[i][1] <= avg_ratios[i+1][1] for i in range(len(avg_ratios)-1)
        ) if len(avg_ratios) > 1 else None

        averaged["by_layer"][layer_key] = {"summary": avg_summary}

        # Size-weighted average
        sw_summary = {}
        for name in ["P", "D", "S", "residual"]:
            vals = subspace_curvatures_weighted[name]
            if vals:
                total_w = sum(w for _, w in vals)
                sw_summary[f"{name}_curvature_ratio"] = round(
                    float(sum(v * w for v, w in vals) / max(total_w, 1)), 4
                )
        size_weighted["by_layer"][layer_key] = {"summary": sw_summary}

    return {
        "per_regime": per_regime_results,
        "averaged": averaged,
        "size_weighted": size_weighted,
        "ordering_per_regime": ordering_per_regime,
        "regime_sizes": regime_sizes,
    }




# ═══════════════════════════════════════════════════════════════════════════════
# PART K — Cosine Similarity Subspace Discrimination
# ═══════════════════════════════════════════════════════════════════════════════

def compute_part_K_cosine_discrimination(
    extraction,
    P_bases,
    unembed_weight,
    group_ids,
    layer="L-2",
    k_max: int = 64,
    s_small_r: int = 16,
    n_shuffles: int = 20,
    seed: int = 42,
    regime_ids=None,
):
    """
    Part K: Cosine Similarity Subspace Discrimination.

    Measures within-group vs between-group pairwise cosine similarity for each
    PDSF subspace and two PCA control subspaces.

    Primary question: Does PDSF decomposition separate groups better than
    variance-ranked PCA (PCA-contiguous)?

    Secondary: Is F group separation lower than D and S (F = shared infrastructure)?

    Four-outcome decision tree for F (interpreting transplant experiment):
        D >> PCA_D > S > F ≈ 0   : expected on SpecA (structured factorial prompts)
        All gaps ≈ 0              : SpecB / Diverse baseline (no strong prior)

    Layer parameter:
        "L-2"  : second-to-last layer (default; resolved at runtime per model)
        int    : explicit layer index (for cross-model fixed-layer comparison)
    """
    import numpy as np
    from pds_continuation import (
        compute_D_basis_global as _compute_D_canon,
        compute_S_basis_global as _compute_S_canon,
    )

    rng = np.random.RandomState(seed)

    # ── Resolve target layer ──────────────────────────────────────────────────
    layers_sorted = sorted(extraction.layers.keys())
    if layer == "L-2":
        target_layer = layers_sorted[-2] if len(layers_sorted) >= 2 else layers_sorted[-1]
        layer_spec = "L-2"
    elif isinstance(layer, int):
        target_layer = layer
        layer_spec = layer
    else:
        raise ValueError(f"layer must be 'L-2' or int, got {layer!r}")

    # ── Extract H at target layer via .H attribute (LayerTensors object) ─────
    layer_obj = extraction.layers.get(target_layer)
    if layer_obj is None:
        raise ValueError(f"Layer {target_layer} not in extraction.layers")
    H = np.array(layer_obj.H, dtype=np.float32)
    n_prompts, d_model = H.shape

    # ── P bases at this layer — indexed as P_bases[layer][prompt_idx] ────────
    Bp_at_layer = P_bases.get(target_layer, {})

    # ── Compute PDSF decomposition using pds_continuation canonical functions ─
    # compute_D_basis_global handles P-projection internally given P_bases dict
    D_result = _compute_D_canon(H, Bp_at_layer, k_D=k_max)
    D_basis = D_result[0] if isinstance(D_result, tuple) else D_result
    k_D = int(D_basis.shape[1])

    S_result = _compute_S_canon(H, Bp_at_layer, D_basis, k_S=s_small_r)
    S_basis = S_result[0] if isinstance(S_result, tuple) else S_result
    if S_basis is None:
        S_basis = np.zeros((d_model, 0), dtype=np.float32)
    k_S = int(S_basis.shape[1])

    # Project into each subspace
    # D: project P-residuals onto D_basis
    H_Presid = np.zeros_like(H)
    for i in range(n_prompts):
        Bp_i = Bp_at_layer.get(i)
        if Bp_i is not None and np.array(Bp_i).size > 0:
            Bp_i = np.array(Bp_i, dtype=np.float32)
            if Bp_i.ndim == 1:
                Bp_i = Bp_i.reshape(-1, 1)
            H_Presid[i] = H[i] - Bp_i @ (Bp_i.T @ H[i])
        else:
            H_Presid[i] = H[i]

    X_D = H_Presid @ D_basis                                 # (n, k_D)
    X_S = (H_Presid - H_Presid @ D_basis @ D_basis.T) @ S_basis if k_S > 0 else np.zeros((n_prompts, 0))

    # F = h - h_P - h_D - h_S
    H_DS_proj = H_Presid @ D_basis @ D_basis.T
    if k_S > 0:
        H_DS_proj = H_DS_proj + (H_Presid - H_Presid @ D_basis @ D_basis.T) @ S_basis @ S_basis.T
    h_P_all = H - H_Presid
    X_F = H - h_P_all - H_DS_proj                            # (n, d_model)

    # ── Full-H PCA for controls ───────────────────────────────────────────────
    H_centered = H - H.mean(axis=0, keepdims=True)
    try:
        _, _, Vt = np.linalg.svd(H_centered, full_matrices=False)
        H_pca_basis = Vt.T                                    # (d_model, min(n,d))
        total_pca_rank = H_pca_basis.shape[1]
    except Exception as e:
        raise ValueError(f"SVD failed: {e}")

    # Estimate p_rank from median P basis shape
    p_rank_vals = [np.array(b).shape[-1] if np.array(b).ndim > 1 else 1
                   for b in Bp_at_layer.values() if b is not None]
    p_rank = int(np.median(p_rank_vals)) if p_rank_vals else 1
    bin_sizes = {
        "bin_P": p_rank,
        "bin_D": k_D,
        "bin_S": k_S,
        "bin_F": max(0, total_pca_rank - p_rank - k_D - k_S),
    }

    # ── Helper: within/between cosine similarity statistics ───────────────────
    def _cosine_stats(X_proj, labels):
        n = X_proj.shape[0]
        if n < 2 or X_proj.shape[1] == 0:
            return {"mean_within": None, "mean_between": None, "gap": None, "cohen_d": None,
                    "n_within_pairs": 0, "n_between_pairs": 0}
        norms = np.linalg.norm(X_proj, axis=1, keepdims=True)
        valid = (norms.flatten() > 1e-8)
        X_norm = np.where(norms > 1e-8, X_proj / np.where(norms > 1e-8, norms, 1.0), 0.0)
        sim = X_norm @ X_norm.T
        within, between = [], []
        for i in range(n):
            for j in range(i + 1, n):
                if not (valid[i] and valid[j]):
                    continue
                s = float(sim[i, j])
                if labels[i] == labels[j]:
                    within.append(s)
                else:
                    between.append(s)
        if not within or not between:
            return {"mean_within": None, "mean_between": None, "gap": None, "cohen_d": None,
                    "n_within_pairs": len(within), "n_between_pairs": len(between)}
        mw, mb = float(np.mean(within)), float(np.mean(between))
        gap = mw - mb
        sw, sb = float(np.std(within)), float(np.std(between))
        nw, nb_ = len(within), len(between)
        pooled = np.sqrt(((nw - 1) * sw**2 + (nb_ - 1) * sb**2) / max(nw + nb_ - 2, 1))
        cohen_d = gap / (pooled + 1e-12)
        return {"mean_within": round(mw, 5), "mean_between": round(mb, 5),
                "gap": round(float(gap), 5), "cohen_d": round(float(cohen_d), 4),
                "n_within_pairs": nw, "n_between_pairs": nb_}

    # ── Compute stats for D, S, F ─────────────────────────────────────────────
    by_subspace = {}
    by_subspace["D"] = _cosine_stats(X_D, group_ids)
    by_subspace["S"] = _cosine_stats(X_S, group_ids) if k_S > 0 else         {"mean_within": None, "mean_between": None, "gap": None, "cohen_d": None}
    by_subspace["F"] = _cosine_stats(X_F, group_ids)

    # ── PCA-contiguous control ────────────────────────────────────────────────
    pca_contiguous = {}
    offset = 0
    for bin_name, bsize in bin_sizes.items():
        end = min(offset + bsize, total_pca_rank)
        if end > offset:
            X_bin = H_centered @ H_pca_basis[:, offset:end]
            pca_contiguous[bin_name] = _cosine_stats(X_bin, group_ids)
        offset = end
    by_subspace["pca_contiguous"] = pca_contiguous

    # ── PCA-shuffled null ─────────────────────────────────────────────────────
    shuffled_accum = {bn: {"mean_within": [], "mean_between": [], "gap": [], "cohen_d": []}
                      for bn in bin_sizes}
    all_idx = list(range(total_pca_rank))
    for _ in range(n_shuffles):
        perm = rng.permutation(all_idx)
        offset2 = 0
        for bn, bsize in bin_sizes.items():
            end2 = min(offset2 + bsize, total_pca_rank)
            if end2 > offset2:
                X_bin = H_centered @ H_pca_basis[:, perm[offset2:end2]]
                st = _cosine_stats(X_bin, group_ids)
                for k in ("mean_within", "mean_between", "gap", "cohen_d"):
                    if st[k] is not None:
                        shuffled_accum[bn][k].append(st[k])
            offset2 = end2
    pca_shuffled = {}
    for bn, acc in shuffled_accum.items():
        pca_shuffled[bn] = {}
        for k, v in acc.items():
            pca_shuffled[bn][k + "_mean"] = round(float(np.mean(v)), 5) if v else None
            pca_shuffled[bn][k + "_std"]  = round(float(np.std(v)), 5) if v else None
    by_subspace["pca_shuffled"] = pca_shuffled

    # ── Regime-level extension ────────────────────────────────────────────────
    regime_level = None
    if regime_ids is not None:
        regime_level = {}
        for sub_name, X_sub in [("D", X_D), ("S", X_S), ("F", X_F)]:
            if X_sub.shape[1] > 0:
                regime_level[sub_name] = _cosine_stats(X_sub, regime_ids)

    # ── Ordering checks ───────────────────────────────────────────────────────
    gap_D     = by_subspace["D"].get("gap")
    gap_S     = by_subspace["S"].get("gap")
    gap_F     = by_subspace["F"].get("gap")
    gap_PCA_D = pca_contiguous.get("bin_D", {}).get("gap")
    shuf_D    = pca_shuffled.get("bin_D", {}).get("gap_mean")
    # Cohen's d values for z-score comparison
    d_D     = by_subspace["D"].get("cohen_d")
    d_F     = by_subspace["F"].get("cohen_d")
    shuf_D_d_mean = pca_shuffled.get("bin_D", {}).get("cohen_d_mean")
    shuf_D_d_std  = pca_shuffled.get("bin_D", {}).get("cohen_d_std")

    ordering_check = {
        # Core PDSF validation: D discriminates, F does not
        "PDSF_D_gt_PDSF_S":      bool(gap_D is not None and gap_S is not None and gap_D > gap_S),
        "PDSF_S_gt_PDSF_F":      bool(gap_S is not None and gap_F is not None and gap_S > gap_F),
        "PDSF_D_gt_PDSF_F":      bool(gap_D is not None and gap_F is not None and gap_D > gap_F),
        # PDSF-D beats random-dimension baseline at matched dimensionality
        "PDSF_D_gt_PCA_shuffled_D": bool(
            d_D is not None and shuf_D_d_mean is not None and shuf_D_d_std is not None
            and shuf_D_d_std > 0 and (d_D - shuf_D_d_mean) / shuf_D_d_std > 1.0
        ),
        # Legacy: PDSF-D vs PCA-contiguous-D (informational, not expected to pass)
        "PDSF_D_gt_PCA_D":       bool(gap_D is not None and gap_PCA_D is not None and gap_D > gap_PCA_D),
    }

    return {
        "layer_used": int(target_layer),
        "layer_spec": layer_spec,
        "n_prompts": n_prompts,
        "n_groups": len(set(group_ids)),
        "basis_ranks": {"D": k_D, "S": k_S, "P_approx": p_rank},
        "bin_sizes": {k: int(v) for k, v in bin_sizes.items()},
        "by_subspace": by_subspace,
        "regime_level": regime_level,
        "ordering_check": ordering_check,
    }

print("✓ compute_part_K_cosine_discrimination available")


# ============================================================================
# PART L: COMPLEXITY DEPTH PROFILES
# ============================================================================
# Computes manifold complexity (curvature ratio, local/global PR, info density)
# for each PDSF subspace at EVERY layer, producing continuous depth trajectories.
#
# DESIGN RATIONALE:
# Part J (Resolution Geometry) already computes these metrics, but it runs
# as a standalone analysis step with its own PCA recomputation at each layer.
# Part L reuses the same _subspace_geometry() core but is designed to run
# alongside Part G: it consumes the same layer_to_H cache (no extra forward
# passes), and outputs a separate JSON file focused on the depth question:
#   "How does manifold complexity evolve across layers for each subspace?"
#
# This is the BASELINE (no-intervention) complexity profile. Future work may
# compare these profiles against post-intervention profiles from Part G, but
# this version measures only the unscrambled hidden states.
#
# OUTPUT:
#   {model}-Geometry-{prompt_set}-part_l_complexity_depth_profile.json
#
# THEORETICAL PREDICTION:
#   P < D < S < F in curvature ratio at late layers:
#   - D (discourse subspace) should be tightly clustered by group -> low curvature ratio
#   - P (prediction) should be sharply peaked -> low-to-moderate curvature ratio
#   - S (situation) should be moderately spread -> higher curvature ratio
#   - F (framework) should be smoothly distributed -> highest curvature ratio
# ============================================================================


def compute_part_L_complexity_depth_profile(
    layer_to_H,
    group_ids,
    P_bases_by_layer,
    unembed_weight,
    k_max: int = 64,
    s_small_r: int = 16,
    k_neighbors: int = 8,
    layers: list = None,
    verbose: bool = True,
):
    """
    Part L: Complexity Depth Profiles — manifold geometry at every layer.

    Computes curvature ratio, local/global participation ratio, info density,
    and softmax gradient for each PDSF subspace (P, D, S, F/residual) at
    every layer in the model.

    This is a BASELINE measurement on unscrambled hidden states. It runs
    purely on cached H matrices (no GPU forward passes required beyond the
    initial extraction).

    Args:
        layer_to_H:       Dict[int, ndarray/Tensor] — cached hidden states per layer.
                          Shape: (n_prompts, d_model) per layer.
        group_ids:        List[str] — group label per prompt (for metadata).
        P_bases_by_layer: Dict[layer][prompt_idx] -> ndarray — P basis vectors.
        unembed_weight:   ndarray — unembedding matrix (vocab × d_model or transposed).
        k_max:            int — max D basis rank for PCA truncation.
        s_small_r:        int — S basis rank.
        k_neighbors:      int — k for nearest-neighbor curvature estimate.
        layers:           List[int] or None — which layers to process (None = all).
        verbose:          bool — print progress.

    Returns:
        Dict with:
          "by_layer": {layer -> {subspaces: {P, D, S, residual -> metrics}, summary}},
          "metadata": {parameters, n_prompts, n_layers, etc.},
          "depth_trajectories": {subspace -> {metric -> [value_per_layer]}},
    """
    import numpy as np
    import torch
    from scipy.spatial.distance import pdist, squareform
    from pds_continuation import (
        compute_D_basis_global as _compute_D_canon,
        compute_S_basis_global as _compute_S_canon,
    )

    # ── Resolve layers ────────────────────────────────────────────────────────
    if layers is None:
        layers = sorted(layer_to_H.keys())
    else:
        layers = sorted(layers)

    # ── Prepare unembedding for P-gradient ────────────────────────────────────
    W = unembed_weight
    if W.shape[0] < W.shape[1]:
        W = W.T  # ensure (vocab, hidden_dim)

    # ── Convert layer_to_H values to numpy ────────────────────────────────────
    def _to_np(x):
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy().astype(np.float32)
        return np.asarray(x, dtype=np.float32)

    n_prompts = _to_np(layer_to_H[layers[0]]).shape[0]
    d_model   = _to_np(layer_to_H[layers[0]]).shape[1]

    # ── Core geometry function (mirrors Part J's _subspace_geometry) ──────────
    def _subspace_geometry(X_proj, k_nn, label):
        """
        Compute manifold geometry metrics for prompts projected into a subspace.

        Metrics computed:
          curvature_ratio:  mean_kNN_distance / mean_global_distance
                            Low values = tight local clustering relative to global spread
                            (i.e., highly structured / "peaked" manifold).
          local_pr:         Mean participation ratio of k-nearest-neighbor patches.
                            Low = locally low-dimensional (prompts lie near a line/plane).
          global_pr:        Participation ratio of full projection.
          pr_folding_ratio: local_pr / global_pr. < 1 means local neighborhoods are
                            lower-dimensional than the global cloud (manifold is "folded").
          info_density:     total_variance / rank. How much information per dimension.

        Args:
            X_proj: (n_prompts, subspace_rank) — coordinates in the subspace.
            k_nn:   Number of nearest neighbors.
            label:  Subspace name (for diagnostics only).

        Returns:
            Dict of metrics, or None if insufficient data.
        """
        n, rank = X_proj.shape
        if n < 3 or rank == 0:
            return None

        # Pairwise Euclidean distances
        dists = squareform(pdist(X_proj, metric="euclidean"))
        np.fill_diagonal(dists, np.inf)

        k_nn_actual = min(k_nn, n - 1)
        knn_dists = np.sort(dists, axis=1)[:, :k_nn_actual]
        mean_knn = float(knn_dists.mean())

        np.fill_diagonal(dists, 0)
        triu = dists[np.triu_indices(n, k=1)]
        mean_global = float(triu.mean())

        curvature_ratio = mean_knn / max(mean_global, 1e-12)

        # Local participation ratios in kNN neighborhoods
        local_prs = []
        for idx in range(n):
            nn_idx = np.argsort(dists[idx])[:k_nn_actual]
            local_patch = X_proj[nn_idx]
            if local_patch.shape[0] >= 3:
                pr_local = participation_ratio(local_patch, center=True)
                local_prs.append(pr_local)
        mean_local_pr = float(np.mean(local_prs)) if local_prs else float("nan")

        global_pr = participation_ratio(X_proj, center=True)
        total_var = float(np.var(X_proj, axis=0).sum())
        info_density = total_var / max(rank, 1)

        return {
            "rank":             rank,
            "n_prompts":        n,
            "curvature_ratio":  round(curvature_ratio, 4),
            "mean_knn_dist":    round(mean_knn, 4),
            "mean_global_dist": round(mean_global, 4),
            "local_pr":         round(mean_local_pr, 2),
            "global_pr":        round(global_pr, 2),
            "pr_folding_ratio": round(mean_local_pr / max(global_pr, 0.01), 3),
            "info_density":     round(info_density, 4),
            "total_variance":   round(total_var, 4),
        }

    # ── Main loop: compute geometry at each layer ─────────────────────────────
    by_layer = {}

    try:
        from tqdm import tqdm as _tqdm
        _layer_iter = _tqdm(layers, desc="Part L: complexity profiles", unit="layer")
    except ImportError:
        _layer_iter = layers

    for layer in _layer_iter:
        H = _to_np(layer_to_H[layer])
        n, d = H.shape

        # --- Step 1: Per-prompt P projections ---
        Bp_at_layer = P_bases_by_layer.get(layer, {})
        HP = np.zeros_like(H)
        for idx in range(n):
            Bp = Bp_at_layer.get(idx)
            if Bp is not None and np.asarray(Bp).size > 0:
                Bp_arr = np.asarray(Bp, dtype=np.float32)
                if Bp_arr.ndim == 1:
                    Bp_arr = Bp_arr.reshape(-1, 1)
                HP[idx] = Bp_arr @ (Bp_arr.T @ H[idx])
        H_perp = H - HP

        # --- Step 2: PR-adaptive P basis ---
        P_proj_vectors = []
        for idx in range(n):
            Bp = Bp_at_layer.get(idx)
            if Bp is not None and np.asarray(Bp).size > 0:
                P_proj_vectors.append(HP[idx])

        if len(P_proj_vectors) >= 3:
            P_proj_matrix = np.stack(P_proj_vectors, axis=0)
            p_pr = participation_ratio(P_proj_matrix, center=True)
            p_rank = max(1, min(int(round(p_pr)), len(P_proj_vectors) - 1, d))
            P_basis = pca_basis(P_proj_matrix, k=p_rank, center=True)
            P_coords = H @ P_basis
        else:
            p_rank = 1
            p_pr = 1.0
            P_coords = np.zeros((n, 1), dtype=np.float32)
            P_basis = None

        # --- Step 3: D basis (recomputed at this layer) ---
        Bd, D_info = _compute_D_canon(H, Bp_at_layer, k_D=min(k_max, 12))
        k_d = D_info["k"]

        # --- Step 4: S basis (recomputed at this layer) ---
        Bs, S_info = _compute_S_canon(H, Bp_at_layer, Bd, k_S=s_small_r)

        # --- Step 5: Project into subspaces ---
        D_coords = H_perp @ Bd
        S_coords = H_perp @ Bs

        # Residual (F): H_perp minus D and S projections
        H_d_component = D_coords @ Bd.T
        H_s_component = S_coords @ Bs.T
        H_residual = H_perp - H_d_component - H_s_component
        res_rank = min(50, H_residual.shape[0] - 1, H_residual.shape[1])
        if res_rank > 0:
            Res_coords = H_residual @ pca_basis(H_residual, k=res_rank, center=True)
        else:
            Res_coords = np.zeros((n, 1), dtype=np.float32)

        # --- Step 6: Compute geometry for each subspace ---
        subspaces = {}
        for name, coords in [("P", P_coords), ("D", D_coords),
                              ("S", S_coords), ("residual", Res_coords)]:
            geo = _subspace_geometry(coords, k_neighbors, name)
            if geo is not None:
                subspaces[name] = geo

        # P-specific: PR-adaptive rank and softmax gradient
        if "P" not in subspaces:
            subspaces["P"] = {"rank": p_rank, "n_prompts": n}
        subspaces["P"]["pr_adaptive_rank"] = p_rank
        subspaces["P"]["pr_raw"] = round(float(p_pr), 2)

        p_grads = []
        for idx in range(n):
            Bp = Bp_at_layer.get(idx)
            if Bp is not None and np.asarray(Bp).shape[-1] > 0:
                Bp_arr = np.asarray(Bp, dtype=np.float32)
                p_dir = Bp_arr[:, 0] if Bp_arr.ndim == 2 else Bp_arr
                logit_grad = W @ p_dir
                logits = W @ H[idx]
                logits_shifted = logits - logits.max()
                probs = np.exp(logits_shifted) / np.exp(logits_shifted).sum()
                weighted_grad = float(np.sqrt(
                    (probs * logit_grad**2).sum() - (probs * logit_grad).sum()**2
                ))
                p_grads.append(weighted_grad)
        subspaces["P"]["softmax_gradient_mean"] = round(float(np.mean(p_grads)), 4) if p_grads else None
        subspaces["P"]["softmax_gradient_std"]  = round(float(np.std(p_grads)), 4) if p_grads else None

        # --- Step 7: Summary and ordering checks ---
        summary = {}
        for name in ["P", "D", "S", "residual"]:
            if name in subspaces and "curvature_ratio" in subspaces[name]:
                summary[f"{name}_curvature_ratio"] = subspaces[name]["curvature_ratio"]
                summary[f"{name}_info_density"]    = subspaces[name]["info_density"]
                summary[f"{name}_local_pr"]        = subspaces[name]["local_pr"]
                summary[f"{name}_global_pr"]       = subspaces[name]["global_pr"]
                summary[f"{name}_pr_folding"]      = subspaces[name]["pr_folding_ratio"]
        if subspaces.get("P", {}).get("softmax_gradient_mean") is not None:
            summary["P_softmax_gradient"] = subspaces["P"]["softmax_gradient_mean"]

        # Ordering check: P < D < S < F in curvature ratio (canonical)
        P_curv = subspaces.get("P",        {}).get("curvature_ratio")
        D_curv = subspaces.get("D",        {}).get("curvature_ratio")
        S_curv = subspaces.get("S",        {}).get("curvature_ratio")
        F_curv = subspaces.get("residual",  {}).get("curvature_ratio")
        pdsf = [P_curv, D_curv, S_curv, F_curv]
        pdsf_valid = [v for v in pdsf if v is not None]
        summary["ordering_PDSF"] = ["P", "D", "S", "F"]
        summary["ordering_PDSF_values"] = [round(v, 4) for v in pdsf_valid]
        summary["ordering_PDSF_monotonic"] = all(
            pdsf_valid[i] < pdsf_valid[i+1] for i in range(len(pdsf_valid)-1)
        ) if len(pdsf_valid) > 1 else None
        # Pairwise checks
        summary["P_lt_D"] = (P_curv < D_curv) if (P_curv is not None and D_curv is not None) else None
        summary["D_lt_S"] = (D_curv < S_curv) if (D_curv is not None and S_curv is not None) else None
        summary["P_lt_S"] = (P_curv < S_curv) if (P_curv is not None and S_curv is not None) else None
        summary["S_lt_F"] = (S_curv < F_curv) if (S_curv is not None and F_curv is not None) else None

        summary["basis_ranks"] = {
            "P": p_rank, "D": k_d, "S": Bs.shape[1], "residual": res_rank,
        }

        by_layer[str(layer)] = {
            "subspaces": subspaces,
            "summary":   summary,
        }

    # ── Build depth trajectories (subspace -> metric -> [values per layer]) ───
    # This is a convenience view for plotting: one trajectory per metric per subspace.
    depth_trajectories = {}
    tracked_metrics = [
        "curvature_ratio", "local_pr", "global_pr", "pr_folding_ratio",
        "info_density", "total_variance", "mean_knn_dist", "mean_global_dist",
    ]
    for sub_name in ["P", "D", "S", "residual"]:
        traj = {"layers": [int(l) for l in sorted(by_layer.keys(), key=int)]}
        for metric in tracked_metrics:
            vals = []
            for l in traj["layers"]:
                v = by_layer[str(l)].get("subspaces", {}).get(sub_name, {}).get(metric)
                vals.append(v)
            traj[metric] = vals
        # P-specific extras
        if sub_name == "P":
            traj["softmax_gradient"] = [
                by_layer[str(l)].get("subspaces", {}).get("P", {}).get("softmax_gradient_mean")
                for l in traj["layers"]
            ]
            traj["pr_adaptive_rank"] = [
                by_layer[str(l)].get("subspaces", {}).get("P", {}).get("pr_adaptive_rank")
                for l in traj["layers"]
            ]
        depth_trajectories[sub_name] = traj

    # ── Aggregate ordering check across layers ────────────────────────────────
    n_mono = sum(
        1 for l in by_layer.values()
        if l.get("summary", {}).get("ordering_PDSF_monotonic") is True
    )
    n_pairwise = {
        "P_lt_D": sum(1 for l in by_layer.values() if l.get("summary", {}).get("P_lt_D") is True),
        "D_lt_S": sum(1 for l in by_layer.values() if l.get("summary", {}).get("D_lt_S") is True),
        "S_lt_F": sum(1 for l in by_layer.values() if l.get("summary", {}).get("S_lt_F") is True),
        "P_lt_S": sum(1 for l in by_layer.values() if l.get("summary", {}).get("P_lt_S") is True),
    }
    n_total = len(by_layer)

    result = {
        "by_layer": by_layer,
        "depth_trajectories": depth_trajectories,
        "metadata": {
            "pipeline_version":  "v11.3",
            "part":              "L",
            "part_label":        "complexity_depth_profile",
            "description":       "Baseline manifold complexity per PDSF subspace at every layer",
            "n_prompts":         n_prompts,
            "d_model":           d_model,
            "n_layers":          len(layers),
            "layers":            layers,
            "k_max":             k_max,
            "s_small_r":         s_small_r,
            "k_neighbors":       k_neighbors,
            "n_groups":          len(set(group_ids)),
            "group_ids_unique":  sorted(set(group_ids)),
        },
        "ordering_summary": {
            "expected_ordering":      "P < D < S < F (curvature_ratio)",
            "n_layers_monotonic":     n_mono,
            "n_layers_total":         n_total,
            "fraction_monotonic":     round(n_mono / max(n_total, 1), 3),
            "pairwise_counts":        n_pairwise,
        },
    }

    if verbose:
        print(f"  Part L complete: {len(layers)} layers, {n_prompts} prompts")
        print(f"    Ordering P<D<S<F monotonic at {n_mono}/{n_total} layers ({100*n_mono/max(n_total,1):.0f}%)")
        for pair, cnt in n_pairwise.items():
            print(f"      {pair}: {cnt}/{n_total}")
        # Print final-layer curvature ratios
        last_key = str(layers[-1])
        last_summary = by_layer.get(last_key, {}).get("summary", {})
        for sub in ["D", "P", "S", "residual"]:
            cr = last_summary.get(f"{sub}_curvature_ratio")
            if cr is not None:
                print(f"    {sub:>8s} curvature_ratio @ L{layers[-1]}: {cr:.4f}")

    return result


print("✓ compute_part_L_complexity_depth_profile available")

print("✓ SpecA analysis functions available from specA_analysis.py")
print("  + compute_energy_spectrum, compute_resolution_geometry, compute_resolution_geometry_stratified (v6/v11.1 additions)")


In [ ]:
# ============================================================
# PIPELINE FUNCTIONS
# ============================================================
#
# ARCHITECTURE:
#   run_full_pipeline(model_key)          ← Entry point (Cell 24 calls this)
#     ├── load_model_for_pipeline()        Load/reuse HuggingFace model
#     ├── Phase 1-3: extract_hidden_states()  Forward pass → hidden states
#     ├── build_extraction_object()        Package into ModelExtraction
#     ├── build_unified_P_bases()          Unembedding vectors → per-prompt P bases
#     ├── Phase 4: run_consolidated_specA_analysis()  ← All Parts A-H (SpecA)
#     │     ├── Precompute: H_perp, D_basis, PR per layer
#     │     ├── compute_part_A_PR()
#     │     ├── compute_part_B_family_separation()
#     │     ├── compute_part_C_factor_effects()
#     │     ├── Part D: inline (uses precomputed D_basis)
#     │     ├── compute_part_E_mu_landscape()
#     │     ├── compute_part_F_rotation_with_per_prompt()
#     │     ├── compute_part_H_injectivity()
#     │     ├── compute_energy_spectrum()
#     │     └── compute_resolution_geometry()
#     ├── Phase 5A: run_part_G_for_pipeline()  SpecA Part G (PDSF interventions)
#     ├── Phase 5B: SpecB geometry (extraction + Parts A-H + Part G)
#     ├── Phase 5C: Diverse geometry (group-level + regime-level + Part G)
#     ├── Phase 6: SpecB continuation experiments
#     ├── Phase 7: Prompt analysis (token distributions)
#     ├── Phase 8: Diverse continuation experiments
#     └── Final: Geometry comparison (all prompt sets)
#
# DATA FLOW:
#   Prompts (JSON) → forward pass → layer_to_H (dict of tensors)
#   → ModelExtraction (dataclass) → P_bases (per-prompt unembedding vectors, unified)
#   → H_perp = H - project(H, P) → SVD → D_basis, S_basis
#   → Per-part analysis → JSON result files
#
# CACHING:
#   Hidden states are cached as .npz files in /workspace/cache/
#   If cache exists, extraction is skipped (saves ~5 min per model).
#   Cache includes: layer_to_H, group_ids, variant_ids, model_id.
#
# OUTPUT FILES (per model, in /workspace/results/Geometry/{model_name}/):
#   {model}-part_a_pr.json              PR(P) and PR(P⊥) per layer
#   {model}-part_b_family_sep.json       Within/between group angles
#   {model}-part_c_factor_effects.json   Cohen's d per factor A/B/C/D
#   {model}-part_d_pr_trajectory.json    PR(D) across layers
#   {model}-part_e_mu_landscape.json     μ energy decomposition
#   {model}-part_f_rotation.json         D rotation between layers
#   {model}-part_f_per_prompt.json       Per-prompt P-D-S trajectories
#   {model}-part_g_scramble.json         S-scramble causal results
#   {model}-part_h_injectivity.json      Near-collision analysis
#   {model}-energy_spectrum.json          Singular value spectrum
#   {model}-resolution_geometry.json     Manifold curvature per PDSF subspace
# ============================================================

def _safe_unembed_to_numpy(model):
    """Extract unembedding weight, handling meta tensors from device_map='auto'."""
    w = get_unembed_weight(model)
    if w.device.type == 'meta':
        # Meta tensor — find the actual parameter on a real device
        # Try lm_head first, then embed_tokens transposed
        for name, param in model.named_parameters():
            if 'lm_head' in name and 'weight' in name:
                if param.device.type != 'meta':
                    return param.detach().cpu().float().numpy()
            if 'embed_tokens' in name and 'weight' in name:
                if param.device.type != 'meta':
                    return param.detach().cpu().float().numpy()
        # Last resort: force-materialize by moving to first available device
        devices = [p.device for p in model.parameters() if p.device.type != 'meta']
        if devices:
            return w.to(devices[0]).detach().cpu().float().numpy()
        raise RuntimeError("All model weights are on meta device — model not properly loaded")
    return w.detach().cpu().float().numpy()


def save_json(data, filepath):
    """Save data to JSON file."""
    import json
    from pathlib import Path
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    return str(filepath)
import gc
import traceback
from datetime import datetime, timezone
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def create_model_bundle(model, tokenizer, model_id):
    """Create a ModelBundle from loaded model and tokenizer."""
    input_device = get_model_input_device(model)
    model_config = model.config
    
    bundle = ModelBundle(
        model=model,
        tokenizer=tokenizer,
        model_id=model_id,
        device=input_device,
        n_layers=model_config.num_hidden_layers,
        hidden_dim=model_config.hidden_size,
        vocab_size=model_config.vocab_size,
        is_quantized=hasattr(model_config, 'quantization_config'),
    )
    return bundle


def load_model_for_pipeline(model_id: str, use_4bit: bool = False, existing_bundle=None):
    """Load model and create bundle."""
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    try:
        import hf_xet  # Required for large file downloads (xet-powered)
    except ImportError:
        pass
    
    cache_dir = "/workspace/hf_cache"
    
    if existing_bundle is not None:
        if hasattr(existing_bundle, 'model_id') and existing_bundle.model_id == model_id:
            print(f"  ✓ Reusing loaded model: {model_id}")
            return existing_bundle
    
    print(f"  Loading {model_id}...")
    
    # Download with retry
    from huggingface_hub import snapshot_download
    import time as _time
    
    # Ensure module-level flags are available even if cells were skipped
    if "PROMPT_ANALYSIS_AVAILABLE" not in globals():
        globals()["PROMPT_ANALYSIS_AVAILABLE"] = False
    PROMPT_ANALYSIS_AVAILABLE = globals()["PROMPT_ANALYSIS_AVAILABLE"]
    import os
    
    if not os.path.isdir(model_id):
        for _attempt in range(3):
            try:
                snapshot_download(model_id, resume_download=True, max_workers=1, cache_dir=cache_dir)
                break
            except Exception as _e:
                if _attempt < 2:
                    _time.sleep(30 * (_attempt + 1))
                else:
                    raise
    
    if use_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=quant_config,
            device_map="auto", cache_dir=cache_dir, trust_remote_code=True,
        )
    else:
        pre_quantized = any(x in model_id.lower() for x in ['gpt-oss', 'mxfp4', 'fp8'])
        is_gemma2 = 'gemma-2' in model_id.lower() or 'gemma2' in model_id.lower()
        
        load_kwargs = {
            "device_map": "auto", "cache_dir": cache_dir,
            "trust_remote_code": True, "low_cpu_mem_usage": True,
        }
        
        # Gemma-2 models need special handling to avoid NaN in hidden states
        if is_gemma2:
            print(f"  ⚠️ Gemma-2 detected - using bfloat16 + eager attention")
            load_kwargs["torch_dtype"] = torch.bfloat16
            load_kwargs["attn_implementation"] = "eager"  # Avoid flash attention issues
        elif not pre_quantized:
            load_kwargs["torch_dtype"] = torch.float16
            
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    return create_model_bundle(model, tokenizer, model_id)


def build_extraction_object(layer_to_H, meta_list, model_id, group_ids_list, variant_ids_list):
    """Build ModelExtraction from extracted hidden states."""
    layers_dict = {
        idx: LayerTensors(H=H.numpy().astype(np.float32) if isinstance(H, torch.Tensor) else H.astype(np.float32))
        for idx, H in layer_to_H.items()
    }
    return ModelExtraction(
        model_id=model_id,
        layers=layers_dict,
        group_ids=group_ids_list,
        variant_ids=variant_ids_list,
        meta_rows=meta_list,
    )

def build_unified_P_bases(pred_ids, unembed_np, extraction):
    """Build per-prompt P bases from model's predicted tokens (unified approach).
    
    P = unembedding vector for each prompt's predicted next token.
    P is rank-1 per prompt and layer-independent.
    The same dict is replicated per layer for API compatibility with Parts A-F.
    
    Args:
        pred_ids: List[int] — argmax token ID per prompt
        unembed_np: Unembedding weight matrix, shape (vocab, hidden_dim)
        extraction: ModelExtraction (to get layer keys)
    
    Returns:
        Dict[layer_idx, Dict[int, np.ndarray]] — P_bases[layer][prompt_idx] = (d, 1)
    """
    from pds_continuation import compute_P_bases_from_predictions
    
    # Compute per-prompt P bases (layer-independent)
    flat_P, P_info = compute_P_bases_from_predictions(pred_ids, unembed_np)
    
    # Replicate across layers for API compat
    # P is layer-independent (unembedding vectors); same dict replicated per layer
    P_bases = {}
    for layer_idx in sorted(extraction.layers.keys()):
        P_bases[layer_idx] = flat_P
    
    return P_bases, P_info

def save_extraction_to_cache(layer_to_H, meta_list, model_id, bundle, cache_npz, cache_meta, group_ids_list, variant_ids_list):
    """Save extraction to cache files."""
    save_dict = {f"H_{layer}": H.numpy().astype(np.float32) if isinstance(H, torch.Tensor) else H.astype(np.float32)
                 for layer, H in layer_to_H.items()}
    np.savez_compressed(cache_npz, **save_dict)
    
    with open(cache_meta, 'w') as f:
        json.dump({"model_id": model_id, "n_samples": len(meta_list),
                   "n_layers": bundle.n_layers, "hidden_dim": bundle.hidden_dim,
                   "group_ids": group_ids_list, "variant_ids": variant_ids_list}, f, indent=2)


def load_extraction_from_cache(cache_npz, cache_meta, bundle, meta_list, expected_tokens_list):
    """Load extraction from cache."""
    with open(cache_meta) as f:
        cache_data = json.load(f)
    
    loaded = np.load(cache_npz)
    layer_to_H = {int(key.split("_")[1]): torch.from_numpy(loaded[key])
                  for key in loaded.files if key.startswith("H_")}
    
    extraction = build_extraction_object(
        layer_to_H, meta_list, cache_data["model_id"],
        cache_data["group_ids"], cache_data["variant_ids"]
    )
    
    # Meta-safe unembedding extraction
    unembed_np = _safe_unembed_to_numpy(bundle.model)
    # Generate pred_ids if not in cache
    pred_ids = cache_data.get("pred_ids")
    if pred_ids is None:
        print("  ⚠ No pred_ids in cache, generating from expected_tokens...")
        pred_ids = []
        for tok in expected_tokens_list:
            ids = bundle.tokenizer.encode(tok, add_special_tokens=False)
            if not ids:
                ids = bundle.tokenizer.encode(" " + tok.strip(), add_special_tokens=False)
            pred_ids.append(ids[0] if ids else 0)
    P_bases, _ = build_unified_P_bases(pred_ids, unembed_np, extraction)
    
    return layer_to_H, extraction, P_bases, pred_ids, unembed_np


def convert_specb_data_to_rows(specb_data: dict, tier: str = "standard") -> list:
    """Convert specb_data dict to list of PromptRow objects for pds_continuation.
    
    Supports both old format (nested groups/variants) and new format (flat prompts list).
    """
    rows = []
    
    # New format: flat list with 'prompts' key
    if "prompts" in specb_data and isinstance(specb_data["prompts"], list):
        for p in specb_data["prompts"]:
            rows.append(PromptRow(
                group_id=p["group_id"],
                variant_id=p["variant_id"],
                prompt=p["prompt"],
                category=p.get("category", ""),
                regime=p.get("regime", ""),
            ))
        return rows
    
    # Old format: nested groups/variants with tier filtering
    tier_groups = {
        "lite": [f"group_{i:02d}" for i in range(1, 7)],
        "standard": [f"group_{i:02d}" for i in range(1, 11)],
        "full": [f"group_{i:02d}" for i in range(1, 13)],
    }
    included = set(tier_groups.get(tier, []))
    
    for group in specb_data.get("groups", []):
        gid = group["group_id"]
        include = not included or gid in included or any(gid.startswith(g) or g in gid for g in included)
        if not include:
            continue
        category = group.get("category", "")
        for variant in group.get("variants", []):
            rows.append(PromptRow(
                group_id=gid,
                variant_id=variant["variant_id"],
                prompt=variant["prompt"],
                category=category
            ))
    return rows


# ============================================================
# CONSOLIDATED SPECA ANALYSIS (Parts A-H)
# ============================================================

def run_consolidated_specA_analysis(
    extraction,
    P_bases,
    unembed_np,
    run_parts: dict,
    output_files: dict,
    model_output_dir,
    k_max: int = 64,
    s_small_r: int = 16,
    verbose: bool = True,
    resolution_geometry_layers: str = "all",
) -> Dict[str, Any]:
    """Run all SpecA analysis parts with shared precomputation."""
    import time
    
    results = {
        "model_id": extraction.model_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "parts_run": [],
        "parts_failed": [],
        "timings": {},
    }
    
    layers_sorted = sorted(extraction.layers.keys())
    last_layer = max(layers_sorted)
    
    if verbose:
        print(f"  Layers: {len(layers_sorted)}, Samples: {len(extraction.group_ids)}")
    
    # Precomputation for Parts D, E, F
    precompute_needed = any([
        run_parts.get("part_d", False),
        run_parts.get("part_e", False),
        run_parts.get("part_f", False),
    ])
    
    precomputed = {}
    if precompute_needed:
        start = time.time()
        if verbose:
            print("  Precomputing shared structures...")
        
        H_perp_by_layer = {}
        D_basis_by_layer = {}
        k_D_by_layer = {}
        PR_P_perp_by_layer = {}
        
        for layer in layers_sorted:
            H = extraction.layers[layer].H
            
            # Check for NaN/Inf in H
            if np.isnan(H).any() or np.isinf(H).any():
                if verbose:
                    print(f"    ⚠️ Layer {layer}: H contains NaN/Inf, cleaning...")
                H = np.nan_to_num(H, nan=0.0, posinf=0.0, neginf=0.0)
            
            H_P = np.zeros_like(H)
            for i in range(H.shape[0]):
                Bp = P_bases[layer].get(i)
                if Bp is not None and Bp.size > 0:
                    H_P[i:i+1, :] = project_onto_basis(H[i:i+1, :], Bp)
            H_perp = H - H_P
            
            # Check for NaN/Inf after projection
            if np.isnan(H_perp).any() or np.isinf(H_perp).any():
                if verbose:
                    print(f"    ⚠️ Layer {layer}: H_perp contains NaN/Inf, cleaning...")
                H_perp = np.nan_to_num(H_perp, nan=0.0, posinf=0.0, neginf=0.0)
            
            H_perp_by_layer[layer] = H_perp
            
            pr_perp = participation_ratio(H_perp, center=True)
            if not np.isfinite(pr_perp):
                pr_perp = 2.0  # Default
            PR_P_perp_by_layer[layer] = pr_perp
            k = max(1, min(int(round(pr_perp)), k_max, H_perp.shape[1],
                           H_perp.shape[0]-1 if H_perp.shape[0] > 1 else 1))
            k_D_by_layer[layer] = k
            
            # Robust PCA with fallback
            try:
                D_basis = pca_basis(H_perp, k=k, center=True)
            except np.linalg.LinAlgError as e:
                if verbose:
                    print(f"    ⚠️ Layer {layer}: SVD failed ({e}), using fallback")
                # Fallback: use truncated SVD with scipy which is more robust
                try:
                    from scipy.sparse.linalg import svds
                    H_centered = H_perp - H_perp.mean(axis=0, keepdims=True)
                    # svds needs k < min(n, d) - 1
                    k_safe = min(k, min(H_centered.shape) - 2)
                    if k_safe > 0:
                        U, s, Vt = svds(H_centered.astype(np.float64), k=k_safe)
                        # svds returns in ascending order, reverse it
                        D_basis = Vt[::-1].T.astype(np.float32)
                    else:
                        D_basis = np.zeros((H_perp.shape[1], 1), dtype=np.float32)
                except Exception as e2:
                    if verbose:
                        print(f"    ⚠️ Layer {layer}: Fallback also failed ({e2}), using zeros")
                    D_basis = np.zeros((H_perp.shape[1], k), dtype=np.float32)
            
            D_basis_by_layer[layer] = D_basis
        
        precomputed = {
            "H_perp": H_perp_by_layer,
            "D_basis": D_basis_by_layer,
            "k_D": k_D_by_layer,
            "PR_P_perp": PR_P_perp_by_layer,
        }
        results["timings"]["precompute"] = time.time() - start
    
    # Part A
    if run_parts.get("part_a", False):
        try:
            start = time.time()
            part_a = compute_part_A_PR(extraction, P_bases, center=True)
            save_json(part_a, output_files["part_a"])
            results["timings"]["part_a"] = time.time() - start
            results["parts_run"].append("part_a")
            s = part_a.get("summary", {})
            if verbose:
                print(f"    Part A: PR(P)={s.get('PR_P',0):.2f}, PR(P⊥)={s.get('PR_P_perp',0):.2f} ({results['timings']['part_a']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_a: {e}")
            if verbose:
                print(f"    Part A: FAILED - {e}")
    
    # Part B
    if run_parts.get("part_b", False):
        try:
            start = time.time()
            part_b = compute_part_B_family_separation(extraction, P_bases, last_layer, k_max=k_max)
            save_json(part_b, output_files["part_b"])
            results["timings"]["part_b"] = time.time() - start
            results["parts_run"].append("part_b")
            if verbose:
                w = part_b.get('within_family_mean_angle_deg', 0)
                b = part_b.get('between_family_mean_angle_deg', 0)
                print(f"    Part B: Within={w:.1f}°, Between={b:.1f}° ({results['timings']['part_b']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_b: {e}")
            if verbose:
                print(f"    Part B: FAILED - {e}")
    
    # Part C
    if run_parts.get("part_c", False):
        try:
            start = time.time()
            part_c = compute_part_C_factor_effects(extraction, P_bases, last_layer, k_max=k_max)
            save_json(part_c, output_files["part_c"])
            results["timings"]["part_c"] = time.time() - start
            results["parts_run"].append("part_c")
            if verbose:
                factors = part_c.get("factors", {})
                d_str = ", ".join([f"{k}:{v.get('cohens_d',0):.2f}" for k,v in factors.items()])
                print(f"    Part C: {d_str} ({results['timings']['part_c']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_c: {e}")
            if verbose:
                print(f"    Part C: FAILED - {e}")
    
    # Part D (uses precomputed)
    if run_parts.get("part_d", False) and precomputed:
        try:
            start = time.time()
            part_d = {"by_layer": {}}
            for layer in layers_sorted:
                H_perp = precomputed["H_perp"][layer]
                D_basis = precomputed["D_basis"][layer]
                k = precomputed["k_D"][layer]
                pr_perp = precomputed["PR_P_perp"][layer]
                H_D = project_onto_basis(H_perp, D_basis)
                pr_d = participation_ratio(H_D, center=True)
                part_d["by_layer"][str(layer)] = {
                    "PR_P_perp": float(pr_perp), "k": int(k), "PR_D": float(pr_d), "n": int(H_perp.shape[0])
                }
            save_json(part_d, output_files["part_d"])
            results["timings"]["part_d"] = time.time() - start
            results["parts_run"].append("part_d")
            if verbose:
                pr_vals = [v["PR_D"] for v in part_d["by_layer"].values()]
                print(f"    Part D: PR(D)=[{min(pr_vals):.1f}, {max(pr_vals):.1f}] ({results['timings']['part_d']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_d: {e}")
            if verbose:
                print(f"    Part D: FAILED - {e}")
    
    # Part E
    if run_parts.get("part_e", False):
        try:
            start = time.time()
            part_e = compute_part_E_mu_landscape(extraction, P_bases, unembed_np, s_small_r=s_small_r, k_max=k_max)
            save_json(part_e, output_files["part_e"])
            results["timings"]["part_e"] = time.time() - start
            results["parts_run"].append("part_e")
            if verbose:
                sep = part_e.get("group_separation", {}).get("by_layer", {})
                last_key = str(max(extraction.layers.keys()))
                if last_key in sep:
                    spread = sep[last_key].get("centroid_spread", {})
                    d_s_ratio = spread.get("D_to_S_ratio", 0)
                    mean_dist = spread.get("mean_distance", 0)
                    print(f"    Part E: μ landscape, D/S ratio={d_s_ratio:.2f}, mean sep={mean_dist:.1f} ({results['timings']['part_e']:.1f}s)")
                else:
                    print(f"    Part E: μ landscape ({results['timings']['part_e']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_e: {e}")
            if verbose:
                print(f"    Part E: FAILED - {e}")
    
    # Part F
    part_f_per_prompt = None
    if run_parts.get("part_f", False):
        try:
            start = time.time()
            part_f, part_f_per_prompt = compute_part_F_rotation_with_per_prompt(
                extraction, P_bases, k_max=k_max, s_small_r=s_small_r
            )
            save_json(part_f, output_files["part_f"])
            save_json(part_f_per_prompt, output_files["per_prompt"])
            results["timings"]["part_f"] = time.time() - start
            results["parts_run"].append("part_f")
            if verbose:
                rot = part_f.get('rotation_mean_deg', 0)
                print(f"    Part F: Mean rotation={rot:.1f}° ({results['timings']['part_f']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_f: {e}")
            if verbose:
                print(f"    Part F: FAILED - {e}")
    
    # Part H
    if run_parts.get("part_h", True):
        try:
            start = time.time()
            identical_pairs = None
            if part_f_per_prompt and "aggregate_analysis" in part_f_per_prompt:
                agg = part_f_per_prompt["aggregate_analysis"]
                if "identical_trajectories" in agg and agg["identical_trajectories"]:
                    pairs_data = agg["identical_trajectories"].get("pairs", [])
                    identical_pairs = [(p["idx_1"], p["idx_2"]) for p in pairs_data]
            
            part_h = compute_part_H_injectivity(
                extraction,
                identical_pairs=identical_pairs,
                group_ids=extraction.group_ids if hasattr(extraction, 'group_ids') else None,
                variant_ids=extraction.variant_ids if hasattr(extraction, 'variant_ids') else None,
            )
            save_json(part_h, output_files["part_h"])
            results["timings"]["part_h"] = time.time() - start
            results["parts_run"].append("part_h")
            if verbose:
                n_collisions = part_h.get("n_potential_collisions", 0)
                print(f"    Part H: {n_collisions} potential collisions ({results['timings']['part_h']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"part_h: {e}")
            if verbose:
                print(f"    Part H: FAILED - {e}")
    
    
    # Part I: Energy Spectrum
    if run_parts.get("run_energy_spectrum", run_parts.get("part_i", True)):
        try:
            start = time.time()
            # Use a few key layers: first, 25%, 50%, 75%, last
            n_layers = len(layers_sorted)
            spectrum_layers = sorted(set([
                layers_sorted[0],
                layers_sorted[n_layers // 4],
                layers_sorted[n_layers // 2],
                layers_sorted[3 * n_layers // 4],
                layers_sorted[-1],
            ]))
            energy_spectrum = compute_energy_spectrum(
                extraction, P_bases, layers=spectrum_layers, n_dims=100
            )
            # Use output_files to prevent cross-prompt-set overwrite
            spectrum_file = output_files.get(
                "energy_spectrum",
                model_output_dir / f"{extraction.model_id.split('/')[-1]}-energy_spectrum.json"
            )
            # Add metadata for provenance tracking
            energy_spectrum["_metadata"] = {
                "prompt_set": output_files.get("_prompt_set_label", "unknown"),
                "n_prompts": len(extraction.group_ids) if hasattr(extraction, "group_ids") else None,
                "n_layers_sampled": len(spectrum_layers),
                "layers_sampled": spectrum_layers,
                "model_id": extraction.model_id,
                "pipeline_version": "v11.1",
            }
            save_json(energy_spectrum, spectrum_file)
            results["timings"]["energy_spectrum"] = time.time() - start
            results["parts_run"].append("energy_spectrum")
            if verbose:
                last_key = str(layers_sorted[-1])
                gl = energy_spectrum["by_layer"].get(last_key, {}).get("global", {})
                if gl:
                    k = gl.get("k_D", 0)
                    ek = gl.get("energy_at_k_D", 0)
                    t10 = gl.get("top_10_energy", 0)
                    print(f"    Energy spectrum: top-{k} capture {ek:.1%}, top-10 capture {t10:.1%} ({results['timings']['energy_spectrum']:.1f}s)")
        except Exception as e:
            results["parts_failed"].append(f"energy_spectrum: {e}")
            if verbose:
                print(f"    Energy spectrum: FAILED - {e}")
    
    
    # Part J: Resolution Geometry
    if run_parts.get("run_resolution_geometry", run_parts.get("part_j", True)):
        try:
            start = time.time()
            # Use local k_max, s_small_r (not globals) so Diverse gets correct params
            # PR-adaptive P. layers=None runs all; "sparse" runs 4 depths.
            if resolution_geometry_layers == "sparse":
                n_layers = len(layers_sorted)
                rg_layers = sorted(set([
                    layers_sorted[max(0, n_layers // 4 - 1)],       # ~25%
                    layers_sorted[max(0, n_layers // 2 - 1)],       # ~50%
                    layers_sorted[max(0, 3 * n_layers // 4 - 1)],   # ~75%
                    layers_sorted[-1],                               # final
                ]))
                if verbose:
                    print(f"    Part J: sparse mode — {len(rg_layers)} layers: {rg_layers}")
            else:
                rg_layers = None  # all layers
            resolution_geo = compute_resolution_geometry(
                extraction, P_bases, unembed_np, layers=rg_layers,
                k_max=k_max, s_small_r=s_small_r, k_neighbors=8,
            )
            # Use output_files to prevent cross-prompt-set overwrite
            res_file = output_files.get(
                "resolution_geometry",
                model_output_dir / f"{extraction.model_id.split('/')[-1]}-resolution_geometry_v11.json"
            )
            # Add metadata for provenance tracking
            resolution_geo["_metadata"] = {
                "prompt_set": output_files.get("_prompt_set_label", "unknown"),
                "n_prompts": len(extraction.group_ids) if hasattr(extraction, "group_ids") else None,
                "k_max": k_max,
                "s_small_r": s_small_r,
                "k_neighbors": 8,
                "n_layers_sampled": len(spectrum_layers),
                "layers_sampled": spectrum_layers,
                "model_id": extraction.model_id,
                "pipeline_version": "v11.1",
            }
            save_json(resolution_geo, res_file)
            results["timings"]["resolution_geometry"] = time.time() - start
            results["parts_run"].append("resolution_geometry")
            if verbose:
                last_key = str(layers_sorted[-1])
                rg = resolution_geo["by_layer"].get(last_key, {}).get("summary", {})
                if rg:
                    # Include P curvature ratio in verbose output
                    p_c = rg.get("P_curvature_ratio", "?")
                    d_c = rg.get("D_curvature_ratio", "?")
                    s_c = rg.get("S_curvature_ratio", "?")
                    r_c = rg.get("residual_curvature_ratio", "?")
                    mono = rg.get("ordering_monotonic", "?")
                    # Extract control results from final layer
                _rg_layers = sorted(resolution_geo.get("by_layer", {}).keys(), key=int)
                _rg_last = resolution_geo["by_layer"][_rg_layers[-1]] if _rg_layers else {}
                _pca_ctrl = _rg_last.get("pca_control", {})
                _shuf_ctrl = _rg_last.get("shuffled_control", {})
                _summary = _rg_last.get("summary", {})
                
                # PCA contiguous control curvatures
                _pca_bins = _pca_ctrl.get("contiguous_bins", {})
                _pca_p = _pca_bins.get("bin_P", {}).get("curvature_ratio", "?")
                _pca_d = _pca_bins.get("bin_D", {}).get("curvature_ratio", "?")
                _pca_s = _pca_bins.get("bin_S", {}).get("curvature_ratio", "?")
                _pca_f = _pca_bins.get("bin_F", {}).get("curvature_ratio", "?")
                _pca_mono = _pca_ctrl.get("contiguous_ordering_monotonic", "?")
                
                # Shuffled control means
                _shuf_bins = _shuf_ctrl.get("bins", {})
                _shuf_p = _shuf_bins.get("bin_P", {}).get("curvature_ratio_mean", "?")
                _shuf_d = _shuf_bins.get("bin_D", {}).get("curvature_ratio_mean", "?")
                _shuf_s = _shuf_bins.get("bin_S", {}).get("curvature_ratio_mean", "?")
                _shuf_f = _shuf_bins.get("bin_F", {}).get("curvature_ratio_mean", "?")
                
                # P-PCA alignment
                _p_pca_angle = _summary.get("P_vs_PCA_mean_angle_deg", "?")
                _p_var_frac = _summary.get("P_variance_fraction_of_H", "?")
                _p_rank = _rg_last.get("basis_ranks", {}).get("P", "?")
                
                print(f"    Resolution geometry ({results['timings']['resolution_geometry']:.1f}s, {len(_rg_layers)} layers):")
                print(f"      PDSF curvature (final):  P={p_c}  D={d_c}  S={s_c}  F={r_c}  monotonic={mono}")
                print(f"      PCA control (contiguous): P={_pca_p}  D={_pca_d}  S={_pca_s}  F={_pca_f}  monotonic={_pca_mono}")
                print(f"      PCA control (shuffled):   P={_shuf_p}  D={_shuf_d}  S={_shuf_s}  F={_shuf_f}")
                print(f"      P-PCA alignment: angle={_p_pca_angle}°  P_var_frac={_p_var_frac}  P_rank={_p_rank}")
        except Exception as e:
            results["parts_failed"].append(f"resolution_geometry: {e}")
            if verbose:
                print(f"    Resolution geometry: FAILED - {e}")
    
    # Part K: Cosine Similarity Subspace Discrimination
    if run_parts.get("part_k", True):
        try:
            start = time.time()
            part_k = compute_part_K_cosine_discrimination(
                extraction,
                P_bases,
                unembed_np,
                group_ids=extraction.group_ids,
                layer=PART_K_LAYER,
                k_max=k_max,
                s_small_r=s_small_r,
                n_shuffles=PART_K_N_SHUFFLES,
                regime_ids=getattr(extraction, "regime_ids", None),
            )
            out_k = output_files.get("part_k")
            if out_k:
                import json as _json
                with open(out_k, "w") as _f:
                    _json.dump(part_k, _f, indent=2, default=float)
            results["part_k"] = part_k
            results["timings"]["part_k"] = time.time() - start
            results["parts_run"].append("part_k")
            if verbose:
                print(f"    Part K: cosine discrimination in {results['timings']['part_k']:.1f}s")
        except Exception as e:
            results["parts_failed"].append(f"part_k: {e}")
            if verbose:
                print(f"    Part K: FAILED - {e}")

    total = sum(results["timings"].values())
    if verbose:
        print(f"  ✓ Analysis: {len(results['parts_run'])} parts in {total:.1f}s")
    
    return results


# ============================================================
# PART G: S-SCRAMBLE
# ============================================================

def run_part_G_for_pipeline(
    bundle,
    layer_to_H,
    prompts,
    group_ids,
    expected_tokens,
    output_file,
    scramble_layers,
    n_scrambles,
    s_rank,
    seed,
    n_prompts=None,
    regime_ids=None,
    verbose=True,
    compute_kl=False,
    kl_topk=2048,
    compute_spirality=False,
    spirality_n_pc_pairs=3,
):
    """Run Part G PDSF Interventions experiment."""
    
    # Subset if requested — group-stratified when group_ids available, stride fallback otherwise
    if n_prompts is not None and n_prompts < len(prompts):
        _unique_groups = list(dict.fromkeys(group_ids)) if group_ids else []  # preserves order
        if len(_unique_groups) > 1:
            # Group-stratified: round-robin across groups so every group is represented
            # before any group gets a second prompt. Guarantees coverage of all domains
            # regardless of how the prompt file is sorted.
            from collections import defaultdict as _dd
            _gid_to_idxs = _dd(list)
            for _i, _gid in enumerate(group_ids):
                _gid_to_idxs[_gid].append(_i)
            indices = []
            _round = 0
            while len(indices) < n_prompts:
                _added = False
                for _gid in _unique_groups:
                    _pool = _gid_to_idxs[_gid]
                    if _round < len(_pool):
                        indices.append(_pool[_round])
                        _added = True
                        if len(indices) == n_prompts:
                            break
                if not _added:
                    break  # all groups exhausted before n_prompts reached
        else:
            # Fallback: uniform stride (no group structure available)
            _step = max(1, len(prompts) // n_prompts)
            indices = list(range(0, len(prompts), _step))[:n_prompts]
        prompts_subset = [prompts[i] for i in indices]
        expected_subset = [expected_tokens[i] for i in indices]
        group_ids_subset = [group_ids[i] for i in indices]
        regime_ids_subset = [regime_ids[i] for i in indices] if regime_ids is not None else None
        H_cached_subset = {
            idx: (H.numpy() if isinstance(H, torch.Tensor) else H)[indices].astype(np.float32)
            for idx, H in layer_to_H.items()
        }
    else:
        # Use all prompts (n_prompts is None or >= len(prompts))
        prompts_subset = prompts
        expected_subset = expected_tokens
        group_ids_subset = group_ids
        regime_ids_subset = regime_ids
        H_cached_subset = {
            idx: (H.numpy() if isinstance(H, torch.Tensor) else H).astype(np.float32)
            for idx, H in layer_to_H.items()
        }
    
    if verbose:
        print(f"  Prompts: {len(prompts_subset)}, Scrambles: {n_scrambles}")
    
    resolved_layers = resolve_scramble_layers(bundle.n_layers, scramble_layers)
    if verbose:
        print(f"  Scramble layers: {resolved_layers}")
    
    device = get_model_input_device(bundle.model)
    all_layers = list(range(bundle.n_layers + 1))
    
    all_results = {}
    for scramble_layer in resolved_layers:
        if verbose:
            print(f"  Layer {scramble_layer}...")
        
        inv_results = run_invariance_control(
            bundle.model, bundle.tokenizer, prompts_subset[:10],
            all_layers, device, verbose=False
        )
        
        scr_results = run_scramble_experiment(
            bundle.model, bundle.tokenizer,
            prompts_subset, expected_subset, group_ids_subset,
            H_cached_subset, all_layers, scramble_layer, device,
            regime_ids=regime_ids_subset,
            n_scrambles=n_scrambles, s_rank=s_rank, verbose=False, seed=seed,
            compute_kl=compute_kl, kl_topk=kl_topk,
            subspaces=GEOMETRY_SUBSPACES,
            intervention_types=GEOMETRY_INTERVENTION_TYPES,
            attenuate_alpha=GEOMETRY_ATTENUATE_ALPHA,
            compute_spirality=compute_spirality,
            spirality_n_pc_pairs=spirality_n_pc_pairs,
        )
        
        all_results[scramble_layer] = {
            "invariance_control": inv_results,
            "scramble_experiment": scr_results,
        }
    
    output = {
        "model_id": bundle.model_id,
        "scramble_layers": resolved_layers,
        "n_scrambles": n_scrambles,
        "s_rank": s_rank,
        "n_prompts_used": len(prompts_subset),
        "by_scramble_layer": all_results,
    }

    save_json(output, output_file)
    if verbose:
        print(f"  ✓ Part G saved: {output_file}")
    return output


# ============================================================
# PART M PIPELINE WRAPPER — Spirality Measures
# ============================================================
# Three phases:
#   Phase A (baseline): runs from layer_to_H cache, alongside Parts A-F.
#     Gated on: run_geometry_part_m (independent of Part G)
#   Phase B (intervention): piggybacks on Part G forward passes.
#     Gated on: run_geometry_part_m_intervention (requires part_g)
#   Phase C (correlation): post-hoc, joins Part M baseline with Part F.
#     Gated on: run_geometry_part_m (independent of Part G)
# ============================================================

def run_part_M_baseline_for_pipeline(
    layer_to_H,
    n_prompts,
    model_id,
    output_file,
    n_pc_pairs=3,
    verbose=True,
):
    """Part M Phase A: Compute baseline spirality from cached hidden states."""
    if pds_spirality is None:
        if verbose:
            print("  ⚠️  pds_spirality not available, skipping Part M baseline")
        return None

    result = compute_baseline_spirality_from_cache(
        layer_to_H, n_prompts,
        n_pc_pairs=n_pc_pairs,
        verbose=verbose,
    )
    output = {
        "model_id": model_id,
        "experiment_version": "1.0",
        "phase": "baseline",
        **result,
    }
    save_json(output, output_file)
    if verbose:
        print(f"  ✓ Part M baseline saved: {output_file}")
    return output


def run_part_M_correlation_for_pipeline(
    part_m_baseline,
    part_f_per_prompt_file,
    model_id,
    output_file,
    verbose=True,
):
    """Part M Phase C: Correlate spirality with Part F rotation measures."""
    if pds_spirality is None:
        if verbose:
            print("  ⚠️  pds_spirality not available, skipping Part M correlation")
        return None

    if part_m_baseline is None:
        if verbose:
            print("  ⚠️  Part M baseline not available, skipping correlation")
        return None

    # Load Part F per-prompt data
    import json as _json
    try:
        with open(part_f_per_prompt_file, 'r') as f:
            part_f_data = _json.load(f)
    except (FileNotFoundError, _json.JSONDecodeError) as e:
        if verbose:
            print(f"  ⚠️  Could not load Part F per-prompt data: {e}")
        return None

    result = correlate_spirality_with_rotation(part_m_baseline, part_f_data)
    output = {
        "model_id": model_id,
        "experiment_version": "1.0",
        "phase": "correlation",
        **result,
    }
    save_json(output, output_file)
    if verbose:
        print(f"  ✓ Part M correlation saved: {output_file}")
        # Print top correlations
        corrs = result.get("correlations", {})
        pairs = [(k, v["pearson_r"]) for k, v in corrs.items()
                 if v.get("pearson_r") is not None]
        if pairs:
            pairs.sort(key=lambda x: abs(x[1]), reverse=True)
            print(f"    Top correlations (spirality × rotation):")
            for k, r in pairs[:3]:
                p = corrs[k]["p_value"]
                print(f"      {k}: r={r:.3f}, p={p:.4f}")
    return output


def extract_part_m_from_part_g(part_g_output, model_id, output_file, verbose=True):
    """Part M Phase B post-processing: extract spirality data from Part G output."""
    part_m = {
        "model_id": model_id,
        "experiment_version": "1.0",
        "source": "part_g_piggyback",
        "scramble_layers": part_g_output.get("scramble_layers"),
        "n_prompts_used": part_g_output.get("n_prompts_used"),
        "by_scramble_layer": {},
    }

    for sl_key, sl_data in part_g_output.get("by_scramble_layer", {}).items():
        scr_exp = sl_data.get("scramble_experiment", {})

        per_prompt = []
        for pr in scr_exp.get("by_prompt", []):
            entry = {
                "prompt_idx": pr.get("prompt_idx"),
                "baseline": pr.get("spirality_baseline"),
                "by_subspace": {},
            }
            for sub, sub_data in pr.get("by_subspace", {}).items():
                entry["by_subspace"][sub] = {}
                for it, it_data in sub_data.get("by_intervention", {}).items():
                    disruptions = []
                    for scr in it_data.get("scrambles", []):
                        sd = scr.get("spirality_disruption")
                        if sd is not None:
                            disruptions.append(sd)
                    if disruptions:
                        entry["by_subspace"][sub][it] = disruptions
            per_prompt.append(entry)

        # Extract aggregate disruptions
        agg = {}
        for sub in scr_exp.get("aggregate", {}):
            agg[sub] = {}
            for it in scr_exp["aggregate"][sub]:
                sda = scr_exp["aggregate"][sub][it].get("spirality_disruption_agg")
                if sda:
                    agg[sub][it] = sda

        part_m["by_scramble_layer"][sl_key] = {
            "per_prompt": per_prompt,
            "aggregate": agg,
        }

    # Also compute causal correlation if data is present
    causal_corr = None
    if pds_spirality is not None:
        try:
            causal_corr = correlate_spirality_disruption_with_part_g(part_g_output)
        except Exception as e:
            if verbose:
                print(f"  ⚠️  Causal correlation failed: {e}")

    if causal_corr is not None:
        part_m["causal_correlation"] = causal_corr

    save_json(part_m, output_file)
    if verbose:
        print(f"  ✓ Part M intervention spirality saved: {output_file}")
        if causal_corr:
            for cond, corrs in causal_corr.get("correlations_by_condition", {}).items():
                key = "phase_r2_weighted_abs_change_vs_D_angle_deg"
                c = corrs.get(key, {})
                r = c.get("pearson_r")
                if r is not None:
                    print(f"    {cond}: spirality×angle r={r:.3f} (p={c['p_value']:.4f}, n={c['n']})")
    return part_m

# ============================================================
# PART L PIPELINE WRAPPER — Complexity Depth Profiles
# ============================================================
# Runs alongside Part G (uses the same layer_to_H cache).
# Outputs a separate JSON file with baseline manifold complexity
# trajectories — one curvature/PR/density measurement per PDSF
# subspace per layer.
# ============================================================

def run_part_L_for_pipeline(
    layer_to_H,
    group_ids,
    P_bases_by_layer,
    unembed_np,
    output_file,
    k_max: int = 64,
    s_small_r: int = 16,
    k_neighbors: int = 8,
    model_id: str = None,
    prompt_set_label: str = None,
    verbose: bool = True,
):
    """
    Run Part L: Complexity Depth Profiles and save results.

    This is a thin wrapper around compute_part_L_complexity_depth_profile
    that handles output file writing and metadata injection.

    Designed to be called immediately after Part G on each prompt set,
    since it uses the same layer_to_H cache and P_bases.

    Args:
        layer_to_H:       Dict[int, ndarray/Tensor] — cached hidden states per layer.
        group_ids:        List[str] — group label per prompt.
        P_bases_by_layer: Dict[layer][prompt_idx] -> P basis.
        unembed_np:       ndarray — unembedding matrix.
        output_file:      Path — where to save the JSON output.
        k_max:            int — max D rank.
        s_small_r:        int — S basis rank.
        k_neighbors:      int — k for kNN curvature.
        model_id:         str — model identifier (for metadata).
        prompt_set_label: str — e.g. "SpecA", "SpecB", "Diverse-group".
        verbose:          bool — print progress.
    """
    if verbose:
        print(f"  Part L: computing complexity depth profiles...")

    result = compute_part_L_complexity_depth_profile(
        layer_to_H=layer_to_H,
        group_ids=group_ids,
        P_bases_by_layer=P_bases_by_layer,
        unembed_weight=unembed_np,
        k_max=k_max,
        s_small_r=s_small_r,
        k_neighbors=k_neighbors,
        layers=None,  # all layers
        verbose=verbose,
    )

    # Inject pipeline metadata
    result["metadata"]["model_id"] = model_id
    result["metadata"]["prompt_set"] = prompt_set_label

    save_json(result, output_file)
    if verbose:
        print(f"  ✓ Part L saved: {output_file}")

    return result

    save_json(output, output_file)
    return output


# ============================================================
# MAIN PIPELINE FUNCTION
# ============================================================

def run_full_pipeline(model_key, options=None, specA_prompts_data=None, continuation_prompts_data=None):
    """Run complete SpecA + SpecB2 pipeline for a single model."""
    import time
    
    if options is None:
        options = PIPELINE_OPTIONS
    if specA_prompts_data is None:
        specA_prompts_data = (prompts, meta, group_ids, variant_ids, expected_tokens)
    if continuation_prompts_data is None:
        continuation_prompts_data = globals().get('specb_data', None)
    
    prompts_list, meta_list, group_ids_list, variant_ids_list, expected_tokens_list = specA_prompts_data
    
    result = {"model_key": model_key, "model_id": None, "status": "started",
              "specA_results": {}, "specB2_results": {}, "timings": {}, "errors": []}
    
    pipeline_start = time.time()
    
    def _phase_time():
        """Elapsed since pipeline start, formatted."""
        elapsed = time.time() - pipeline_start
        h, rem = divmod(int(elapsed), 3600)
        m, s = divmod(rem, 60)
        return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"
    
    bundle = None
    layer_to_H = None
    extraction = None
    P_bases = None
    
    print("\n" + "=" * 70)
    print(f"PIPELINE: {model_key}")
    print("=" * 70)
    
    # Debug: Show options
    print("\nPipeline options:")
    for k, v in options.items():
        print(f"  {k}: {v}")
    
    try:
        # ============================================================
        # Phase 1: Setup
        # ============================================================
        print(f"\n[{_phase_time()}] [Phase 1/9] Model Setup")
        print("-" * 40)
        setup_model_run(model_key)
        result["model_id"] = MODEL_ID
        
        # ============================================================
        # Phase 2: Load Model
        # ============================================================
        print(f"\n[{_phase_time()}] [Phase 2/9] Model Loading")
        print("-" * 40)
        phase_start = time.time()
        bundle = load_model_for_pipeline(MODEL_ID, USE_4BIT_QUANTIZATION)
        print(f"  ✓ Loaded: {bundle.n_layers} layers, {bundle.hidden_dim} dim")
        result["timings"]["model_loading"] = time.time() - phase_start
        
        # ============================================================
        # Phase 3: SpecA Extraction
        # ============================================================
        # Extraction runs automatically when geometry analysis or Part G needs it
        needs_extraction = (
            options.get("run_specA_analysis", True)
            or options.get("run_specA_part_g", True)
            or options.get("run_geometry_comparison", True)
        )
        print(f"\n[{_phase_time()}] [Phase 3/9] Extraction (needed: {needs_extraction})")
        print("-" * 40)
        
        if needs_extraction:
            phase_start = time.time()
            
            cache_exists = CACHE_NPZ.exists() and CACHE_META.exists()
            use_cache = options.get("use_extraction_cache", True) and cache_exists
            print(f"  Cache exists: {cache_exists}, Use cache: {use_cache}")
            
            if use_cache:
                print(f"  Loading from cache: {CACHE_NPZ}")
                try:
                    layer_to_H, extraction, P_bases, pred_ids, unembed_np = load_extraction_from_cache(
                        CACHE_NPZ, CACHE_META, bundle, meta_list, expected_tokens_list)
                    print(f"  ✓ Loaded {len(layer_to_H)} layers from cache")
                except Exception as e:
                    print(f"  ⚠️ Cache load failed: {e}")
                    print(f"  Falling back to fresh extraction...")
                    use_cache = False
            
            if not use_cache:
                print(f"  Extracting {len(prompts_list)} prompts...")
                layer_to_H, _, pred_ids, _ = extract_hidden_states(
                    bundle=bundle, prompts=prompts_list,
                    move_to_cpu=True, show_progress=True)
                print(f"  ✓ Extracted {len(layer_to_H)} layers")
                
                print("  Building extraction object...")
                extraction = build_extraction_object(
                    layer_to_H, meta_list, MODEL_ID, group_ids_list, variant_ids_list)
                
                print("  Building P bases...")
                unembed_np = _safe_unembed_to_numpy(bundle.model)
                P_bases, _ = build_unified_P_bases(pred_ids, unembed_np, extraction)
                print(f"  ✓ P bases built for {len(P_bases)} layers")
                
                if options.get("save_extraction_cache", True):
                    print(f"  Saving to cache: {CACHE_NPZ}")
                    save_extraction_to_cache(
                        layer_to_H, meta_list, MODEL_ID, bundle,
                        CACHE_NPZ, CACHE_META, group_ids_list, variant_ids_list)
                    print("  ✓ Cache saved")
            
            result["timings"]["extraction"] = time.time() - phase_start
        else:
            print("  SKIPPED: no downstream phases need it")
        
        # ============================================================
        # Phase 4: SpecA Analysis (Parts A-H)
        # ============================================================
        run_analysis = options.get("run_specA_analysis", True)
        print(f"\n[{_phase_time()}] [Phase 4/9] SpecA Analysis (enabled: {run_analysis}, has extraction: {extraction is not None})")
        print("-" * 40)
        
        if run_analysis and extraction is not None:
            phase_start = time.time()
            print("  Running consolidated analysis...")
            
            # Get unembedding matrix
            # Meta-safe unembedding extraction
            unembed_np = _safe_unembed_to_numpy(bundle.model)
            
            specA_analysis = run_consolidated_specA_analysis(
                extraction=extraction,
                P_bases=P_bases,
                unembed_np=unembed_np,
                run_parts=RUN_PARTS,
                output_files=OUTPUT_FILES,
                model_output_dir=MODEL_OUTPUT_DIR,
                k_max=K_MAX,
                s_small_r=S_SMALL_R,
                verbose=True,
                resolution_geometry_layers=RESOLUTION_GEOMETRY_LAYERS,
            )
            result["specA_results"] = {"analysis": specA_analysis}
            result["timings"]["analysis"] = time.time() - phase_start
        else:
            reasons = []
            if not run_analysis:
                reasons.append("disabled in options")
            if extraction is None:
                reasons.append("no extraction data")
            print(f"  SKIPPED: {', '.join(reasons)}")
        
        # ============================================================
        # Phase 4M: Part M baseline spirality (SpecA)
        # ============================================================
        part_m_baseline_specA = None
        run_part_m = options.get("run_geometry_part_m", False)
        run_part_m_intervention = options.get("run_geometry_part_m_intervention", False)
        if run_part_m and layer_to_H is not None and pds_spirality is not None:
            try:
                part_m_start = time.time()
                print(f"\n[{_phase_time()}] Part M baseline spirality (SpecA)...")
                part_m_baseline_specA = run_part_M_baseline_for_pipeline(
                    layer_to_H={int(k): v for k, v in layer_to_H.items()},
                    n_prompts=len(prompts_list),
                    model_id=bundle.model_id,
                    output_file=OUTPUT_FILES.get("part_m_baseline",
                        MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_baseline_spirality.json"),
                    n_pc_pairs=SPIRALITY_N_PC_PAIRS,
                    verbose=True,
                )
                result["specA_results"]["part_m_baseline"] = {"status": "completed"}
                result["timings"]["part_m_baseline_specA"] = time.time() - part_m_start
            except Exception as e:
                print(f"  ⚠️ Part M baseline (SpecA) failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Part M baseline SpecA: {e}")

        # ============================================================
        # Phase 5A: SpecA Part G (PDSF Interventions)
        # ============================================================
        run_part_g = options.get("run_specA_part_g", True)
        print(f"\n[{_phase_time()}] [Phase 5A/9] SpecA Part G (enabled: {run_part_g}, has layer_to_H: {layer_to_H is not None})")
        print("-" * 40)
        
        if run_part_g and layer_to_H is not None:
            phase_start = time.time()
            print(f"  Scramble layers: {SCRAMBLE_LAYERS}")
            print(f"  N scrambles: {N_SCRAMBLES}")
            
            try:
                speca_domain_ids = [SPECA_GROUP_TO_DOMAIN.get(gid, "unknown") for gid in group_ids_list]
                part_g_results = run_part_G_for_pipeline(
                    bundle=bundle,
                    layer_to_H=layer_to_H,
                    prompts=prompts_list,
                    group_ids=group_ids_list,
                    expected_tokens=pred_ids,  # Use model predictions, not experimenter tokens
                    regime_ids=speca_domain_ids,
                    output_file=OUTPUT_FILES.get("part_g_regime_transplant", OUTPUT_FILES.get("part_g_v11", OUTPUT_FILES["part_g"])),
                    scramble_layers=SCRAMBLE_LAYERS,
                    n_scrambles=N_SCRAMBLES,
                    s_rank=S_RANK,
                    seed=SCRAMBLE_SEED,
                    n_prompts=N_SCRAMBLE_PROMPTS,
                    verbose=True,
                    compute_kl=COMPUTE_KL,
                    kl_topk=KL_TOPK,
                    compute_spirality=run_part_m_intervention and pds_spirality is not None,
                    spirality_n_pc_pairs=SPIRALITY_N_PC_PAIRS,
                )
                result["specA_results"]["part_g"] = {"status": "completed"}
                
                # ── Part L: Complexity Depth Profiles (runs with Part G data) ──
                # Uses the same layer_to_H and P_bases, no extra forward passes.
                run_part_l = options.get("run_geometry_part_l", RUN_PARTS.get("part_l", False))
                if run_part_l and extraction is not None and P_bases is not None:
                    try:
                        part_l_start = time.time()
                        run_part_L_for_pipeline(
                            layer_to_H={int(k): v for k, v in layer_to_H.items()},
                            group_ids=group_ids_list,
                            P_bases_by_layer=P_bases,
                            unembed_np=unembed_np,
                            output_file=OUTPUT_FILES.get("part_l",
                                MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_l_complexity_depth_profile.json"),
                            k_max=K_MAX if 'K_MAX' in dir() else 64,
                            s_small_r=S_RANK,
                            model_id=bundle.model_id,
                            prompt_set_label="SpecA",
                            verbose=True,
                        )
                        result["specA_results"]["part_l"] = {"status": "completed"}
                        result["timings"]["part_l_specA"] = time.time() - part_l_start
                    except Exception as e:
                        print(f"  ⚠️ Part L (SpecA) failed: {e}")
                        traceback.print_exc()
                        result["errors"].append(f"Part L SpecA: {e}")

                # ── Part M Phase B: Extract intervention spirality from Part G ──
                if run_part_m_intervention and part_g_results is not None and pds_spirality is not None:
                    try:
                        extract_part_m_from_part_g(
                            part_g_results, bundle.model_id,
                            OUTPUT_FILES.get("part_m",
                                MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_spirality.json"),
                            verbose=True,
                        )
                        result["specA_results"]["part_m_intervention"] = {"status": "completed"}
                    except Exception as e:
                        print(f"  ⚠️ Part M intervention (SpecA) failed: {e}")
                        traceback.print_exc()
                        result["errors"].append(f"Part M intervention SpecA: {e}")

                # ── Part M Phase C: Correlation with Part F rotation data ──
                if run_part_m and part_m_baseline_specA is not None and pds_spirality is not None:
                    per_prompt_file = OUTPUT_FILES.get("per_prompt")
                    if per_prompt_file and Path(str(per_prompt_file)).exists():
                        try:
                            run_part_M_correlation_for_pipeline(
                                part_m_baseline_specA,
                                per_prompt_file,
                                model_id=bundle.model_id,
                                output_file=OUTPUT_FILES.get("part_m_correlation",
                                    MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-part_m_correlation.json"),
                                verbose=True,
                            )
                            result["specA_results"]["part_m_correlation"] = {"status": "completed"}
                        except Exception as e:
                            print(f"  ⚠️ Part M correlation (SpecA) failed: {e}")
                            traceback.print_exc()
                            result["errors"].append(f"Part M correlation SpecA: {e}")
                    else:
                        print("  Part M correlation skipped: Part F per-prompt file not found")

            except Exception as e:
                print(f"  ⚠️ Part G failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Part G: {e}")
            
            result["timings"]["part_g"] = time.time() - phase_start
        else:
            reasons = []
            if not run_part_g:
                reasons.append("disabled in options")
            if layer_to_H is None:
                reasons.append("no layer_to_H data")
            print(f"  SKIPPED: {', '.join(reasons)}")
        
        
        # ============================================================
        # Phase 5B: Geometry Analysis on SpecB Prompts
        # ============================================================
        run_geometry_on_specB = options.get("run_geometry_on_continuation_prompts", False) or options.get("run_geometry_on_specB", False)
        print(f"\n[{_phase_time()}] [Phase 5B/9] SpecB Geometry (enabled: {run_geometry_on_specB})")
        print("-" * 40)
        
        specB_extraction = None
        specB_layer_to_H = None
        
        if run_geometry_on_specB and continuation_prompts_data and bundle is not None:
            phase_start = time.time()
            
            try:
                # Parse SpecB prompts into same format as SpecA
                specB_prompts_list = []
                specB_group_ids_list = []
                specB_variant_ids_list = []
                specB_expected_tokens_list = []
                specB_meta_list = []
                
                # Support both old (nested groups) and new (flat prompts) formats
                if "prompts" in continuation_prompts_data:
                    # New format: flat list with regime info
                    for p in continuation_prompts_data["prompts"]:
                        specB_prompts_list.append(p["prompt"])
                        specB_group_ids_list.append(p["group_id"])
                        specB_variant_ids_list.append(p["variant_id"])
                        specB_expected_tokens_list.append(None)
                        specB_meta_list.append({
                            "group_id": p["group_id"],
                            "variant_id": p["variant_id"],
                            "prompt": p["prompt"],
                            "category": p.get("category", ""),
                            "regime": p.get("regime", ""),
                            "expected_token": None,
                        })
                else:
                    # Old format: nested groups/variants
                    for group in continuation_prompts_data.get("groups", []):
                        gid = group.get("group_id", "")
                        expected = None
                        for var in group.get("variants", []):
                            specB_prompts_list.append(var.get("prompt", ""))
                            specB_group_ids_list.append(gid)
                            specB_variant_ids_list.append(var.get("variant_id", ""))
                            specB_expected_tokens_list.append(expected)
                            specB_meta_list.append({
                                "group_id": gid,
                                "variant_id": var.get("variant_id", ""),
                                "prompt": var.get("prompt", ""),
                                "expected_token": expected,
                            })
                
                print(f"  SpecB prompts: {len(specB_prompts_list)}")
                print(f"  Groups: {len(set(specB_group_ids_list))}")
                
                # Extract hidden states for SpecB prompts
                print("  Extracting hidden states for SpecB prompts...")
                specB_layer_to_H, _, specB_pred_ids, _ = extract_hidden_states(
                    bundle=bundle, prompts=specB_prompts_list,
                    move_to_cpu=True, show_progress=True)
                
                print(f"  ✓ Extracted {len(specB_layer_to_H)} layers")
                
                # Build extraction object for SpecB
                specB_extraction = build_extraction_object(
                    specB_layer_to_H, specB_meta_list, bundle.model_id,
                    specB_group_ids_list, specB_variant_ids_list
                )
                
                # Build P bases for SpecB (using expected tokens if available)
                print("  Building P bases for SpecB...")
                # Meta-safe unembedding extraction
                unembed_np = _safe_unembed_to_numpy(bundle.model)
                
                # Build per-prompt P bases from actual model predictions
                # Each prompt gets P = span{unembed[argmax_token]}
                specB_P_bases, _ = build_unified_P_bases(specB_pred_ids, unembed_np, specB_extraction)
                
                # P bases are per-prompt only (no group-level P needed with unified approach)
                
                n_unique = len(set(specB_pred_ids))
                print(f"  ✓ Per-prompt P bases built ({n_unique} unique predicted tokens across {len(specB_pred_ids)} prompts)")
                
                # Run geometry analysis on SpecB
                print("  Running geometry analysis on SpecB prompts...")
                
                # SpecB has no factorial design (no A/B/C/D factors),
                # so skip Part C. All other parts work on generic group structure.
                specB_run_parts = {k: v for k, v in RUN_PARTS.items()}
                specB_run_parts["part_c"] = False  # No factorial factors in SpecB

                specB_analysis_results = run_consolidated_specA_analysis(
                    extraction=specB_extraction,
                    P_bases=specB_P_bases,
                    unembed_np=unembed_np,
                    run_parts=specB_run_parts,
                    output_files=OUTPUT_FILES_SPECB,
                    model_output_dir=MODEL_OUTPUT_DIR,
                    k_max=K_MAX,
                    s_small_r=S_SMALL_R,
                    verbose=True,
                    resolution_geometry_layers=RESOLUTION_GEOMETRY_LAYERS,
                )
                
                result["specB_geometry_results"] = specB_analysis_results
                print("  ✓ Geometry analysis on SpecB complete")
                
                # Compute spectrum layers for SpecB (shared by energy spectrum + resolution geometry)
                specB_layers_sorted = sorted(specB_extraction.layers.keys())
                n_specB_layers = len(specB_layers_sorted)
                specB_spectrum_layers = sorted(set([
                    specB_layers_sorted[0],
                    specB_layers_sorted[n_specB_layers // 4],
                    specB_layers_sorted[n_specB_layers // 2],
                    specB_layers_sorted[3 * n_specB_layers // 4],
                    specB_layers_sorted[-1],
                ]))
                
                # --- Energy Spectrum on SpecB ---
                if options.get("run_energy_spectrum", True):
                 try:
                    start = time.time()
                    specB_energy_spectrum = compute_energy_spectrum(
                        specB_extraction, specB_P_bases, layers=specB_spectrum_layers, n_dims=100
                    )
                    specB_es_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-energy_spectrum.json"
                    save_json(specB_energy_spectrum, specB_es_file)
                    print(f"  ✓ SpecB energy spectrum saved")
                 except Exception as e:
                    print(f"  ⚠️ SpecB energy spectrum failed: {e}")
                    result["errors"].append(f"SpecB energy_spectrum: {e}")
                
                # --- Part J: Resolution Geometry on SpecB ---
                if options.get("run_resolution_geometry", True):
                 try:
                    start = time.time()
                    # PR-adaptive P; layer selection from config
                    if RESOLUTION_GEOMETRY_LAYERS == "sparse":
                        _ly_sorted = sorted(specB_extraction.layers.keys())
                        _n_ly = len(_ly_sorted)
                        _rg_ly = sorted(set([
                            _ly_sorted[max(0, _n_ly // 4 - 1)],
                            _ly_sorted[max(0, _n_ly // 2 - 1)],
                            _ly_sorted[max(0, 3 * _n_ly // 4 - 1)],
                            _ly_sorted[-1],
                        ]))
                    else:
                        _rg_ly = None
                    specB_resolution_geo = compute_resolution_geometry(
                        specB_extraction, specB_P_bases, unembed_np,
                        layers=_rg_ly,
                        k_max=K_MAX, s_small_r=S_SMALL_R, k_neighbors=8,
                    )
                    specB_rg_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-resolution_geometry_v11.json"
                    save_json(specB_resolution_geo, specB_rg_file)
                    _sb_layers = sorted(specB_resolution_geo.get("by_layer", {}).keys(), key=int)
                    if _sb_layers:
                        _sb_last = specB_resolution_geo["by_layer"][_sb_layers[-1]]
                        _sb_sum = _sb_last.get("summary", {})
                        _sb_pca = _sb_last.get("pca_control", {}).get("contiguous_bins", {})
                        _sb_shuf = _sb_last.get("shuffled_control", {}).get("bins", {})
                        print(f"  ✓ SpecB resolution geometry ({len(_sb_layers)} layers):")
                        print(f"      PDSF:     D={_sb_sum.get('D_curvature_ratio','?')} P={_sb_sum.get('P_curvature_ratio','?')} S={_sb_sum.get('S_curvature_ratio','?')} F={_sb_sum.get('residual_curvature_ratio','?')} mono={_sb_sum.get('ordering_DPSF_monotonic','?')}")
                        print(f"      PCA ctrl: D={_sb_pca.get('bin_D',{}).get('curvature_ratio','?')} P={_sb_pca.get('bin_P',{}).get('curvature_ratio','?')} S={_sb_pca.get('bin_S',{}).get('curvature_ratio','?')} F={_sb_pca.get('bin_F',{}).get('curvature_ratio','?')} mono={_sb_last.get('pca_control',{}).get('contiguous_ordering_monotonic','?')}")
                        print(f"      Shuffled: D={_sb_shuf.get('bin_D',{}).get('curvature_ratio_mean','?')} P={_sb_shuf.get('bin_P',{}).get('curvature_ratio_mean','?')} S={_sb_shuf.get('bin_S',{}).get('curvature_ratio_mean','?')} F={_sb_shuf.get('bin_F',{}).get('curvature_ratio_mean','?')}")
                        print(f"      P-PCA: angle={_sb_sum.get('P_vs_PCA_mean_angle_deg','?')}° var={_sb_sum.get('P_variance_fraction_of_H','?')} rank={_sb_last.get('basis_ranks',{}).get('P','?')}")
                    else:
                        print(f"  ✓ SpecB resolution geometry saved")
                 except Exception as e:
                    print(f"  ⚠️ SpecB resolution geometry failed: {e}")
                    result["errors"].append(f"SpecB resolution_geometry: {e}")
                
                # Part G on SpecB prompts
                if options.get("run_geometry_part_g_specB", False) and specB_layer_to_H is not None:
                    print("\n  Running Part G on SpecB prompts...")
                    try:
                        specB_domain_ids = [SPECB_GROUP_TO_DOMAIN.get(gid, "unknown") for gid in specB_group_ids_list]
                        specB_part_g = run_part_G_for_pipeline(
                            bundle=bundle,
                            layer_to_H=specB_layer_to_H,
                            prompts=specB_prompts_list,
                            group_ids=specB_group_ids_list,
                            expected_tokens=specB_pred_ids,  # Pass pred_ids (int) directly
                            regime_ids=specB_domain_ids,
                            output_file=OUTPUT_FILES_SPECB.get("part_g_regime_transplant", OUTPUT_FILES_SPECB["part_g"]),
                            scramble_layers=SCRAMBLE_LAYERS,
                            n_scrambles=N_SCRAMBLES,
                            s_rank=S_RANK,
                            seed=SCRAMBLE_SEED,
                            n_prompts=SPECB_N_SCRAMBLE_PROMPTS,
                            verbose=True,
                            compute_kl=COMPUTE_KL,
                            kl_topk=KL_TOPK,
                            compute_spirality=options.get("run_geometry_part_m_intervention_specB", False) and pds_spirality is not None,
                            spirality_n_pc_pairs=SPIRALITY_N_PC_PAIRS,
                        )
                        print("  ✓ Part G on SpecB complete")

                        # ── Part M Phase B on SpecB ──
                        if options.get("run_geometry_part_m_intervention_specB", False) and specB_part_g is not None and pds_spirality is not None:
                            try:
                                extract_part_m_from_part_g(
                                    specB_part_g, bundle.model_id,
                                    OUTPUT_FILES_SPECB.get("part_m",
                                        MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_m_spirality.json"),
                                    verbose=True,
                                )
                            except Exception as e:
                                print(f"  ⚠️ Part M intervention (SpecB) failed: {e}")

                        # ── Part L on SpecB ──
                        run_part_l = options.get("run_geometry_part_l", RUN_PARTS.get("part_l", False))
                        if run_part_l and specB_extraction is not None and specB_P_bases is not None:
                            try:
                                run_part_L_for_pipeline(
                                    layer_to_H={int(k): v for k, v in specB_layer_to_H.items()},
                                    group_ids=specB_group_ids_list,
                                    P_bases_by_layer=specB_P_bases,
                                    unembed_np=unembed_np,
                                    output_file=OUTPUT_FILES_SPECB.get("part_l",
                                        MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_l_complexity_depth_profile.json"),
                                    k_max=K_MAX if 'K_MAX' in dir() else 64,
                                    s_small_r=S_RANK,
                                    model_id=bundle.model_id,
                                    prompt_set_label="SpecB",
                                    verbose=True,
                                )
                            except Exception as e:
                                print(f"  ⚠️ Part L (SpecB) failed: {e}")
                    except Exception as e:
                        print(f"  ⚠️ Part G on SpecB failed: {e}")
                        traceback.print_exc()
                        result["errors"].append(f"Part G on SpecB: {e}")
                
            except Exception as e:
                print(f"  ⚠️ Geometry on SpecB failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Geometry on SpecB: {e}")
            
            result["timings"]["geometry_on_specB"] = time.time() - phase_start
        else:
            reasons = []
            if not run_geometry_on_specB:
                reasons.append("disabled in options")
            if not continuation_prompts_data:
                reasons.append("no SpecB prompts loaded")
            if bundle is None:
                reasons.append("no model bundle")
            print(f"  SKIPPED: {', '.join(reasons)}")
        
        # ============================================================
        # Phase 5C: Diverse Geometry (group-level + regime-level + Part G)
        # ============================================================
        run_diverse_geometry = options.get("run_geometry_on_diverse_prompts", False)
        print(f"\n[{_phase_time()}] [Phase 5C/9] Diverse Geometry (enabled: {run_diverse_geometry})")
        print("-" * 40)
        
        # These variables are shared with Phase 8 (Diverse Continuation)
        diverse_prompts_list = []
        diverse_group_ids_list = []
        diverse_variant_ids_list = []
        diverse_regime_ids_list = []
        diverse_meta_list = []
        diverse_prompts_parsed = False
        
        if run_diverse_geometry and diverse_prompts_data and bundle is not None:
            phase_start = time.time()
            
            try:
                # Parse diverse prompts (shared with Phase 8)
                if "prompts" in diverse_prompts_data:
                    for p in diverse_prompts_data["prompts"]:
                        diverse_prompts_list.append(p["prompt"])
                        diverse_group_ids_list.append(p["group_id"])
                        diverse_variant_ids_list.append(p["variant_id"])
                        diverse_regime_ids_list.append(p.get("regime", ""))
                        diverse_meta_list.append({
                            "group_id": p["group_id"],
                            "variant_id": p["variant_id"],
                            "prompt": p["prompt"],
                            "category": p.get("category", ""),
                            "regime": p.get("regime", ""),
                            "expected_token": None,
                        })
                else:
                    for group in diverse_prompts_data.get("groups", []):
                        gid = group.get("group_id", "")
                        for var in group.get("variants", []):
                            diverse_prompts_list.append(var.get("prompt", ""))
                            diverse_group_ids_list.append(gid)
                            diverse_variant_ids_list.append(var.get("variant_id", ""))
                            diverse_regime_ids_list.append(group.get("regime", ""))
                            diverse_meta_list.append({
                                "group_id": gid,
                                "variant_id": var.get("variant_id", ""),
                                "prompt": var.get("prompt", ""),
                                "category": group.get("category", ""),
                                "regime": group.get("regime", ""),
                                "expected_token": None,
                            })
                diverse_prompts_parsed = True
                
                n_regimes = len(set(diverse_regime_ids_list))
                print(f"  Diverse prompts: {len(diverse_prompts_list)} ({n_regimes} regimes)")
                
                # Extraction
                print("\n  Extracting hidden states for diverse prompts...")
                diverse_layer_to_H, _, diverse_pred_ids, _ = extract_hidden_states(
                    bundle=bundle, prompts=diverse_prompts_list,
                    move_to_cpu=True, show_progress=True)
                print(f"  ✓ Extracted {len(diverse_layer_to_H)} layers")
                
                # Build P bases from model predictions
                # Meta-safe unembedding extraction
                unembed_np = _safe_unembed_to_numpy(bundle.model)
                
                diverse_extraction = build_extraction_object(
                    diverse_layer_to_H, diverse_meta_list, bundle.model_id,
                    diverse_group_ids_list, diverse_variant_ids_list
                )
                diverse_P_bases, _ = build_unified_P_bases(diverse_pred_ids, unembed_np, diverse_extraction)
                
                n_unique_tokens = len(set(diverse_pred_ids))
                print(f"  ✓ P bases built ({n_unique_tokens} unique tokens)")
                
                # Skip Part C (no factorial design)
                diverse_run_parts = {k: v for k, v in RUN_PARTS.items()}
                diverse_run_parts["part_c"] = False
                
                # --- Group-level geometry (21 groups × 4 variants) ---
                print("\n  Running geometry at GROUP level (21 groups)...")
                diverse_group_results = run_consolidated_specA_analysis(
                    extraction=diverse_extraction,
                    P_bases=diverse_P_bases,
                    unembed_np=unembed_np,
                    run_parts=diverse_run_parts,
                    output_files=OUTPUT_FILES_DIVERSE_GROUP,
                    model_output_dir=MODEL_OUTPUT_DIR,
                    k_max=K_MAX,
                    s_small_r=DIVERSE_S_RANK,
                    verbose=True,
                    resolution_geometry_layers=RESOLUTION_GEOMETRY_LAYERS,
                )
                result["diverse_geometry_group"] = diverse_group_results
                print("  ✓ Group-level geometry complete")
                
                # Compute spectrum layers for Diverse (shared by energy spectrum + resolution geometry)
                diverse_layers_sorted = sorted(diverse_extraction.layers.keys())
                n_diverse_layers = len(diverse_layers_sorted)
                diverse_spectrum_layers = sorted(set([
                    diverse_layers_sorted[0],
                    diverse_layers_sorted[n_diverse_layers // 4],
                    diverse_layers_sorted[n_diverse_layers // 2],
                    diverse_layers_sorted[3 * n_diverse_layers // 4],
                    diverse_layers_sorted[-1],
                ]))
                
                # --- Energy Spectrum on Diverse (group-level) ---
                if options.get("run_energy_spectrum", True):
                 try:
                    start = time.time()
                    diverse_energy_spectrum = compute_energy_spectrum(
                        diverse_extraction, diverse_P_bases, layers=diverse_spectrum_layers, n_dims=100
                    )
                    diverse_es_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-energy_spectrum.json"
                    save_json(diverse_energy_spectrum, diverse_es_file)
                    print(f"  ✓ Diverse (group) energy spectrum saved")
                 except Exception as e:
                    print(f"  ⚠️ Diverse energy spectrum failed: {e}")
                    result["errors"].append(f"Diverse energy_spectrum: {e}")
                
                # --- Part J: Resolution Geometry on Diverse (naive/unstratified) ---
                if options.get("run_resolution_geometry", True):
                 try:
                    start = time.time()
                    # PR-adaptive P; layer selection from config
                    if RESOLUTION_GEOMETRY_LAYERS == "sparse":
                        _ly_sorted = sorted(diverse_extraction.layers.keys())
                        _n_ly = len(_ly_sorted)
                        _rg_ly = sorted(set([
                            _ly_sorted[max(0, _n_ly // 4 - 1)],
                            _ly_sorted[max(0, _n_ly // 2 - 1)],
                            _ly_sorted[max(0, 3 * _n_ly // 4 - 1)],
                            _ly_sorted[-1],
                        ]))
                    else:
                        _rg_ly = None
                    diverse_resolution_geo = compute_resolution_geometry(
                        diverse_extraction, diverse_P_bases, unembed_np,
                        layers=_rg_ly,
                        k_max=K_MAX, s_small_r=DIVERSE_S_RANK, k_neighbors=8,
                    )
                    diverse_rg_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11.json"
                    save_json(diverse_resolution_geo, diverse_rg_file)
                    _dv_layers = sorted(diverse_resolution_geo.get("by_layer", {}).keys(), key=int)
                    if _dv_layers:
                        _dv_last = diverse_resolution_geo["by_layer"][_dv_layers[-1]]
                        _dv_sum = _dv_last.get("summary", {})
                        _dv_pca = _dv_last.get("pca_control", {}).get("contiguous_bins", {})
                        _dv_shuf = _dv_last.get("shuffled_control", {}).get("bins", {})
                        print(f"  ✓ Diverse resolution geometry ({len(_dv_layers)} layers):")
                        print(f"      PDSF:     D={_dv_sum.get('D_curvature_ratio','?')} P={_dv_sum.get('P_curvature_ratio','?')} S={_dv_sum.get('S_curvature_ratio','?')} F={_dv_sum.get('residual_curvature_ratio','?')} mono={_dv_sum.get('ordering_DPSF_monotonic','?')}")
                        print(f"      PCA ctrl: D={_dv_pca.get('bin_D',{}).get('curvature_ratio','?')} P={_dv_pca.get('bin_P',{}).get('curvature_ratio','?')} S={_dv_pca.get('bin_S',{}).get('curvature_ratio','?')} F={_dv_pca.get('bin_F',{}).get('curvature_ratio','?')} mono={_dv_last.get('pca_control',{}).get('contiguous_ordering_monotonic','?')}")
                        print(f"      Shuffled: D={_dv_shuf.get('bin_D',{}).get('curvature_ratio_mean','?')} P={_dv_shuf.get('bin_P',{}).get('curvature_ratio_mean','?')} S={_dv_shuf.get('bin_S',{}).get('curvature_ratio_mean','?')} F={_dv_shuf.get('bin_F',{}).get('curvature_ratio_mean','?')}")
                        print(f"      P-PCA: angle={_dv_sum.get('P_vs_PCA_mean_angle_deg','?')}° var={_dv_sum.get('P_variance_fraction_of_H','?')} rank={_dv_last.get('basis_ranks',{}).get('P','?')}")
                    else:
                        print(f"  ✓ Diverse (group) resolution geometry saved")
                 except Exception as e:
                    print(f"  ⚠️ Diverse resolution geometry failed: {e}")
                    result["errors"].append(f"Diverse resolution_geometry: {e}")
                
                # --- Stratified Resolution Geometry on Diverse (per-regime) ---
                if options.get("run_resolution_geometry", True):
                 try:
                    start = time.time()
                    # Full trajectory
                    if RESOLUTION_GEOMETRY_LAYERS == "sparse":
                        _ly_sorted = sorted(diverse_extraction.layers.keys())
                        _n_ly = len(_ly_sorted)
                        _rg_ly = sorted(set([
                            _ly_sorted[max(0, _n_ly // 4 - 1)],
                            _ly_sorted[max(0, _n_ly // 2 - 1)],
                            _ly_sorted[max(0, 3 * _n_ly // 4 - 1)],
                            _ly_sorted[-1],
                        ]))
                    else:
                        _rg_ly = None
                    diverse_stratified_geo = compute_resolution_geometry_stratified(
                        diverse_extraction, diverse_P_bases, unembed_np,
                        regime_labels=diverse_regime_ids_list,
                        layers=_rg_ly,
                        k_max=K_MAX, s_small_r=DIVERSE_S_RANK, k_neighbors=5,
                    )
                    diverse_strat_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11_stratified.json"
                    save_json(diverse_stratified_geo, diverse_strat_file)
                    print(f"  ✓ Diverse stratified resolution geometry saved ({len(set(diverse_regime_ids_list))} regimes)")
                 except Exception as e:
                    print(f"  ⚠️ Diverse stratified resolution geometry failed: {e}")
                    result["errors"].append(f"Diverse stratified_resolution_geometry: {e}")
                
                # --- v11.1: Within-regime curvature using global bases ---
                if options.get("run_resolution_geometry", True):
                 try:
                    start = time.time()
                    # Full trajectory, PR-adaptive P, global bases
                    if RESOLUTION_GEOMETRY_LAYERS == "sparse":
                        _ly_sorted_wr = sorted(diverse_extraction.layers.keys())
                        _n_ly_wr = len(_ly_sorted_wr)
                        _rg_ly_wr = sorted(set([
                            _ly_sorted_wr[max(0, _n_ly_wr // 4 - 1)],
                            _ly_sorted_wr[max(0, _n_ly_wr // 2 - 1)],
                            _ly_sorted_wr[max(0, 3 * _n_ly_wr // 4 - 1)],
                            _ly_sorted_wr[-1],
                        ]))
                    else:
                        _rg_ly_wr = None
                    diverse_within_regime_geo = compute_resolution_geometry_within_regime(
                        diverse_extraction, diverse_P_bases, unembed_np,
                        regime_labels=diverse_regime_ids_list,
                        layers=_rg_ly_wr,
                        k_max=K_MAX, s_small_r=DIVERSE_S_RANK, k_neighbors=5,
                    )
                    diverse_within_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11_within_regime.json"
                    diverse_within_regime_geo["_metadata"] = {
                        "prompt_set": "Diverse",
                        "method": "global_basis_within_regime_curvature",
                        "n_prompts": len(diverse_regime_ids_list),
                        "n_regimes": len(set(diverse_regime_ids_list)),
                        "k_max": K_MAX,
                        "s_small_r": DIVERSE_S_RANK,
                        "k_neighbors": 5,
                        "model_id": MODEL_ID,
                        "pipeline_version": "v11.1",
                    }
                    save_json(diverse_within_regime_geo, diverse_within_file)
                    n_regimes = len(set(diverse_regime_ids_list))
                    pooled_last = diverse_within_regime_geo.get("pooled", {}).get("by_layer", {})
                    if pooled_last:
                        last_key = sorted(pooled_last.keys(), key=int)[-1]
                        ps = pooled_last[last_key].get("summary", {})
                        d_c = ps.get("D_curvature_ratio", "?")
                        p_c = ps.get("P_curvature_ratio", "?")
                        s_c = ps.get("S_curvature_ratio", "?")
                        f_c = ps.get("residual_curvature_ratio", "?")
                        mono = ps.get("ordering_DPSF_monotonic", "?")
                        print(f"  ✓ Within-regime curvature ({n_regimes} regimes): D={d_c} P={p_c} S={s_c} F={f_c} DPSF={mono} ({time.time()-start:.1f}s)")
                 except Exception as e:
                    print(f"  ⚠️ Diverse within-regime resolution geometry failed: {e}")
                    result["errors"].append(f"Diverse within_regime_resolution_geometry: {e}")
                
                # Part G on diverse prompts (group-level)
                if options.get("run_geometry_part_g_diverse", True) and diverse_layer_to_H is not None:
                    print("\n  Running Part G on diverse prompts (group-level)...")
                    try:
                        diverse_part_g = run_part_G_for_pipeline(
                            bundle=bundle,
                            layer_to_H=diverse_layer_to_H,
                            prompts=diverse_prompts_list,
                            group_ids=diverse_group_ids_list,
                            expected_tokens=diverse_pred_ids,
                            regime_ids=diverse_regime_ids_list,
                            output_file=OUTPUT_FILES_DIVERSE_GROUP.get("part_g_regime_transplant", OUTPUT_FILES_DIVERSE_GROUP.get("part_g_v11", OUTPUT_FILES_DIVERSE_GROUP["part_g"])),
                            scramble_layers=SCRAMBLE_LAYERS,
                            n_scrambles=N_SCRAMBLES,
                            s_rank=S_RANK,
                            seed=SCRAMBLE_SEED,
                            n_prompts=DIVERSE_N_SCRAMBLE_PROMPTS,
                            verbose=True,
                            compute_kl=COMPUTE_KL,
                            kl_topk=KL_TOPK,
                            compute_spirality=options.get("run_geometry_part_m_intervention_diverse", False) and pds_spirality is not None,
                            spirality_n_pc_pairs=SPIRALITY_N_PC_PAIRS,
                        )
                        print("  ✓ Part G on diverse prompts complete")

                        # ── Part M Phase B on Diverse (group-level) ──
                        if options.get("run_geometry_part_m_intervention_diverse", False) and diverse_part_g is not None and pds_spirality is not None:
                            try:
                                extract_part_m_from_part_g(
                                    diverse_part_g, bundle.model_id,
                                    OUTPUT_FILES_DIVERSE_GROUP.get("part_m",
                                        MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_m_spirality.json"),
                                    verbose=True,
                                )
                            except Exception as e:
                                print(f"  ⚠️ Part M intervention (Diverse group) failed: {e}")

                        # ── Part L on Diverse (group-level) ──
                        run_part_l = options.get("run_geometry_part_l", RUN_PARTS.get("part_l", False))
                        if run_part_l and diverse_extraction is not None and diverse_P_bases is not None:
                            try:
                                run_part_L_for_pipeline(
                                    layer_to_H={int(k): v for k, v in diverse_layer_to_H.items()},
                                    group_ids=diverse_group_ids_list,
                                    P_bases_by_layer=diverse_P_bases,
                                    unembed_np=unembed_np,
                                    output_file=OUTPUT_FILES_DIVERSE_GROUP.get("part_l",
                                        MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_l_complexity_depth_profile.json"),
                                    k_max=K_MAX if 'K_MAX' in dir() else 64,
                                    s_small_r=S_RANK,
                                    model_id=bundle.model_id,
                                    prompt_set_label="Diverse-group",
                                    verbose=True,
                                )
                            except Exception as e:
                                print(f"  ⚠️ Part L (Diverse) failed: {e}")
                    except Exception as e:
                        print(f"  ⚠️ Part G on diverse prompts failed: {e}")
                        traceback.print_exc()
                        result["errors"].append(f"Part G on diverse: {e}")
                
                # --- Regime-level geometry (8 regimes × ~10 prompts) ---
                print("\n  Running geometry at REGIME level (8 regimes)...")
                
                diverse_extraction_regime = build_extraction_object(
                    diverse_layer_to_H, diverse_meta_list, bundle.model_id,
                    diverse_regime_ids_list, diverse_variant_ids_list
                )
                
                regime_P_bases, _ = build_unified_P_bases(diverse_pred_ids, unembed_np, diverse_extraction_regime)
                
                diverse_regime_results = run_consolidated_specA_analysis(
                    extraction=diverse_extraction_regime,
                    P_bases=regime_P_bases,
                    unembed_np=unembed_np,
                    run_parts=diverse_run_parts,
                    output_files=OUTPUT_FILES_DIVERSE_REGIME,
                    model_output_dir=MODEL_OUTPUT_DIR,
                    k_max=K_MAX,
                    s_small_r=DIVERSE_S_RANK,
                    verbose=True,
                    resolution_geometry_layers=RESOLUTION_GEOMETRY_LAYERS,
                )
                result["diverse_geometry_regime"] = diverse_regime_results
                print("  ✓ Regime-level geometry complete")
                
            except Exception as e:
                print(f"  ⚠️ Diverse geometry failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Diverse geometry: {e}")
            
            result["timings"]["diverse_geometry"] = time.time() - phase_start
        else:
            reasons = []
            if not run_diverse_geometry:
                reasons.append("disabled in options")
            if not diverse_prompts_data:
                reasons.append("no diverse prompts loaded")
            if bundle is None:
                reasons.append("no model bundle")
            print(f"  SKIPPED: {', '.join(reasons)}")
        
        # ============================================================
        # Cross-Prompt-Set Comparison (energy spectrum + curvature)
        # ============================================================
        # Computed after Phases 5A-5C, compares results across prompt sets.
        # Only runs if at least 2 prompt sets have resolution geometry data.
        
        try:
            cross_comparison = {"model_id": MODEL_ID, "prompt_sets": {}}
            
            # Collect resolution geometry and energy spectrum results per prompt set
            specA_rg_file = MODEL_OUTPUT_DIR / f"{model_key}-resolution_geometry_v11.json"
            specA_es_file = MODEL_OUTPUT_DIR / f"{model_key}-energy_spectrum.json"
            specB_rg_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-resolution_geometry_v11.json"
            specB_es_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-energy_spectrum.json"
            diverse_rg_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11.json"
            diverse_es_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-energy_spectrum.json"
            diverse_strat_file = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-resolution_geometry_v11_stratified.json"
            
            def _load_if_exists(filepath):
                p = Path(str(filepath))
                if p.exists():
                    with open(p) as f:
                        return json.load(f)
                return None
            
            def _extract_curvature_at_layer(rg_data, layer_key):
                """Extract curvature values for D, P, S, residual at a given layer."""
                if not rg_data or "by_layer" not in rg_data:
                    return None
                layer_data = rg_data["by_layer"].get(layer_key, {})
                summary = layer_data.get("summary", {})
                return {
                    "P_curvature_ratio": summary.get("P_curvature_ratio"),
                    "D_curvature_ratio": summary.get("D_curvature_ratio"),
                    "S_curvature_ratio": summary.get("S_curvature_ratio"),
                    "residual_curvature_ratio": summary.get("residual_curvature_ratio"),
                    "ordering_monotonic": summary.get("ordering_monotonic"),
                    "curvature_ordering": summary.get("curvature_ordering"),
                }
            
            def _extract_energy_at_layer(es_data, layer_key):
                """Extract energy spectrum summary at a given layer."""
                if not es_data or "by_layer" not in es_data:
                    return None
                gl = es_data["by_layer"].get(layer_key, {}).get("global", {})
                if not gl:
                    return None
                return {
                    "k_D": gl.get("k_D"),
                    "energy_at_k_D": gl.get("energy_at_k_D"),
                    "PR": gl.get("PR"),
                    "top_10_energy": gl.get("top_10_energy"),
                }
            
            prompt_set_files = {
                "SpecA": (specA_rg_file, specA_es_file),
                "SpecB": (specB_rg_file, specB_es_file),
                "Diverse-naive": (diverse_rg_file, diverse_es_file),
            }
            
            loaded_rg = {}
            loaded_es = {}
            for ps_name, (rg_f, es_f) in prompt_set_files.items():
                rg = _load_if_exists(rg_f)
                es = _load_if_exists(es_f)
                if rg is not None:
                    loaded_rg[ps_name] = rg
                if es is not None:
                    loaded_es[ps_name] = es
            
            # Also load stratified results
            strat_data = _load_if_exists(diverse_strat_file)
            
            if len(loaded_rg) >= 2:
                # Find the last layer key (should be consistent across prompt sets)
                all_layer_keys = set()
                for rg in loaded_rg.values():
                    all_layer_keys.update(rg.get("by_layer", {}).keys())
                last_layer_key = max(all_layer_keys, key=lambda x: int(x)) if all_layer_keys else None
                
                if last_layer_key:
                    # --- Curvature comparison at final layer ---
                    curvature_comparison = {}
                    for ps_name, rg in loaded_rg.items():
                        curv = _extract_curvature_at_layer(rg, last_layer_key)
                        if curv:
                            curvature_comparison[ps_name] = curv
                    
                    # Add stratified average if available
                    if strat_data and "averaged" in strat_data:
                        avg_summary = strat_data["averaged"].get("by_layer", {}).get(last_layer_key, {}).get("summary", {})
                        if avg_summary:
                            curvature_comparison["Diverse-stratified-mean"] = {
                                "P_curvature_ratio": avg_summary.get("P_curvature_ratio"),
                                "D_curvature_ratio": avg_summary.get("D_curvature_ratio"),
                                "S_curvature_ratio": avg_summary.get("S_curvature_ratio"),
                                "residual_curvature_ratio": avg_summary.get("residual_curvature_ratio"),
                                "ordering_monotonic": avg_summary.get("ordering_monotonic"),
                                "curvature_ordering": avg_summary.get("curvature_ordering"),
                            }
                    
                    cross_comparison["curvature_at_final_layer"] = curvature_comparison
                    cross_comparison["final_layer"] = last_layer_key
                    
                    # --- Minimum pairwise gap per prompt set ---
                    for ps_name, curv in curvature_comparison.items():
                        ordering = curv.get("curvature_ordering", [])
                        if len(ordering) > 1:
                            gaps = [ordering[i+1][1] - ordering[i][1] for i in range(len(ordering)-1)]
                            curv["min_pairwise_gap"] = round(min(gaps), 4) if gaps else None
                            curv["weakest_link"] = f"{ordering[gaps.index(min(gaps))][0]}->{ordering[gaps.index(min(gaps))+1][0]}" if gaps else None
                    
                    # --- Energy spectrum comparison at final layer ---
                    energy_comparison = {}
                    for ps_name, es in loaded_es.items():
                        en = _extract_energy_at_layer(es, last_layer_key)
                        if en:
                            energy_comparison[ps_name] = en
                    
                    if energy_comparison:
                        # Delta k_D between prompt sets
                        ps_names = sorted(energy_comparison.keys())
                        deltas = {}
                        for i in range(len(ps_names)):
                            for j in range(i+1, len(ps_names)):
                                k1 = energy_comparison[ps_names[i]].get("k_D")
                                k2 = energy_comparison[ps_names[j]].get("k_D")
                                if k1 is not None and k2 is not None:
                                    deltas[f"{ps_names[i]}_vs_{ps_names[j]}"] = k1 - k2
                        energy_comparison["delta_k_D"] = deltas
                        cross_comparison["energy_spectrum_at_final_layer"] = energy_comparison
                    
                    # --- D/F energy relationship from per-prompt trajectories ---
                    df_energy = {}
                    traj_files = {
                        "SpecA": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecA-per_prompt_trajectories.json",
                        "SpecB": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-per_prompt_trajectories.json",
                        "Diverse-group": MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-per_prompt_trajectories.json",
                    }
                    for ps_name, tf in traj_files.items():
                        traj_data = _load_if_exists(tf)
                        if traj_data and "prompts" in traj_data:
                            D_fracs = []
                            F_fracs = []
                            for p in traj_data["prompts"]:
                                ld = p.get("trajectory", {}).get(last_layer_key, {})
                                d_e = ld.get("D_energy_frac")
                                f_e = ld.get("F_energy_frac")
                                if d_e is not None:
                                    D_fracs.append(d_e)
                                if f_e is not None:
                                    F_fracs.append(f_e)
                            if D_fracs and F_fracs:
                                mean_D = float(np.mean(D_fracs))
                                mean_F = float(np.mean(F_fracs))
                                df_energy[ps_name] = {
                                    "mean_D_energy_frac": round(mean_D, 6),
                                    "mean_F_energy_frac": round(mean_F, 6),
                                    "D_dominates_F": mean_D > mean_F,
                                }
                    
                    if df_energy:
                        # Key test: does curvature ordering hold even when D > F?
                        for ps_name, df in df_energy.items():
                            rg_name = ps_name if ps_name != "Diverse-group" else "Diverse-naive"
                            curv = curvature_comparison.get(rg_name, {})
                            df["curvature_ordering_holds"] = curv.get("ordering_monotonic")
                        cross_comparison["df_energy_relationship"] = df_energy
                
                # Save comparison file
                comp_file = OUTPUT_FILES_COMPARISON.get(
                    "cross_prompt_set",
                    MODEL_OUTPUT_DIR / f"{model_key}-Geometry-cross_prompt_set_comparison.json"
                )
                save_json(cross_comparison, comp_file)
                print(f"\n  ✓ Cross-prompt-set comparison saved ({len(loaded_rg)} prompt sets)")
            else:
                print(f"\n  Cross-prompt-set comparison: SKIPPED (only {len(loaded_rg)} prompt set(s) have resolution geometry)")
        
        except Exception as e:
            print(f"\n  ⚠️ Cross-prompt-set comparison failed: {e}")
            traceback.print_exc()
            result["errors"].append(f"Cross-prompt-set comparison: {e}")
        
        # ============================================================
        # Phase 6: SpecB Continuation Experiments
        # ============================================================
        run_specb2 = options.get("run_specB2", False)
        print(f"\n[{_phase_time()}] [Phase 6/9] SpecB Continuation Experiments")
        print("-" * 40)
        print(f"  run_continuation option: {run_specb2}")
        print(f"  continuation_available: {SPECB2_AVAILABLE}")
        print(f"  continuation_prompts_data: {'loaded' if continuation_prompts_data else 'None'}")
        
        if run_specb2 and SPECB2_AVAILABLE and continuation_prompts_data:
            phase_start = time.time()
            
            # Convert specb_data dict to List[PromptRow]
            print("  Converting prompts to PromptRow objects...")
            specb2_tier = globals().get('SPECB2_TIER', 'standard')
            specb_rows = convert_specb_data_to_rows(continuation_prompts_data, tier=specb2_tier)
            print(f"  ✓ Converted {len(specb_rows)} prompts (tier: {specb2_tier})")
            
            if len(specb_rows) == 0:
                print("  ⚠️ No prompts after tier filtering!")
                result["errors"].append("SpecB continuation: No prompts after tier filtering")
            else:
                specb2_options = {
                    "run_P_scramble": options.get("run_specB2_P_scramble", True),
                    "run_D_scramble": options.get("run_specB2_D_scramble", True),
                    "run_S_scramble": options.get("run_specB2_S_scramble", True),
                    "run_random_control": options.get("run_specB2_random_control", False),
                    "run_F_attenuate": options.get("run_continuation_F_tests", True),
                    "run_F_mix": options.get("run_continuation_F_tests", True),
                    "run_F_transplant": options.get("run_continuation_F_tests", True),
                    "run_F_early_attenuate": options.get("run_continuation_F_early", True),
                    "run_F_early_mix": options.get("run_continuation_F_early", True),
                    "run_F_early_transplant": options.get("run_continuation_F_early", True),
                    # Cache options - load existing bases/baselines if available
                    "use_cached_bases": options.get("use_cached_bases", True),
                    "use_cached_baselines": options.get("use_cached_baselines", True),
                    "bases_file": options.get("bases_file"),
                    "baselines_file": options.get("baselines_file"),
                }
                specb2_config = {
                    "scramble_layer_fraction": SPECB2_SCRAMBLE_LAYER_FRACTION,
                    "n_generate": SPECB2_N_GENERATE,
                    "seed": SPECB2_SEED,
                    "pair_policy": PAIR_POLICY,
                    "attenuate_alpha_control": ATTENUATE_ALPHA_CONTROL,
                    "F_attenuate_alpha": ATTENUATE_ALPHA_CONTROL,
                    "p_rank": SPECB2_P_RANK,
                    "d_rank": SPECB2_D_RANK,
                    "s_rank": SPECB2_S_RANK,
                    "random_rank": SPECB2_RANDOM_RANK,
                    "scramble_type": SPECB2_SCRAMBLE_TYPE,
                    "file_prefix": "Continuation-SpecB",
                    "f_early_layer_fraction": CONTINUATION_F_EARLY_LAYER_FRACTION,
                }
                
                print(f"  Config: layer_frac={specb2_config['scramble_layer_fraction']}, n_gen={specb2_config['n_generate']}")
                enabled = [k.replace('run_','') for k,v in specb2_options.items() if v is True and k.startswith('run_')]
                print(f"  Enabled experiments: {', '.join(enabled) if enabled else 'NONE'}")
                
                try:
                    result["specB2_results"] = run_specB2_for_pipeline(
                        bundle.model, bundle.tokenizer, specb_rows,
                        SPECB2_MODEL_OUTPUT_DIR, model_key, bundle.model_id,
                        specb2_config, specb2_options, verbose=options.get("verbose", True))
                    print("  ✓ SpecB continuation complete")
                except Exception as e:
                    print(f"  ⚠️ SpecB continuation failed: {e}")
                    traceback.print_exc()
                    result["errors"].append(f"SpecB continuation: {e}")
            
            result["timings"]["specB2"] = time.time() - phase_start
        else:
            reasons = []
            if not run_specb2:
                reasons.append("run_specB2=False")
            if not SPECB2_AVAILABLE:
                reasons.append("continuation module not available")
            if not continuation_prompts_data:
                reasons.append("continuation_prompts_data=None")
            print(f"  SKIPPED: {', '.join(reasons)}")
        
        # ============================================================
        # Phase 7: Prompt Analysis (Token Distributions & SpecA-style on SpecB)
        # ============================================================
        run_prompt_analysis = options.get("run_prompt_analysis", False)
        print(f"\n[{_phase_time()}] [Phase 7/9] Prompt Analysis (enabled: {run_prompt_analysis})")
        print("-" * 40)
        
        if run_prompt_analysis and PROMPT_ANALYSIS_AVAILABLE:
            phase_start = time.time()
            
            try:
                from pds_prompt_analysis import (
                    compute_prompt_set_distributions,
                    compare_prompt_set_distributions,
                    plot_distribution_comparison_figure,
                    plot_cumulative_probability_curves,
                    run_specA_analysis_on_prompts,
                    save_distribution_stats,
                )
                
                prompt_analysis_dir = OUTPUT_DIR / "prompt_analysis"
                prompt_analysis_dir.mkdir(parents=True, exist_ok=True)
                
                # --- Token Distribution Comparison ---
                if options.get("run_token_distribution_comparison", True):
                    print("\n  Running token distribution comparison...")
                    
                    # Prepare SpecA prompts
                    specA_prompt_list = []
                    for item in meta_list:
                        prompt_id = f"{item['group_id']}_{item['variant_id']}"
                        specA_prompt_list.append((prompt_id, item['prompt']))
                    
                    # Prepare SpecB prompts
                    specB_prompt_list = []
                    if continuation_prompts_data:
                        for group in continuation_prompts_data.get('groups', []):
                            group_id = group.get('group_id', '')
                            for var in group.get('variants', []):
                                var_id = var.get('variant_id', '')
                                prompt_text = var.get('prompt', '')
                                specB_prompt_list.append((f"{group_id}_{var_id}", prompt_text))
                    
                    n_analysis = globals().get('PROMPT_ANALYSIS_N_PROMPTS', 50)
                    
                    if specA_prompt_list and specB_prompt_list:
                        # Compute distributions
                        stats_A = compute_prompt_set_distributions(
                            bundle.model, bundle.tokenizer,
                            specA_prompt_list[:n_analysis], "SpecA",
                            model_id=bundle.model_id, verbose=options.get("verbose", True)
                        )
                        stats_B = compute_prompt_set_distributions(
                            bundle.model, bundle.tokenizer,
                            specB_prompt_list[:n_analysis], "SpecB",
                            model_id=bundle.model_id, verbose=options.get("verbose", True)
                        )
                        
                        # Compare and save
                        comparison = compare_prompt_set_distributions(stats_A, stats_B, verbose=True)
                        
                        # Save results
                        save_distribution_stats(stats_A, prompt_analysis_dir, model_key)
                        save_distribution_stats(stats_B, prompt_analysis_dir, model_key)
                        
                        # Save comparison
                        comparison_file = prompt_analysis_dir / f"{model_key}-distribution_comparison.json"
                        with open(comparison_file, 'w') as f:
                            json.dump(comparison, f, indent=2)
                        
                        # Generate figures
                        try:
                            plot_distribution_comparison_figure(
                                stats_A, stats_B, n_examples=3,
                                save_path=prompt_analysis_dir / f"{model_key}-distribution_comparison.png"
                            )
                            plot_cumulative_probability_curves(
                                stats_A, stats_B,
                                save_path=prompt_analysis_dir / f"{model_key}-cumulative_curves.png"
                            )
                        except Exception as fig_e:
                            print(f"    ⚠️ Could not generate figures: {fig_e}")
                        
                        result["token_distribution_results"] = {
                            "specA_entropy": stats_A.mean_entropy,
                            "specB_entropy": stats_B.mean_entropy,
                            "specA_pr": stats_A.mean_pr,
                            "specB_pr": stats_B.mean_pr,
                        }
                        print("  ✓ Token distribution comparison complete")
                    else:
                        print("  ⚠️ Missing prompts for comparison")
                
                # --- SpecA-style Analysis on SpecB Prompts ---
                if options.get("run_specA_style_on_specB", True) and continuation_prompts_data:
                    print("\n  Running SpecA-style analysis on SpecB prompts...")
                    
                    # Prepare SpecB prompts as (group_id, variant_id, prompt_text)
                    specB_for_analysis = []
                    for group in continuation_prompts_data.get('groups', []):
                        group_id = group.get('group_id', '')
                        for var in group.get('variants', []):
                            var_id = var.get('variant_id', '')
                            prompt_text = var.get('prompt', '')
                            specB_for_analysis.append((group_id, var_id, prompt_text))
                    
                    # Check for cached bases from SpecB2
                    cached_bases = None
                    if 'SPECB2_MODEL_OUTPUT_DIR' in dir() or 'SPECB2_MODEL_OUTPUT_DIR' in globals():
                        specb2_dir = globals().get('SPECB2_MODEL_OUTPUT_DIR', OUTPUT_DIR / "SpecB2" / model_key)
                        bases_file = specb2_dir / f"{model_key}-SpecB2-bases.json"
                        if bases_file.exists():
                            try:
                                cached_bases = load_specB2_bases(bases_file)
                                print(f"    Using cached bases from SpecB continuation")
                            except:
                                cached_bases = None
                    
                    specA_on_specB_dir = OUTPUT_DIR / "specA_on_specB"
                    specA_style_results = run_specA_analysis_on_prompts(
                        bundle.model,
                        bundle.tokenizer,
                        specB_for_analysis,
                        bundle.model_id,
                        specA_on_specB_dir,
                        model_key,
                        layer_fraction=0.70,
                        config={"p_rank": "adaptive", "d_rank": "adaptive", "s_rank": "adaptive"},
                        precomputed_bases=cached_bases,
                        verbose=options.get("verbose", True)
                    )
                    
                    result["specA_on_specB_results"] = specA_style_results
                    print("  ✓ SpecA-style analysis on SpecB complete")
                
            except Exception as e:
                print(f"  ⚠️ Prompt analysis failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Prompt analysis: {e}")
            
            result["timings"]["prompt_analysis"] = time.time() - phase_start
        
        elif run_prompt_analysis and not PROMPT_ANALYSIS_AVAILABLE:
            print("  SKIPPED: PROMPT_ANALYSIS_AVAILABLE=False")
        else:
            print("  SKIPPED: disabled in options")
        
        # ============================================================
        # Phase 8: Diverse Continuation Experiments
        # ============================================================
        run_diverse = options.get("run_diverse_continuation_tests", False)
        print(f"\n[{_phase_time()}] [Phase 8/9] Diverse Continuation (enabled: {run_diverse})")
        print("-" * 40)
        
        if run_diverse and SPECB2_AVAILABLE and diverse_prompts_data and bundle is not None:
            phase_start = time.time()
            
            try:
                # Parse diverse prompts if not already done in Phase 5C
                if not diverse_prompts_parsed:
                    if "prompts" in diverse_prompts_data:
                        for p in diverse_prompts_data["prompts"]:
                            diverse_prompts_list.append(p["prompt"])
                            diverse_group_ids_list.append(p["group_id"])
                            diverse_variant_ids_list.append(p["variant_id"])
                            diverse_regime_ids_list.append(p.get("regime", ""))
                            diverse_meta_list.append({
                                "group_id": p["group_id"],
                                "variant_id": p["variant_id"],
                                "prompt": p["prompt"],
                                "category": p.get("category", ""),
                                "regime": p.get("regime", ""),
                                "expected_token": None,
                            })
                    else:
                        for group in diverse_prompts_data.get("groups", []):
                            gid = group.get("group_id", "")
                            for var in group.get("variants", []):
                                diverse_prompts_list.append(var.get("prompt", ""))
                                diverse_group_ids_list.append(gid)
                                diverse_variant_ids_list.append(var.get("variant_id", ""))
                                diverse_regime_ids_list.append(group.get("regime", ""))
                                diverse_meta_list.append({
                                    "group_id": gid,
                                    "variant_id": var.get("variant_id", ""),
                                    "prompt": var.get("prompt", ""),
                                    "category": group.get("category", ""),
                                    "regime": group.get("regime", ""),
                                    "expected_token": None,
                                })
                    diverse_prompts_parsed = True
                
                print(f"  Diverse prompts: {len(diverse_prompts_list)}")
                
                DIVERSE_MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
                
                diverse_rows = convert_specb_data_to_rows(diverse_prompts_data, tier="full")
                print(f"  ✓ {len(diverse_rows)} PromptRow objects")
                
                if len(diverse_rows) > 0:
                    diverse_options = {
                        "run_P_scramble": options.get("run_diverse_P_scramble", False),
                        "run_D_scramble": options.get("run_diverse_D_scramble", False),
                        "run_S_scramble": options.get("run_diverse_S_scramble", False),
                        "run_random_control": False,
                        "run_F_attenuate": options.get("run_diverse_F_tests", False),
                        "run_F_mix": options.get("run_diverse_F_tests", False),
                        "run_F_transplant": options.get("run_diverse_F_tests", False),
                        "run_F_early_attenuate": options.get("run_diverse_F_early", False),
                        "run_F_early_mix": options.get("run_diverse_F_early", False),
                        "run_F_early_transplant": options.get("run_diverse_F_early", False),
                        "use_cached_bases": options.get("use_cached_bases", True),
                        "use_cached_baselines": options.get("use_cached_baselines", True),
                    }
                    diverse_config = {
                        "scramble_layer_fraction": SPECB2_SCRAMBLE_LAYER_FRACTION,
                        "n_generate": SPECB2_N_GENERATE,
                        "seed": SPECB2_SEED,
                        "pair_policy": PAIR_POLICY,
                        "attenuate_alpha_control": ATTENUATE_ALPHA_CONTROL,
                        "F_attenuate_alpha": ATTENUATE_ALPHA_CONTROL,
                        "p_rank": SPECB2_P_RANK,
                        "d_rank": SPECB2_D_RANK,
                        "s_rank": SPECB2_S_RANK,
                        "random_rank": SPECB2_RANDOM_RANK,
                        "scramble_type": SPECB2_SCRAMBLE_TYPE,
                        "file_prefix": "Continuation-Diverse",
                        "f_early_layer_fraction": CONTINUATION_F_EARLY_LAYER_FRACTION,
                    }
                    
                    result["diverse_continuation"] = run_specB2_for_pipeline(
                        bundle.model, bundle.tokenizer, diverse_rows,
                        SPECB2_MODEL_OUTPUT_DIR, model_key, bundle.model_id,
                        diverse_config, diverse_options,
                        verbose=options.get("verbose", True))
                    print("  ✓ Diverse continuation tests complete")
                else:
                    print("  ⚠️ No diverse prompts after conversion")
                
            except Exception as e:
                print(f"  ⚠️ Diverse continuation failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Diverse continuation: {e}")
            
            result["timings"]["diverse_continuation"] = time.time() - phase_start
        else:
            reasons = []
            if not run_diverse:
                reasons.append("disabled in options")
            if not SPECB2_AVAILABLE:
                reasons.append("continuation module not available")
            if not diverse_prompts_data:
                reasons.append("no diverse prompts loaded")
            if bundle is None:
                reasons.append("no model bundle")
            print(f"  SKIPPED: {', '.join(reasons)}")
        

        # ============================================================
        # Final: Geometry Comparison (all prompt sets)
        # ============================================================
        run_comparison = options.get("run_geometry_comparison", True)
        print(f"\n[Final] Geometry Comparison (enabled: {run_comparison})")
        print("-" * 40)
        
        if run_comparison:
            phase_start = time.time()
            
            try:
                from pds_prompt_analysis import compare_geometry_analyses
                
                # Collect all prompt sets that have geometry results
                prompt_sets = {}
                token_dist_files = {}
                
                # SpecA (always present if geometry ran)
                if extraction is not None:
                    prompt_sets["SpecA"] = "SpecA"
                    td = OUTPUT_FILES.get("token_dist")
                    if td and Path(str(td)).exists():
                        token_dist_files["SpecA"] = td
                
                # SpecB (standard continuation prompts)
                specB_check = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-SpecB-part_a_pr.json"
                if specB_check.exists():
                    prompt_sets["SpecB"] = "SpecB"
                    td = OUTPUT_FILES_SPECB.get("token_dist")
                    if td and Path(str(td)).exists():
                        token_dist_files["SpecB"] = td
                
                # Diverse (group-level)
                diverse_group_check = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-group-part_a_pr.json"
                if diverse_group_check.exists():
                    prompt_sets["Diverse-group"] = "Diverse-group"
                    td = OUTPUT_FILES_DIVERSE_GROUP.get("token_dist")
                    if td and Path(str(td)).exists():
                        token_dist_files["Diverse-group"] = td
                
                # Diverse (regime-level)
                diverse_regime_check = MODEL_OUTPUT_DIR / f"{model_key}-Geometry-Diverse-regime-part_a_pr.json"
                if diverse_regime_check.exists():
                    prompt_sets["Diverse-regime"] = "Diverse-regime"
                    td = OUTPUT_FILES_DIVERSE_REGIME.get("token_dist")
                    if td and Path(str(td)).exists():
                        token_dist_files["Diverse-regime"] = td
                
                if len(prompt_sets) >= 2:
                    comparison_result = compare_geometry_analyses(
                        results_dir=MODEL_OUTPUT_DIR,
                        output_file=OUTPUT_FILES_COMPARISON["comparison"],
                        model_key=model_key,
                        prompt_sets=prompt_sets,
                        model_id=MODEL_ID,
                        token_dist_files=token_dist_files,
                        run_parts=RUN_PARTS,
                        verbose=True,
                    )
                    result["geometry_comparison"] = {"status": "completed", "sets": list(prompt_sets.keys())}
                    print(f"  ✓ Comparison complete ({len(prompt_sets)} sets)")
                else:
                    print(f"  SKIPPED: only {len(prompt_sets)} set(s) available (need ≥2)")
                
            except Exception as e:
                print(f"  ⚠️ Geometry comparison failed: {e}")
                traceback.print_exc()
                result["errors"].append(f"Geometry comparison: {e}")
            
            result["timings"]["geometry_comparison"] = time.time() - phase_start
        else:
            print("  SKIPPED: disabled")
        
        result["status"] = "completed"
        
    except Exception as e:
        result["status"] = "failed"
        result["errors"].append(str(e))
        print(f"\n⚠️ PIPELINE ERROR: {e}")
        traceback.print_exc()
    
    finally:
        # Cleanup
        print("\n" + "-" * 40)
        print("Cleanup")
        if bundle is not None:
            try:
                del bundle.model
                del bundle
            except:
                pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("  ✓ Cleanup complete")
    
    result["total_time"] = time.time() - pipeline_start
    print(f"\n{'='*70}")
    print(f"PIPELINE COMPLETE: {model_key} ({result['total_time']/60:.1f} min, elapsed {_phase_time()})")
    print(f"{'='*70}")
    
    if result["errors"]:
        print(f"\nErrors ({len(result['errors'])}):")
        for err in result["errors"]:
            print(f"  - {err}")
    
    return result


print("✓ Pipeline functions defined (v10.2)")
print("  - convert_specb_data_to_rows: included")
print("  - Continuation uses PromptRow objects: YES")
print("  - Phase order: geometry (4,5A-C) → continuation (6,8) → comparison")


---
# Section 6: Execute Pipeline

Run the pipeline on selected models.

In [ ]:
# EXECUTE PIPELINE
# ============================================================

import time as _time
from datetime import timedelta

PIPELINE_RESULTS = {}
_global_start = _time.time()

def _elapsed():
    """Return formatted elapsed time since pipeline start."""
    return str(timedelta(seconds=int(_time.time() - _global_start)))

print("=" * 60)
print("EXECUTING PIPELINE")
print("=" * 60)
print(f"Models: {MODELS_TO_RUN}")
print(f"Started: {_time.strftime('%Y-%m-%d %H:%M:%S')}")

for model_key in MODELS_TO_RUN:
    try:
        result = run_full_pipeline(model_key, options=PIPELINE_OPTIONS)
        PIPELINE_RESULTS[model_key] = result
    except Exception as e:
        print(f"\n✗ {model_key} FAILED: {e}")
        PIPELINE_RESULTS[model_key] = {"status": "failed", "error": str(e)}
    
    # Optionally clear extraction cache between models to save disk space
    if PIPELINE_OPTIONS.get("clear_cache_between_models", False):
        try:
            cache_npz = CACHE_DIR / f"extraction_{model_key}.npz"
            cache_meta = CACHE_DIR / f"extraction_{model_key}.meta.json"
            for cf in [cache_npz, cache_meta]:
                if cf.exists():
                    cf.unlink()
                    print(f"  🗑 Cleared cache: {cf.name}")
        except Exception as e:
            print(f"  ⚠️ Cache clear failed: {e}")

# Summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 60)

completed = [k for k, v in PIPELINE_RESULTS.items() if v.get("status") == "completed"]
failed = [k for k, v in PIPELINE_RESULTS.items() if v.get("status") != "completed"]

print(f"\nCompleted ({len(completed)}):")
for m in completed:
    t = PIPELINE_RESULTS[m].get("total_time", 0)
    print(f"  ✓ {m} ({t/60:.1f} min)")

if failed:
    print(f"\nFailed ({len(failed)}):")
    for m in failed:
        print(f"  ✗ {m}")

print(f"\nResults saved to:")
print(f"  Geometry: {OUTPUT_DIR}/")
print(f"  Continuation: {CONTINUATION_OUTPUT_DIR}/")

_total_elapsed = _time.time() - _global_start
print(f"\n{'='*60}")
print(f"ALL MODELS COMPLETE")
print(f"{'='*60}")
print(f"  Started:  {_time.strftime('%H:%M:%S', _time.localtime(_global_start))}")
print(f"  Finished: {_time.strftime('%H:%M:%S')}")
print(f"  Duration: {str(timedelta(seconds=int(_total_elapsed)))}")
if _total_elapsed > 3600:
    print(f"           ({_total_elapsed/3600:.1f} hours)")
elif _total_elapsed > 60:
    print(f"           ({_total_elapsed/60:.1f} minutes)")


---
# Section 7: Cleanup

Clean up GPU memory and optionally clear HuggingFace cache.

In [ ]:
# CLEANUP
# ============================================================

print("=" * 60)
print("CLEANUP")
print("=" * 60)

# Clear any remaining model from memory
if 'bundle' in dir() and bundle is not None:
    clear_model_from_memory(bundle=bundle, verbose=True)
    try:
        del bundle
    except:
        pass

# Clear GPU memory
cleanup_gpu_memory()
print("  ✓ GPU memory cleared")

# Show cache status
print("\nCache status:")
print_cache_status()

# To clear cache for a specific model, uncomment:
# clear_hf_cache_for_model("meta-llama/Llama-3.3-70B-Instruct", verbose=True)

# To clear all cached models, uncomment:
# clear_all_hf_cache(confirm=True, verbose=True)

print("\n✓ Cleanup complete")